# Abstract

This project develops and evaluates a **Pairs Trading Strategy** as part of the CQF final assessment.  
The methodology combines rolling OLS regression for hedge ratio estimation, z-score standardization of the spread, and rule-based entry/exit signals.  
A backtest framework with transaction costs and slippage is implemented to simulate realistic trading conditions.  
Performance is assessed through CAGR, Sharpe ratio, volatility, and maximum drawdown, while diagnostic charts provide visual validation of the strategy.  
The study highlights both the strengths (robust implementation, reproducibility, exportable results) and the limitations (synthetic data, absence of formal cointegration tests), establishing a solid foundation for further quantitative research and professional application.


# Executive Summary

This project develops a reproducible pipeline for pairs trading, integrating data collection, preprocessing, rolling hedge ratio estimation, signal generation, and backtesting with transaction costs and slippage. Using equity data (AAPL, GOOGL, META), the framework provides trade logs, performance metrics, and diagnostic charts. The methodology demonstrates the mechanics of statistical arbitrage, highlights limitations (absence of formal cointegration testing, simplified risk management), and proposes extensions such as volatility targeting and multivariate Johansen cointegration. The results confirm the feasibility of the approach and provide a foundation for further research and professional applications.


# Introduction

This project implements a **Pairs Trading Strategy** as part of the CQF program.  
The goal is to identify a cointegration relationship between two assets and to trade the mean reversion of their spread.  

**Key objectives:**
- Collect and align historical price data for two assets.
- Estimate the hedge ratio using rolling OLS regression.
- Construct the spread and evaluate its stationarity with z-scores.
- Generate trading signals based on z-score thresholds.
- Backtest the strategy including transaction costs and slippage.
- Evaluate the performance using financial metrics and visual diagnostics.


# Methodology

The strategy is based on the classical pairs trading framework:

1. **Data Handling**  
   - Price series of two assets are read from CSV files (`Date`, `Close`).  
   - If files are missing, synthetic data is generated for demonstration purposes.  
   - Prices are aligned to a common business-day calendar.

2. **Hedge Ratio Estimation**  
   - The hedge ratio is estimated via rolling Ordinary Least Squares (OLS).  
   - The spread is defined as:  
     \[
     \text{Spread}_t = Y_t - \beta_t X_t
     \]

3. **Z-Score and Signals**  
   - A rolling mean and standard deviation are used to standardize the spread.  
   - Trading rules:  
     - Go **short spread** when \( z > entry \).  
     - Go **long spread** when \( z < -entry \).  
     - Close position when \(|z| < exit\).

4. **Backtesting**  
   - Positions are executed with a one-day lag (T+1).  
   - Transaction costs and slippage are applied at each trade.  
   - Equity curve starts at a fixed notional (e.g., 1,000,000).  
   - PnL and performance metrics are computed daily.

5. **Performance Evaluation**  
   - Metrics include: CAGR, annualized return, volatility, Sharpe ratio, and maximum drawdown.  
   - Visual outputs: aligned prices, spread, z-score, and equity curve.


In [ ]:
# Environment print (reproducibility)
import platform, numpy as np, pandas as pd, matplotlib
print("Env:",
      "Python", platform.python_version(),
      "| numpy", np.__version__,
      "| pandas", pd.__version__,
      "| matplotlib", matplotlib.__version__)


In [ ]:
# === 0A: GLOBAL IMPORTS & SETUP ===
# (Keep this cell at the very top of the notebook)

# If you plan to export Word in Part 6, uncomment this:
# !pip -q install python-docx

import os, sys, re, warnings, math, itertools, textwrap
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Stats / econometrics
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, coint
from scipy import stats

# Global display & plot style
np.set_printoptions(suppress=True, linewidth=120)
pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore")

mpl.rcParams["figure.dpi"] = 130
mpl.rcParams["axes.grid"] = True
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["axes.labelsize"] = 10
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["font.family"] = "DejaVu Sans"

# Global constants
SEED = 42
np.random.seed(SEED)
TRADING_DAYS = 252

# Output folders (used across parts; feel free to adjust)
OUT_DIRS = {
    "part1": "/content",
    "part3": "/content/part3_outputs",
    "part4": "/content/part4_outputs",
    "part5": "/content/part5_outputs",
    "export": "/content"
}
for d in OUT_DIRS.values():
    os.makedirs(d, exist_ok=True)

# Stress windows aligned with your PDF (no COVID)
STRESS_WINDOWS = [
    ("OilCrash_2014_2016", "2014-06-01", "2016-02-29"),
    ("Russia_Ukraine_2022", "2022-01-01", "2022-12-31"),
]


In [ ]:
# === 0B: SHARED UTILS (Optional, shared across parts) ===

# ---------- Data parsing ----------
def util_find_datetime_column(df: pd.DataFrame) -> str:
    for c in df.columns:
        if any(k in c.lower() for k in ["date","time","timestamp","datetime","unnamed","index"]):
            return c
    return df.columns[0]

def util_set_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
    c = util_find_datetime_column(df)
    df[c] = pd.to_datetime(df[c], errors="coerce")
    df = df.set_index(c).sort_index()
    return df

# ---------- Basic transforms ----------
def util_winsorize_returns(ret: pd.Series, q_low=0.01, q_high=0.99) -> pd.Series:
    if ret.dropna().empty: return ret
    lo, hi = ret.quantile([q_low, q_high])
    return ret.clip(lower=lo, upper=hi)

def util_rebuild_price_from_returns(ret: pd.Series, start_price: float) -> pd.Series:
    return (1.0 + ret.fillna(0.0)).cumprod() * float(start_price)

# ---------- Metrics & risk ----------
def util_ann_metrics(daily_ret: pd.Series) -> dict:
    s = pd.Series(daily_ret).dropna()
    if s.empty:
        return {"CAGR": np.nan, "Vol_ann": np.nan, "Sharpe": np.nan, "MaxDD": np.nan}
    years = len(s) / TRADING_DAYS
    cagr = (1 + s).prod() ** (1/years) - 1 if years > 0 else np.nan
    vol  = s.std() * np.sqrt(TRADING_DAYS)
    sharpe = (s.mean() * np.sqrt(TRADING_DAYS)) / (s.std() + 1e-12)
    eq = (1 + s).cumprod()
    dd = eq / eq.cummax() - 1
    return {"CAGR": float(cagr), "Vol_ann": float(vol), "Sharpe": float(sharpe), "MaxDD": float(dd.min())}

def util_hist_var_es(daily_ret: pd.Series, alpha=0.95) -> dict:
    s = pd.Series(daily_ret).dropna().values
    if s.size == 0:
        return {"VaR": np.nan, "ES": np.nan}
    var = float(np.quantile(s, 1 - alpha))
    es  = float(s[s <= var].mean()) if np.any(s <= var) else var
    return {"VaR": var, "ES": es}

# ---------- Engle–Granger / signals ----------
def util_eg_alpha_beta(y: pd.Series, x: pd.Series):
    X = sm.add_constant(x.values)
    model = sm.OLS(y.values, X, missing="drop").fit()
    return float(model.params[0]), float(model.params[1])

def util_zscore(series: pd.Series, L: int):
    m = series.rolling(L).mean()
    s = series.rolling(L).std(ddof=0)
    return (series - m) / s

def util_state_machine(Z: pd.Series, zin: float, zout: float, zstop: float):
    Z = Z.copy()
    st = np.zeros(len(Z), dtype=int)
    for t in range(1, len(Z)):
        z = Z.iloc[t]; prev = st[t-1]
        if np.isnan(z): st[t] = 0; continue
        if prev == 0:
            if z <= -zin: st[t] = 1
            elif z >=  zin: st[t] = -1
            else: st[t] = 0
        else:
            if abs(z) <= zout or abs(z) >= zstop: st[t] = 0
            else: st[t] = prev
    return pd.Series(st, index=Z.index, name="state")

# ---------- Backtest kernel (with turnover costs) ----------
def util_backtest_pair_params(prices_df: pd.DataFrame, symY: str, symX: str, lookback: int,
                              zin: float, zout: float, zstop_mode="fixed", zstop_fixed=3.0,
                              target_vol_ann=0.10, window_vol=60,
                              tx_cost_bps_per_leg=5, max_leverage=5.0,
                              min_obs_required=200):
    px = prices_df[[symY, symX]].dropna()
    if len(px) < min_obs_required:
        return pd.Series(dtype=float), {"CAGR":np.nan,"Vol_ann":np.nan,"Sharpe":np.nan,"MaxDD":np.nan}

    logp = np.log(px)
    rets = px.pct_change()

    y, x = logp[symY], logp[symX]
    rY, rX = rets[symY], rets[symX]
    alpha, beta = util_eg_alpha_beta(y, x)

    spread = y - (alpha + beta * x)
    Z = util_zscore(spread, lookback)
    zstop = zstop_fixed if zstop_mode == "fixed" else (2.0 * zin)
    state = util_state_machine(Z, zin, zout, zstop)

    ret0 = state.shift(1) * (rY - beta * rX)
    ret0 = ret0.fillna(0.0)

    vol_roll = ret0.rolling(window_vol).std() * np.sqrt(TRADING_DAYS)
    weight = (target_vol_ann / vol_roll.replace(0, np.nan)).clip(upper=max_leverage).fillna(0.0)
    w = weight.shift(1).fillna(0.0)
    ret_gross = w * ret0

    # Transaction costs by turnover (Δposition)
    position = (state.shift(1).fillna(0.0)) * w
    turnover = position.diff().abs().fillna(position.abs())  # cost on first entry too
    cost_unit = (tx_cost_bps_per_leg / 10000.0) * 2
    cost = cost_unit * turnover

    ret_net = (ret_gross - cost).fillna(0.0)
    return ret_net, util_ann_metrics(ret_net)

# ---------- Plots ----------
def util_plot_equity_with_drawdown(eq: pd.Series, title: str, out_png: str = None):
    eq = eq.dropna()
    rollmax = eq.cummax()
    dd = eq / rollmax - 1.0
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(eq.index, eq.values, label="Equity")
    ax1.set_title(title); ax1.set_ylabel("Cumulative (×)")
    ax1.legend(loc="upper left")
    ax2 = ax1.twinx()
    ax2.fill_between(dd.index, dd.values, 0, alpha=0.2, step="pre")
    ax2.set_ylabel("Drawdown")
    plt.tight_layout()
    if out_png: plt.savefig(out_png, dpi=150)
    plt.show()

def util_plot_zscore(Z: pd.Series, zin: float, zout: float, zstop: float,
                     title: str, out_png: str = None):
    Z = Z.dropna()
    plt.figure(figsize=(10, 4))
    plt.plot(Z.index, Z.values, label="Z-score")
    plt.axhline(0, linestyle="--", label="mean")
    plt.axhline( zin, linestyle="--", label="+zin")
    plt.axhline(-zin, linestyle="--", label="-zin")
    plt.axhline( zout, linestyle=":",  label="+zout")
    plt.axhline(-zout, linestyle=":",  label="-zout")
    plt.axhline( zstop, linestyle=":", alpha=0.7, label="+zstop")
    plt.axhline(-zstop, linestyle=":", alpha=0.7, label="-zstop")
    plt.title(title); plt.legend()
    plt.tight_layout()
    if out_png: plt.savefig(out_png, dpi=150)
    plt.show()

def util_plot_sharpe_surface(df_grid: pd.DataFrame, lookback: int,
                             zin_grid: list, zout_grid: list,
                             title: str, out_png: str = None):
    d = df_grid[df_grid["lookback"] == lookback].pivot(index="zin", columns="zout", values="Sharpe")
    plt.figure(figsize=(7, 5))
    im = plt.imshow(d.values, aspect="auto", origin="lower",
                    extent=[min(zout_grid), max(zout_grid), min(zin_grid), max(zin_grid)],
                    cmap="viridis")
    plt.colorbar(im, label="Sharpe Ratio")
    plt.xlabel("zout (exit band)")
    plt.ylabel("zin (entry band)")
    plt.title(title)
    # star at maximum
    if np.isfinite(d.values).any():
        idx = np.nanargmax(d.values)
        i, j = np.unravel_index(idx, d.values.shape)
        best_zin  = float(d.index.values[i])
        best_zout = float(d.columns.values[j])
        best_val  = float(d.values[i, j])
        plt.scatter([best_zout], [best_zin], marker="o", color="red")
        plt.text(best_zout, best_zin, f"★ {best_val:.2f}", fontsize=10, color="red",
                 ha="left", va="bottom")
    plt.tight_layout()
    if out_png: plt.savefig(out_png, dpi=150)
    plt.show()


In [ ]:
# === PART 1: Data Ingestion & Cleaning ===
# CSVs source: Stooq (https://stooq.com)

# One-line summary for the console:
SUMMARY = ("Part 1: Import Stooq daily CSVs (AAPL/GOOG/META), align to business days, "
           "winsorize returns (1–99%), rebuild cleaned prices, save CSVs, and plot diagnostics.")
print(SUMMARY)

# ------------------ CONFIG ------------------
FILES = {
    "AAPL": "/content/aapl_us_d.csv",
    "GOOG": "/content/goog_us_d.csv",
    "META": "/content/meta_us_d.csv",
}
BUSINESS_FREQ = "B"        # business days
FFILL_LIMIT   = 3          # forward-fill up to 3 business days
WINSOR_Q      = (0.01, 0.99)
OUT_DIR       = "/content"

# Fallback if globals were not set in 0A:
if "TRADING_DAYS" not in globals():
    TRADING_DAYS = 252

# ------------------ HELPERS ------------------
def find_datetime_column(df: pd.DataFrame) -> str:
    for c in df.columns:
        if any(k in c.lower() for k in ["date","time","timestamp","datetime","unnamed","index"]):
            return c
    return df.columns[0]

def find_price_column(df: pd.DataFrame) -> str:
    prefs = [
        "adj close","adj_close","adjusted close","adjusted_close","adj. close",
        "close","last","price","px_last","value"
    ]
    lower = {c.lower(): c for c in df.columns}
    for key in prefs:
        if key in lower: return lower[key]
    for key in prefs:
        for k,v in lower.items():
            if key in k: return v
    for c in df.columns:
        if c.lower()=="close": return c
    for c in df.columns[::-1]:
        if pd.api.types.is_numeric_dtype(df[c]): return c
    return df.columns[-1]

def load_one_raw(path: str) -> pd.Series:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    df = pd.read_csv(path)
    dcol = find_datetime_column(df)
    pcol = find_price_column(df)
    df[dcol] = pd.to_datetime(df[dcol], errors="coerce", utc=False)
    df = df[[dcol, pcol]].dropna(subset=[dcol]).copy()
    df = df.rename(columns={dcol: "Date", pcol: "Price"})
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
    df = df[df["Price"] > 0]
    df = df.drop_duplicates(subset=["Date"]).sort_values("Date").set_index("Date")
    return df["Price"]

def resample_and_fill(px: pd.Series, limit: int=3) -> pd.Series:
    s = px.asfreq(BUSINESS_FREQ)
    return s.ffill(limit=limit)

def winsorize_returns(ret: pd.Series, q_low=0.01, q_high=0.99) -> pd.Series:
    if ret.dropna().empty: return ret
    lo, hi = ret.quantile([q_low, q_high])
    return ret.clip(lower=lo, upper=hi)

def rebuild_price_from_returns(ret: pd.Series, start_price: float) -> pd.Series:
    return (1.0 + ret.fillna(0.0)).cumprod() * float(start_price)

# ------------------ LOAD & ALIGN ------------------
raw = {}
for ticker, path in FILES.items():
    s_raw  = load_one_raw(path)
    s_fill = resample_and_fill(s_raw, limit=FFILL_LIMIT)
    raw[ticker] = s_fill

prices_raw = pd.DataFrame(raw)

common_idx = prices_raw.dropna().index
if len(common_idx)==0:
    raise ValueError("No common dates across tickers after limited forward-fill. "
                     "Increase FFILL_LIMIT or check input files.")
prices = prices_raw.loc[common_idx]

# ------------------ RETURNS, WINSORIZE, REBUILD ------------------
rets = prices.pct_change()

clean_prices  = {}
clean_returns = {}
for col in prices.columns:
    r   = rets[col]
    r_w = winsorize_returns(r, q_low=WINSOR_Q[0], q_high=WINSOR_Q[1])
    p0  = prices[col].iloc[0]
    p_clean = rebuild_price_from_returns(r_w, start_price=p0)
    clean_prices[col]  = p_clean
    clean_returns[col] = r_w

clean_prices      = pd.DataFrame(clean_prices)
clean_returns     = pd.DataFrame(clean_returns)
clean_prices_norm = clean_prices / clean_prices.iloc[0]

# ------------------ SUMMARY ------------------
summary_rows = []
for col in clean_prices.columns:
    r = clean_returns[col].dropna()
    summary_rows.append({
        "Ticker": col,
        "Start": clean_prices.index.min().date(),
        "End":   clean_prices.index.max().date(),
        "Obs":   int(len(clean_prices)),
        "RetMean(%)":    float(r.mean()*TRADING_DAYS*100) if len(r) else np.nan,
        "RetVolAnn(%)":  float(r.std()*np.sqrt(TRADING_DAYS)*100) if len(r) else np.nan,
        "RetSkew":       float(r.skew()) if len(r) else np.nan,
        "RetKurt":       float(r.kurt()) if len(r) else np.nan,
        "WinsorLow(%)":  float(r.quantile(WINSOR_Q[0])*100) if len(r) else np.nan,
        "WinsorHigh(%)": float(r.quantile(WINSOR_Q[1])*100) if len(r) else np.nan,
        "Source": "Stooq"
    })
summary_df = pd.DataFrame(summary_rows)

# ------------------ SAVE ------------------
clean_prices.reset_index().rename(columns={"index":"Date"}).to_csv(
    os.path.join(OUT_DIR,"clean_prices.csv"), index=False)
clean_returns.reset_index().rename(columns={"index":"Date"}).to_csv(
    os.path.join(OUT_DIR,"clean_logreturns.csv"), index=False)  # simple returns (kept name for compatibility)
clean_prices_norm.reset_index().rename(columns={"index":"Date"}).to_csv(
    os.path.join(OUT_DIR,"clean_prices_normalized.csv"), index=False)
summary_df.to_csv(os.path.join(OUT_DIR,"clean_summary.csv"), index=False)

print("Saved CSVs in /content: clean_prices.csv, clean_logreturns.csv, clean_prices_normalized.csv, clean_summary.csv")
print("Data source: Stooq")

# ------------------ PLOTS (colorful but minimal) ------------------
tickers = list(clean_prices.columns)
colors  = [plt.cm.tab10(i % 10) for i in range(len(tickers))]

# 1) Normalized prices (no big title)
plt.figure(figsize=(11,4))
for i,t in enumerate(tickers):
    plt.plot(clean_prices_norm.index, clean_prices_norm[t].values, label=t, linewidth=2, color=colors[i])
plt.ylabel("Index (×)")
plt.legend(ncol=len(tickers))
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"clean_prices_normalized.png"), dpi=150)
plt.show()

# 2) Returns time series
plt.figure(figsize=(11,4))
for i,t in enumerate(tickers):
    plt.plot(clean_returns.index, clean_returns[t].values, label=t, alpha=0.75, color=colors[i])
plt.ylabel("Return")
plt.legend(ncol=len(tickers))
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"clean_returns_timeseries.png"), dpi=150)
plt.show()

# 3) Returns histogram overlay
plt.figure(figsize=(10,4))
bins = 120
for i,t in enumerate(tickers):
    vals = clean_returns[t].dropna().values
    plt.hist(vals, bins=bins, alpha=0.35, density=True, label=t, color=colors[i])
plt.xlabel("Return"); plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"clean_returns_hist.png"), dpi=150)
plt.show()

# Console peek (kept small)
print("\nHead(clean_prices):")
print(clean_prices.head().round(6))
print("\nSummary:")
print(summary_df.round(4))


In [ ]:
# Part 1: Import Stooq CSVs (AAPL/GOOG/META), align B-days, winsorize returns,
# rebuild clean prices, save cleaned CSVs, and plot diagnostics. Safe to re-run.

import os, io, sys, math, urllib.request
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------- Config ----------
ROOT = Path("/content")           # change if needed
ROOT.mkdir(exist_ok=True, parents=True)

FFILL_LIMIT = 10                  # max consecutive days to forward-fill
WINS_Q = (0.01, 0.99)             # winsor bounds (1% - 99%)
TICKERS = ["AAPL", "GOOG", "META"]
STOOQ = {"AAPL": "aapl.us", "GOOG": "goog.us", "META": "meta.us"}

# Local filenames expected for raw Stooq CSVs
RAW_PATHS = {t: ROOT / f"{STOOQ[t].replace('.', '_')}_d.csv" for t in TICKERS}
# Filenames for the cleaned CSVs we will create
CLEAN_PATHS = {t: ROOT / f"{STOOQ[t].replace('.', '_')}_d_clean.csv" for t in TICKERS}
PLOT_DIR = ROOT / "plots_cleaning"
PLOT_DIR.mkdir(exist_ok=True, parents=True)

def download_stooq_if_missing(ticker: str) -> Path:
    """Download from https://stooq.com/q/d/l/?s=xxx&i=d if the file does not exist."""
    local_path = RAW_PATHS[ticker]
    if local_path.exists():
        return local_path
    code = STOOQ[ticker]
    url = f"https://stooq.com/q/d/l/?s={code}&i=d"
    print(f"[DL] {ticker}: {url} -> {local_path}")
    urllib.request.urlretrieve(url, local_path)
    # basic sanity checks
    df_check = pd.read_csv(local_path)
    lc = [c.lower() for c in df_check.columns]
    if not any(c.startswith("date") for c in lc):
        raise RuntimeError("Downloaded CSV lacks a Date column.")
    if not ("close" in lc or "adj close" in lc or "adj_close" in lc):
        raise RuntimeError("Downloaded CSV lacks a Close/Adj Close column.")
    return local_path

def parse_stooq_series(path: Path) -> pd.Series:
    """Read a Stooq CSV and return a Close series indexed by datetime."""
    df = pd.read_csv(path)
    # detect date column
    dcol = None
    for c in df.columns:
        if c.lower().startswith("date"):
            dcol = c; break
    if dcol is None:
        raise ValueError(f"No Date column in {path}")
    df[dcol] = pd.to_datetime(df[dcol], errors="coerce", utc=False)
    df = df.sort_values(dcol).dropna(subset=[dcol]).set_index(dcol)

    # detect price column
    pcol = None
    for c in ["Close","close","Adj Close","adj close","adj_close","AdjClose","adjclose","price","Price"]:
        if c in df.columns:
            pcol = c; break
    if pcol is None:
        raise ValueError(f"No Close/Adj Close column in {path}")
    s = pd.to_numeric(df[pcol], errors="coerce")
    s = s[~s.index.duplicated(keep="last")].dropna()
    return s

def align_to_business_days(s: pd.Series, limit: int = FFILL_LIMIT) -> pd.Series:
    """Reindex to business days and forward-fill with a cap."""
    idx = pd.date_range(s.index.min(), s.index.max(), freq="B")
    s2 = s.reindex(idx).ffill(limit=limit)
    return s2

def winsorize_returns_from_price(s: pd.Series, q_low=0.01, q_high=0.99):
    """
    Compute pct_change, winsorize at global quantiles, and rebuild a cleaned price series.
    Returns: (raw_returns, winsorized_returns, cleaned_price)
    """
    r = s.pct_change()
    q1, q99 = r.quantile([q_low, q_high])
    r_w = r.clip(q1, q99)
    # rebuild cleaned price starting from the first non-null price
    first_valid = s.dropna().iloc[0]
    clean = (1.0 + r_w).cumprod() * first_valid
    # ensure the very first point equals the original first price
    clean.iloc[0] = first_valid
    return r, r_w, clean

# ---------- Pipeline ----------
raw_price = {}
clean_price = {}
rets_raw = {}
rets_wins = {}

for t in TICKERS:
    # 1) ensure raw CSV is present (download if missing)
    raw_path = download_stooq_if_missing(t)
    # 2) parse Close series
    s_raw = parse_stooq_series(raw_path)
    # 3) align to business days
    s_aligned = align_to_business_days(s_raw)
    # 4) winsorize returns and rebuild cleaned price
    r, r_w, s_clean = winsorize_returns_from_price(s_aligned, *WINS_Q)

    # store
    raw_price[t] = s_aligned
    clean_price[t] = s_clean
    rets_raw[t] = r
    rets_wins[t] = r_w

    # 5) save cleaned CSV (Date, Close)
    df_clean = s_clean.rename("Close").to_frame()
    df_clean.index.name = "Date"
    df_clean.reset_index().to_csv(CLEAN_PATHS[t], index=False)

    # 6) diagnostics plots
    #   a) raw vs cleaned price
    plt.figure(figsize=(9,4.5))
    plt.plot(s_aligned.index, s_aligned.values, label=f"{t} raw")
    plt.plot(s_clean.index,   s_clean.values,   label=f"{t} cleaned", alpha=0.8)
    plt.title(f"{t} — Price: raw vs cleaned (winsorized returns {int(WINS_Q[0]*100)}–{int(WINS_Q[1]*100)}%)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{t}_price_raw_vs_clean.png")
    plt.close()

    #   b) histogram of returns: raw vs winsorized
    plt.figure(figsize=(9,4.5))
    pd.Series(rets_raw[t]).dropna().hist(bins=80, alpha=0.6, label="raw")
    pd.Series(rets_wins[t]).dropna().hist(bins=80, alpha=0.6, label="winsorized")
    plt.title(f"{t} — Daily returns: raw vs winsorized")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{t}_returns_hist.png")
    plt.close()

print("Cleaned CSVs saved to:")
for t in TICKERS:
    print(" -", CLEAN_PATHS[t])

print("Diagnostic plots saved to:", PLOT_DIR)


In [ ]:
# === PART 2: Stationarity & Cointegration ===
# One-line summary :
SUMMARY = ("Part 2: From cleaned prices, build Engle–Granger spreads for chosen pairs, "
           "run ADF/KPSS/coint tests, estimate OU half-life, compute Z-score, "
           "and save CSVs/PNGs for the report.")
print(SUMMARY)

# ------------------ CONFIG ------------------
CLEAN_PATH   = "/content/clean_prices.csv"
OUT_DIR      = "/content/part2_outputs"
PAIRS        = [("META","GOOG"), ("AAPL","META")]
Z_LOOKBACK   = 60          # rolling window for Z-score
Z_IN, Z_OUT  = 2.0, 0.5    # entry/exit for plots only
Z_STOP       = 3.0         # stop band for plots only

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------ HELPERS ------------------
def load_clean_prices(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # pick datetime-like column
    dcol = None
    for c in df.columns:
        if any(k in c.lower() for k in ["date","time","timestamp","unnamed","index"]):
            dcol = c; break
    if dcol is None: dcol = df.columns[0]
    df[dcol] = pd.to_datetime(df[dcol], errors="coerce")
    df = df.set_index(dcol).sort_index()
    # only numeric columns (tickers)
    for c in list(df.columns):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.dropna(how="all", axis=1)

def eg_alpha_beta(y: pd.Series, x: pd.Series):
    X = sm.add_constant(x.values)
    model = sm.OLS(y.values, X, missing="drop").fit()
    return float(model.params[0]), float(model.params[1])

def zscore(series: pd.Series, L: int):
    m = series.rolling(L).mean()
    s = series.rolling(L).std(ddof=0)
    return (series - m) / s

def ou_half_life(spread: pd.Series):
    s = pd.Series(spread).dropna()
    if len(s) < 20:
        return np.nan
    # ΔS_t = a + b * S_{t-1} + ε  →  φ = 1 + b
    ds  = s.diff().dropna()
    lag = s.shift(1).dropna()
    X = sm.add_constant(lag.loc[ds.index].values)
    model = sm.OLS(ds.values, X, missing="drop").fit()
    b = float(model.params[1])
    phi = 1.0 + b
    # Only if 0 < φ < 1 (mean-reversion)
    if not (0.0 < phi < 1.0):
        return np.nan
    return float(-np.log(2.0) / np.log(phi))



In [ ]:
# === PART 3: Signals & Backtest ===
# One-line summary:
print("Part 3: Build Z-score band signals, hedge by EG beta, backtest per pair with vol targeting and turnover costs; save CSVs/PNGs and a per-pair summary.")

# ------------------ CONFIG ------------------
CLEAN_PATH  = "/content/clean_prices.csv"
OUT_DIR     = "/content/part3_outputs"
PAIRS       = [("META","GOOG"), ("AAPL","META")]

Z_LOOKBACK  = 60         # rolling window for Z-score
Z_IN  = 2.0              # entry band  (|Z| >= Z_IN)
Z_OUT = 0.5              # exit band   (|Z| <= Z_OUT)
Z_STOP = 3.0             # safety stop (|Z| >= Z_STOP)

TARGET_VOL_ANN      = 0.10
WINDOW_VOL          = 60
TX_COST_BPS_PER_LEG = 5
MAX_LEV             = 5.0
TRADING_DAYS = globals().get("TRADING_DAYS", 252)
MIN_OBS_REQUIRED    = 200

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------ HELPERS ------------------
def load_clean_prices(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    dcol = None
    for c in df.columns:
        if any(k in c.lower() for k in ["date","time","timestamp","unnamed","index"]):
            dcol = c; break
    if dcol is None: dcol = df.columns[0]
    df[dcol] = pd.to_datetime(df[dcol], errors="coerce")
    df = df.set_index(dcol).sort_index()
    for c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.dropna(how="all", axis=1)

def eg_alpha_beta(y: pd.Series, x: pd.Series):
    X = sm.add_constant(x.values)
    model = sm.OLS(y.values, X, missing="drop").fit()
    return float(model.params[0]), float(model.params[1])

def zscore(series: pd.Series, L: int):
    m = series.rolling(L).mean()
    s = series.rolling(L).std(ddof=0)
    return (series - m) / s

def state_machine(Z: pd.Series, zin: float, zout: float, zstop: float):
    Z = Z.copy()
    st = np.zeros(len(Z), dtype=int)
    for t in range(1, len(Z)):
        z = Z.iloc[t]; prev = st[t-1]
        if np.isnan(z): st[t] = 0; continue
        if prev == 0:
            if z <= -zin: st[t] = 1
            elif z >=  zin: st[t] = -1
            else: st[t] = 0
        else:
            if abs(z) <= zout or abs(z) >= zstop: st[t] = 0
            else: st[t] = prev
    return pd.Series(st, index=Z.index, name="state")

def ann_metrics(daily_ret: pd.Series) -> dict:
    s = pd.Series(daily_ret).dropna()
    if s.empty:
        return {"CAGR":np.nan,"Vol_ann":np.nan,"Sharpe":np.nan,"MaxDD":np.nan,"Turnover":np.nan}
    years = len(s)/TRADING_DAYS
    cagr  = (1+s).prod()**(1/years)-1 if years>0 else np.nan
    vol   = s.std()*np.sqrt(TRADING_DAYS)
    sharpe= (s.mean()*np.sqrt(TRADING_DAYS))/(s.std()+1e-12)
    eq    = (1+s).cumprod()
    dd    = eq/eq.cummax()-1
    return {"CAGR":float(cagr),"Vol_ann":float(vol),"Sharpe":float(sharpe),"MaxDD":float(dd.min())}

def backtest_pair(prices_df: pd.DataFrame, symY: str, symX: str,
                  lookback: int, zin: float, zout: float, zstop: float,
                  target_vol_ann: float, window_vol: int,
                  tx_cost_bps_per_leg: float, max_lev: float):
    px = prices_df[[symY, symX]].dropna()
    if len(px) < MIN_OBS_REQUIRED:
        return None

    logp = np.log(px)
    rets = px.pct_change()

    y, x = logp[symY], logp[symX]
    rY, rX = rets[symY], rets[symX]
    alpha, beta = eg_alpha_beta(y, x)

    spread = y - (alpha + beta*x)
    Z = zscore(spread, lookback)

    state = state_machine(Z, zin, zout, zstop)

    ret0 = state.shift(1) * (rY - beta*rX)
    ret0 = ret0.fillna(0.0)

    vol_roll = ret0.rolling(window_vol).std()*np.sqrt(TRADING_DAYS)
    weight = (target_vol_ann/vol_roll.replace(0,np.nan)).clip(upper=max_lev).fillna(0.0)
    w = weight.shift(1).fillna(0.0)

    ret_gross = w*ret0

    # turnover-based costs
    position = w * state.shift(1).fillna(0.0)  # signed exposure
    turnover = position.diff().abs().fillna(position.abs())
    cost_unit = (tx_cost_bps_per_leg/10000.0)*2
    cost = cost_unit * turnover

    ret_net = (ret_gross - cost).fillna(0.0)

    out = pd.DataFrame({
        "state": state,
        "weight": w,
        "ret_gross": ret_gross,
        "ret_net": ret_net,
        "turnover": turnover,
        "Z": Z,
        "spread": spread
    }).dropna(subset=["ret_net"])

    metrics = ann_metrics(out["ret_net"])
    metrics.update({"alpha":alpha, "beta":beta, "avg_turnover": float(turnover.mean())})
    return out, metrics

def plot_equity_with_drawdown(eq: pd.Series, title: str, out_png: str = None):
    eq = eq.dropna()
    rollmax = eq.cummax()
    dd = eq/rollmax - 1.0
    fig, ax1 = plt.subplots(figsize=(10,4))
    ax1.plot(eq.index, eq.values, label="Equity")
    ax1.set_title(title); ax1.set_ylabel("Cumulative (×)")
    ax1.legend(loc="upper left")
    ax2 = ax1.twinx()
    ax2.fill_between(dd.index, dd.values, 0, alpha=0.2, step="pre")
    ax2.set_ylabel("Drawdown")
    plt.tight_layout()
    if out_png: plt.savefig(out_png, dpi=150)
    plt.show()

def plot_zscore(Z: pd.Series, zin: float, zout: float, zstop: float, title: str, out_png: str = None):
    Z = Z.dropna()
    plt.figure(figsize=(10,4))
    plt.plot(Z.index, Z.values, label="Z-score")
    plt.axhline(0, linestyle="--", label="mean")
    plt.axhline( zin, linestyle="--", label="+zin")
    plt.axhline(-zin, linestyle="--", label="-zin")
    plt.axhline( zout, linestyle=":",  label="+zout")
    plt.axhline(-zout, linestyle=":",  label="-zout")
    plt.axhline( zstop, linestyle=":", alpha=0.7, label="+zstop")
    plt.axhline(-zstop, linestyle=":", alpha=0.7, label="-zstop")
    plt.title(title); plt.legend(ncol=7, fontsize=8)
    plt.tight_layout()
    if out_png: plt.savefig(out_png, dpi=150)
    plt.show()

# ------------------ RUN ------------------
px = load_clean_prices(CLEAN_PATH)

summary_rows = []
for (symY, symX) in PAIRS:
    res = backtest_pair(px, symY, symX, Z_LOOKBACK, Z_IN, Z_OUT, Z_STOP,
                        TARGET_VOL_ANN, WINDOW_VOL, TX_COST_BPS_PER_LEG, MAX_LEV)
    if res is None:
        print(f"[WARN] Not enough data for {symY}-{symX}.")
        continue
    df_bt, met = res

    # Save per-pair BT CSV (used later by REAL METRICS)
    csv_path = os.path.join(OUT_DIR, f"bt_{symY}_{symX}.csv")
    df_bt.reset_index().rename(columns={"index":"Date"}).to_csv(csv_path, index=False)

    # Equity plot with DD
    eq = (1+df_bt["ret_net"]).cumprod()
    plot_equity_with_drawdown(eq, f"Equity — {symY} vs {symX} (vol-targeted, net of costs)",
                              os.path.join(OUT_DIR, f"equity_{symY}_{symX}.png"))

    # Z-score plot with bands
    plot_zscore(df_bt["Z"], Z_IN, Z_OUT, Z_STOP, f"Z-score — {symY} vs {symX}",
                os.path.join(OUT_DIR, f"z_{symY}_{symX}.png"))

    # Summary row
    summary_rows.append({
        "Pair": f"{symY}-{symX}",
        "alpha": met["alpha"], "beta": met["beta"],
        "CAGR": met["CAGR"], "Vol_ann": met["Vol_ann"],
        "Sharpe": met["Sharpe"], "MaxDD": met["MaxDD"],
        "avg_turnover": met["avg_turnover"]
    })

# Save summary table
summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(OUT_DIR, "part3_summary.csv")
summary_df.to_csv(summary_csv, index=False)

print("\nSaved Part 3 artifacts in", OUT_DIR)
print("- bt_<Y>_<X>.csv  (includes ret_net for REAL METRICS)")
print("- equity_<Y>_<X>.png")
print("- z_<Y>_<X>.png")
print("- part3_summary.csv")
print("\nHead(part3_summary):")
print(summary_df.round(4).head())


In [ ]:
# === PART 4: Portfolio Construction & Risk ===
print("Part 4: Build a multi-pair portfolio (equal-risk via inverse rolling vol), compute VaR/ES, regimes, stress windows, and export metrics/plots/CSV for Part 6 and REAL METRICS.")

# ------------------ CONFIG ------------------
PAIRS = [("META","GOOG"), ("AAPL","META")]
PART3_DIR   = "/content/part3_outputs"
ROOT_DIR    = "/content"
OUT_DIR     = "/content/part4_outputs"

ROLL_VOL_W  = 60     # window for per-pair rolling vol (days)
ROLL_SHRP_W = 126    # window for rolling Sharpe on the portfolio
ALPHA_VAR   = 0.95   # VaR/ES level

TRADING_DAYS = globals().get("TRADING_DAYS", 252)
STRESS_WINDOWS = globals().get("STRESS_WINDOWS", [
    ("OilCrash_2014_2016", "2014-06-01", "2016-02-29"),
    ("Russia_Ukraine_2022", "2022-01-01", "2022-12-31"),
])

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------ HELPERS ------------------
def read_bt_pair(y, x):
    """Load per-pair backtest CSV produced in Part 3. Try root and part3_outputs."""
    fn = f"bt_{y}_{x}.csv"
    p_root = os.path.join(ROOT_DIR, fn)
    p_dir  = os.path.join(PART3_DIR, fn)
    path = p_root if os.path.exists(p_root) else p_dir
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing backtest file for {y}-{x}. Looked at:\n- {p_root}\n- {p_dir}")
    df = pd.read_csv(path)
    # infer datetime column
    dcol = None
    for c in df.columns:
        if any(k in c.lower() for k in ["date","time","timestamp","unnamed","index"]):
            dcol = c; break
    if dcol is None: dcol = df.columns[0]
    df[dcol] = pd.to_datetime(df[dcol], errors="coerce")
    df = df.set_index(dcol).sort_index()
    # Ensure required cols exist
    for col in ["ret_net"]:
        if col not in df.columns:
            raise ValueError(f"Column 'ret_net' not found in {path}")
    return df

def ann_metrics(daily_ret: pd.Series):
    s = pd.Series(daily_ret).dropna()
    if s.empty:
        return {"CAGR":np.nan, "Vol_ann":np.nan, "Sharpe":np.nan, "MaxDD":np.nan}
    years = len(s)/TRADING_DAYS
    cagr  = (1+s).prod()**(1/years)-1 if years>0 else np.nan
    vol   = s.std()*np.sqrt(TRADING_DAYS)
    sharpe= (s.mean()*np.sqrt(TRADING_DAYS))/(s.std()+1e-12)
    eq    = (1+s).cumprod()
    dd    = eq/eq.cummax()-1
    return {"CAGR":float(cagr), "Vol_ann":float(vol), "Sharpe":float(sharpe), "MaxDD":float(dd.min())}

def hist_var_es(daily_ret: pd.Series, alpha=0.95):
    s = pd.Series(daily_ret).dropna().values
    if s.size==0:
        return {"VaR":np.nan, "ES":np.nan}
    var = float(np.quantile(s, 1-alpha))
    es  = float(s[s<=var].mean()) if np.any(s<=var) else var
    return {"VaR":var, "ES":es}

def plot_equity_with_drawdown(eq: pd.Series, title: str, out_png: str = None):
    eq = eq.dropna()
    rollmax = eq.cummax()
    dd = eq/rollmax - 1.0
    fig, ax1 = plt.subplots(figsize=(10,4))
    ax1.plot(eq.index, eq.values, label="Equity")
    ax1.set_title(title); ax1.set_ylabel("Cumulative (×)")
    ax1.legend(loc="upper left")
    ax2 = ax1.twinx()
    ax2.fill_between(dd.index, dd.values, 0, alpha=0.2, step="pre")
    ax2.set_ylabel("Drawdown")
    plt.tight_layout()
    if out_png: plt.savefig(out_png, dpi=150)
    plt.show()

# ------------------ LOAD PAIR RETURNS ------------------
# Assemble a DataFrame with columns like "META-GOOG", each = ret_net series
pair_rets = {}
for (y,x) in PAIRS:
    df = read_bt_pair(y,x)
    pair_name = f"{y}-{x}"
    pair_rets[pair_name] = df["ret_net"].copy()

rets_df = pd.DataFrame(pair_rets).dropna(how="all")
rets_df = rets_df.fillna(0.0)  # conservative; or inner-join could be used depending on preference

# ------------------ EQUAL-RISK DYNAMIC WEIGHTS (inverse rolling vol) ------------------
# σ_i(t) = rolling std_i * sqrt(252);  w_i(t) ∝ 1/σ_i(t),  normalized to sum=1
rolling_sigma = rets_df.rolling(ROLL_VOL_W).std() * np.sqrt(TRADING_DAYS)
w_raw = 1.0 / rolling_sigma.replace(0, np.nan)
w = w_raw.div(w_raw.sum(axis=1), axis=0)

# For initial window where rolling vol not available, use equal weights
equal_w = pd.Series(1.0/len(PAIRS), index=rets_df.columns)
w = w.fillna(method="ffill").fillna(equal_w)

# ------------------ PORTFOLIO RETURNS & EQUITY ------------------
port_ret = (w * rets_df).sum(axis=1)
port_eq  = (1 + port_ret).cumprod()

# ------------------ METRICS ------------------
met = ann_metrics(port_ret)
vres = hist_var_es(port_ret, alpha=ALPHA_VAR)
metrics_row = {**met, f"{ALPHA_VAR:.2f}_VaR": vres["VaR"], f"{ALPHA_VAR:.2f}_ES": vres["ES"]}
metrics_df = pd.DataFrame([metrics_row])

# ------------------ REGIMES (tercis de vol) ------------------
roll_vol = port_ret.rolling(ROLL_VOL_W).std() * np.sqrt(TRADING_DAYS)
q1, q2 = roll_vol.quantile([1/3, 2/3])
def regime_label(v):
    if pd.isna(v): return np.nan
    if v <= q1: return "LowVol"
    if v <= q2: return "MidVol"
    return "HighVol"
regime = roll_vol.apply(regime_label)
# Per-regime summary (share of time, Sharpe, Vol, mean)
reg_rows = []
for rg in ["LowVol","MidVol","HighVol"]:
    mask = (regime==rg)
    s = port_ret[mask].dropna()
    if s.empty:
        reg_rows.append({"Regime":rg,"Share(%)":0.0,"Sharpe":np.nan,"Vol_ann":np.nan,"Mean_ann(%)":np.nan})
        continue
    share = 100.0 * mask.mean()
    vol   = s.std()*np.sqrt(TRADING_DAYS)
    sharpe= (s.mean()*np.sqrt(TRADING_DAYS))/(s.std()+1e-12)
    mean_ann = s.mean()*TRADING_DAYS*100.0
    reg_rows.append({"Regime":rg,"Share(%)":share,"Sharpe":float(sharpe),"Vol_ann":float(vol),"Mean_ann(%)":float(mean_ann)})
regime_df = pd.DataFrame(reg_rows)

# ------------------ STRESS WINDOWS ------------------
stress_rows = []
for name, a, b in STRESS_WINDOWS:
    s = port_ret.loc[a:b].dropna()
    if s.empty:
        stress_rows.append({"Window":name,"Sharpe":np.nan,"MaxDD(%)":np.nan,"CAGR(%)":np.nan})
        continue
    # metrics in window
    vol = s.std()*np.sqrt(TRADING_DAYS)
    sharpe = (s.mean()*np.sqrt(TRADING_DAYS))/(s.std()+1e-12)
    yrs = len(s)/TRADING_DAYS
    cagr = (1+s).prod()**(1/yrs)-1 if yrs>0 else np.nan
    eq = (1+s).cumprod()
    dd = eq/eq.cummax()-1
    stress_rows.append({"Window":name, "Sharpe":float(sharpe), "MaxDD(%)":float(dd.min()*100.0), "CAGR(%)":float(cagr*100.0)})
stress_df = pd.DataFrame(stress_rows)

# ------------------ SAVE CORE ARTIFACTS ------------------
# Daily returns for REAL METRICS
port_ret_df = pd.DataFrame({"port_ret": port_ret})
# Save to part4 dir
port_ret_df.reset_index().rename(columns={"index":"Date"}).to_csv(os.path.join(OUT_DIR, "portfolio_daily_returns.csv"), index=False)
metrics_df.to_csv(os.path.join(OUT_DIR, "portfolio_metrics.csv"), index=False)
regime_df.to_csv(os.path.join(OUT_DIR, "portfolio_regime_breakdown.csv"), index=False)
stress_df.to_csv(os.path.join(OUT_DIR, "portfolio_stress_metrics.csv"), index=False)
# Save also to /content (root) for compatibility with REAL METRICS
port_ret_df.reset_index().rename(columns={"index":"Date"}).to_csv(os.path.join(ROOT_DIR, "portfolio_daily_returns.csv"), index=False)
metrics_df.to_csv(os.path.join(ROOT_DIR, "portfolio_metrics.csv"), index=False)
regime_df.to_csv(os.path.join(ROOT_DIR, "portfolio_regime_breakdown.csv"), index=False)
stress_df.to_csv(os.path.join(ROOT_DIR, "portfolio_stress_metrics.csv"), index=False)

# Also place bt_* copies at root if missing (to satisfy REAL METRICS that reads /content/bt_*.csv)
for (y,x) in PAIRS:
    src = os.path.join(PART3_DIR, f"bt_{y}_{x}.csv")
    dst = os.path.join(ROOT_DIR,    f"bt_{y}_{x}.csv")
    if os.path.exists(src) and not os.path.exists(dst):
        try:
            pd.read_csv(src).to_csv(dst, index=False)
        except Exception:
            pass

# ------------------ PLOTS ------------------
plot_equity_with_drawdown(port_eq, "Portfolio Equity (equal-risk, inverse rolling vol)", os.path.join(ROOT_DIR,"portfolio_equity.png"))

# Rolling Sharpe
roll_sharpe = (port_ret.rolling(ROLL_SHRP_W).mean()*np.sqrt(TRADING_DAYS)) / (port_ret.rolling(ROLL_SHRP_W).std()+1e-12)
plt.figure(figsize=(10,4))
plt.plot(roll_sharpe.index, roll_sharpe.values)
plt.title(f"Rolling Sharpe ({ROLL_SHRP_W}d)")
plt.tight_layout()
plt.savefig(os.path.join(ROOT_DIR,"portfolio_rolling_sharpe.png"), dpi=150)
plt.show()

# Rolling Annualized Vol
roll_vol_ann = port_ret.rolling(ROLL_VOL_W).std()*np.sqrt(TRADING_DAYS)
plt.figure(figsize=(10,4))
plt.plot(roll_vol_ann.index, roll_vol_ann.values)
plt.title(f"Rolling Annualized Vol ({ROLL_VOL_W}d)")
plt.tight_layout()
plt.savefig(os.path.join(ROOT_DIR,"portfolio_rolling_vol.png"), dpi=150)
plt.show()

# ------------------ PRINTS ------------------
print("\nSaved Part 4 artifacts:")
print(f"- {ROOT_DIR}/portfolio_daily_returns.csv")
print(f"- {ROOT_DIR}/portfolio_metrics.csv")
print(f"- {ROOT_DIR}/portfolio_regime_breakdown.csv")
print(f"- {ROOT_DIR}/portfolio_stress_metrics.csv")
print(f"- {ROOT_DIR}/portfolio_equity.png")
print(f"- {ROOT_DIR}/portfolio_rolling_sharpe.png")
print(f"- {ROOT_DIR}/portfolio_rolling_vol.png")
print("\nPortfolio metrics (rounded):")
print(metrics_df.round(4))
print("\nRegime breakdown (rounded):")
print(regime_df.round(4))
print("\nStress metrics (rounded):")
print(stress_df.round(4))


In [ ]:
# === PART 5: Sharpe Surfaces + Walk-Forward ===
print("Part 5: Grid Sharpe surfaces and walk-forward (≈2y train / 1y test) with turnover costs; save CSVs and PNGs per pair.")

# ------------------ CONFIG ------------------
CLEAN_PATH = "/content/clean_prices.csv"
OUT_DIR    = "/content/part5_outputs"
PAIRS      = [("META","GOOG"), ("AAPL","META")]

# Parameter grids
LOOKBACK_GRID = [40, 60, 90]
ZIN_GRID      = [1.5, 2.0, 2.5, 3.0]
ZOUT_GRID     = [0.25, 0.5, 0.75, 1.0]

# Stops / risk
ZSTOP_MODE  = "fixed"   # "fixed" or "double_zin"
ZSTOP_FIXED = 3.0

TARGET_VOL_ANN      = 0.10
WINDOW_VOL          = 60
TX_COST_BPS_PER_LEG = 5
MAX_LEVERAGE        = 5.0

TRADING_DAYS        = globals().get("TRADING_DAYS", 252)
MIN_OBS_REQUIRED    = 200

# Walk-forward windows by number of observations (≈252 trading days/year)
TRAIN_DAYS = 504   # ~ 2 years
TEST_DAYS  = 252   # ~ 1 year

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------ HELPERS ------------------
def load_clean_prices(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    dcol = None
    for c in df.columns:
        if any(k in c.lower() for k in ["date","time","timestamp","unnamed","index"]):
            dcol = c; break
    if dcol is None: dcol = df.columns[0]
    df[dcol] = pd.to_datetime(df[dcol], errors="coerce")
    df = df.set_index(dcol).sort_index()
    for c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.dropna(how="all", axis=1)

def eg_alpha_beta(y: pd.Series, x: pd.Series):
    X = sm.add_constant(x.values)
    model = sm.OLS(y.values, X, missing="drop").fit()
    return float(model.params[0]), float(model.params[1])

def zscore(series: pd.Series, L: int):
    m = series.rolling(L).mean()
    s = series.rolling(L).std(ddof=0)
    return (series - m) / s

def state_machine(Z: pd.Series, zin: float, zout: float, zstop: float):
    Z = Z.copy()
    st = np.zeros(len(Z), dtype=int)
    for t in range(1, len(Z)):
        z = Z.iloc[t]; prev = st[t-1]
        if np.isnan(z): st[t] = 0; continue
        if prev == 0:
            if z <= -zin: st[t] = 1
            elif z >=  zin: st[t] = -1
            else: st[t] = 0
        else:
            if abs(z) <= zout or abs(z) >= zstop: st[t] = 0
            else: st[t] = prev
    return pd.Series(st, index=Z.index, name="state")

def ann_metrics(daily_ret: pd.Series):
    s = pd.Series(daily_ret).dropna()
    if s.empty:
        return {"CAGR":np.nan,"Vol_ann":np.nan,"Sharpe":np.nan,"MaxDD":np.nan}
    years = len(s)/TRADING_DAYS
    cagr  = (1+s).prod()**(1/years)-1 if years>0 else np.nan
    vol   = s.std()*np.sqrt(TRADING_DAYS)
    shrp  = (s.mean()*np.sqrt(TRADING_DAYS))/(s.std()+1e-12)
    eq    = (1+s).cumprod()
    dd    = eq/eq.cummax()-1
    return {"CAGR":float(cagr), "Vol_ann":float(vol), "Sharpe":float(shrp), "MaxDD":float(dd.min())}

def backtest_pair_params(prices_df: pd.DataFrame, symY: str, symX: str, lookback: int,
                         zin: float, zout: float, zstop_mode="fixed", zstop_fixed=3.0,
                         target_vol_ann=0.10, window_vol=60,
                         tx_cost_bps_per_leg=5, max_leverage=5.0):
    px = prices_df[[symY, symX]].dropna()
    if len(px) < MIN_OBS_REQUIRED:
        return pd.Series(dtype=float), {"CAGR":np.nan,"Vol_ann":np.nan,"Sharpe":np.nan,"MaxDD":np.nan}
    logp = np.log(px)
    rets = px.pct_change()
    y, x = logp[symY], logp[symX]
    rY, rX = rets[symY], rets[symX]
    alpha, beta = eg_alpha_beta(y, x)
    spread = y - (alpha + beta*x)
    Z = zscore(spread, lookback)
    zstop = zstop_fixed if zstop_mode=="fixed" else (2.0*zin)
    state = state_machine(Z, zin, zout, zstop)

    ret0 = state.shift(1) * (rY - beta*rX)
    ret0 = ret0.fillna(0.0)

    vol_roll = ret0.rolling(window_vol).std()*np.sqrt(TRADING_DAYS)
    weight = (target_vol_ann/vol_roll.replace(0,np.nan)).clip(upper=max_leverage).fillna(0.0)
    w = weight.shift(1).fillna(0.0)

    ret_gross = w*ret0

    # turnover-based costs (Δposition)
    position = (state.shift(1).fillna(0.0)) * w
    turnover = position.diff().abs().fillna(position.abs())
    cost_unit = (tx_cost_bps_per_leg/10000.0)*2
    cost = cost_unit * turnover

    ret_net = (ret_gross - cost).fillna(0.0)
    return ret_net, ann_metrics(ret_net)

def save_sharpe_surface(prices_df: pd.DataFrame, symY: str, symX: str,
                        lookbacks, zins, zouts,
                        zstop_mode, zstop_fixed,
                        target_vol_ann, window_vol, tx_cost_bps_per_leg, max_leverage,
                        out_dir):
    rows = []
    for L in lookbacks:
        for zin in zins:
            for zout in zouts:
                if zout >= zin:
                    rows.append({"pair":f"{symY}-{symX}","lookback":L,"zin":zin,"zout":zout,"Sharpe":np.nan})
                    continue
                ret, met = backtest_pair_params(prices_df, symY, symX, L, zin, zout,
                                                zstop_mode, zstop_fixed,
                                                target_vol_ann, window_vol, tx_cost_bps_per_leg, max_leverage)
                rows.append({"pair":f"{symY}-{symX}","lookback":L,"zin":zin,"zout":zout,"Sharpe":met["Sharpe"]})
    df_grid = pd.DataFrame(rows)
    df_grid.to_csv(os.path.join(out_dir, f"grid_sharpe_{symY}_{symX}.csv"), index=False)

    # Heatmap per lookback with star at optimum
    for L in lookbacks:
        d = df_grid[df_grid["lookback"]==L].pivot(index="zin", columns="zout", values="Sharpe")
        plt.figure(figsize=(7,5))
        im = plt.imshow(d.values, aspect="auto", origin="lower",
                        extent=[min(ZOUT_GRID), max(ZOUT_GRID), min(ZIN_GRID), max(ZIN_GRID)],
                        cmap="viridis")
        plt.colorbar(im, label="Sharpe")
        plt.xlabel("zout (exit)"); plt.ylabel("zin (entry)")
        plt.title(f"Sharpe Surface — {symY} vs {symX} | L={L}")
        if np.isfinite(d.values).any():
            idx = np.nanargmax(d.values)
            i,j = np.unravel_index(idx, d.values.shape)
            best_zin  = float(d.index.values[i])
            best_zout = float(d.columns.values[j])
            best_val  = float(d.values[i,j])
            plt.scatter([best_zout],[best_zin], marker="o", color="red")
            plt.text(best_zout, best_zin, f"★ {best_val:.2f}", fontsize=10, color="red",
                     ha="left", va="bottom")
        out_png = os.path.join(out_dir, f"sharpe_surface_{symY}_{symX}_L{L}.png")
        plt.tight_layout(); plt.savefig(out_png, dpi=150); plt.show()
    return df_grid

def walk_forward(prices_df: pd.DataFrame, symY: str, symX: str,
                 lookbacks, zins, zouts,
                 zstop_mode, zstop_fixed,
                 target_vol_ann, window_vol, tx_cost_bps_per_leg, max_leverage,
                 train_days, test_days, out_dir):
    px = prices_df[[symY, symX]].dropna()
    idx = px.index.to_list()
    n = len(idx)
    if n < (train_days + test_days + 20):
        raise ValueError(f"Not enough data for walk-forward on {symY}-{symX}.")

    folds, oos_list = [], []
    start = 0
    while start + train_days + test_days <= n:
        tr_slice = slice(start, start + train_days)
        te_slice = slice(start + train_days, start + train_days + test_days)

        pr_tr = px.iloc[tr_slice]
        pr_te = px.iloc[te_slice]

        # Gridsearch on train
        best = {"Sharpe": -1e9, "L": None, "zin": None, "zout": None}
        for L in lookbacks:
            for zin in zins:
                for zout in zouts:
                    if zout >= zin:
                        continue
                    ret_tr, met_tr = backtest_pair_params(pr_tr, symY, symX, L, zin, zout,
                                                          zstop_mode, zstop_fixed,
                                                          target_vol_ann, window_vol, tx_cost_bps_per_leg, max_leverage)
                    if np.isfinite(met_tr["Sharpe"]) and met_tr["Sharpe"] > best["Sharpe"]:
                        best = {"Sharpe": met_tr["Sharpe"], "L": L, "zin": zin, "zout": zout}

        # Test with best params
        ret_te, met_te = backtest_pair_params(pr_te, symY, symX, best["L"], best["zin"], best["zout"],
                                              zstop_mode, zstop_fixed,
                                              target_vol_ann, window_vol, tx_cost_bps_per_leg, max_leverage)
        ret_te.name = f"OOS_{symY}_{symX}"
        oos_list.append(ret_te)

        folds.append({
            "pair": f"{symY}-{symX}",
            "train_start": pr_tr.index[0], "train_end": pr_tr.index[-1],
            "test_start": pr_te.index[0],  "test_end":  pr_te.index[-1],
            "L*": best["L"], "zin*": best["zin"], "zout*": best["zout"],
            "train_Sharpe": best["Sharpe"], **ann_metrics(ret_te)
        })

        start += test_days  # advance by one test window

    folds_df = pd.DataFrame(folds)
    oos


In [ ]:
# === REAL METRICS (SEPARATE): Executive Summary + Overleaf fragments ===

pairs = [("META","GOOG"), ("AAPL","META")]
pm_path = "/content/portfolio_metrics.csv"
port_ret_path = "/content/portfolio_daily_returns.csv"

def read_with_datetime_index(path):
    df = pd.read_csv(path)
    date_col = None
    for c in df.columns:
        if any(k in c.lower() for k in ["date","time","timestamp","unnamed","index"]):
            date_col = c; break
    if date_col is None: date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.set_index(date_col).sort_index()
    return df

def read_pair_metrics(y, x):
    df = read_with_datetime_index(f"/content/bt_{y}_{x}.csv")
    ret = df["ret_net"].dropna()
    eq = (1+ret).cumprod()
    dd = eq/eq.cummax()-1
    maxdd = float(dd.min()) if len(dd) else np.nan
    vol = ret.std()*np.sqrt(252) if len(ret) else np.nan
    shrp = (ret.mean()*np.sqrt(252))/(ret.std()+1e-12) if len(ret) else np.nan
    years = len(ret)/252 if len(ret) else np.nan
    cagr = (1+ret).prod()**(1/years)-1 if years and years>0 else np.nan
    q = np.quantile(ret, 0.05) if len(ret) else np.nan
    es = ret[ret<=q].mean() if len(ret) and (ret<=q).any() else q
    return {"CAGR":cagr,"Vol_ann":vol,"Sharpe":shrp,"MaxDD":maxdd,"VaR95":q,"ES95":es}

# Portfolio metrics
pm = pd.read_csv(pm_path)
def first_val(col):
    cols = [c for c in pm.columns if col.lower() in c.lower()]
    return float(pm[cols[0]].iloc[0]) if cols else np.nan
port = {
    "CAGR": first_val("CAGR"),
    "Vol_ann": first_val("Vol"),
    "Sharpe": first_val("Sharpe"),
    "MaxDD": first_val("MaxDD"),
    "VaR95": first_val("VaR"),
    "ES95": first_val("ES"),
}

# Consolidated table
rows = []
for (y,x) in pairs:
    m = read_pair_metrics(y,x)
    rows.append([f"{y}-{x}", m["CAGR"], m["Vol_ann"], m["Sharpe"], m["MaxDD"], m["VaR95"], m["ES95"]])
rows.append(["Portfolio", port["CAGR"], port["Vol_ann"], port["Sharpe"], port["MaxDD"], port["VaR95"], port["ES95"]])
tbl = pd.DataFrame(rows, columns=["Pair / Portfolio","CAGR","Vol (ann)","Sharpe","MaxDD","VaR(95%)","ES(95%)"])

# Stress tests (do seu PDF)
pr = read_with_datetime_index(port_ret_path)["port_ret"].dropna()
def stress_port(start,end):
    s = pr.loc[start:end]
    if s.empty: return {"Sharpe":np.nan,"MaxDD":np.nan,"CAGR":np.nan}
    vol = s.std()*np.sqrt(252); shrp = (s.mean()*np.sqrt(252))/(s.std()+1e-12)
    years = len(s)/252; cagr = (1+s).prod()**(1/years)-1 if years>0 else np.nan
    eq = (1+s).cumprod(); dd = eq/eq.cummax()-1
    return {"Sharpe":float(shrp),"MaxDD":float(dd.min()),"CAGR":float(cagr)}

stress_windows = [
    ("Oil Crash 2014–2016",  "2014-06-01", "2016-02-29"),
    ("Russia–Ukraine 2022",  "2022-01-01", "2022-12-31"),
]
stress_lines = []
for name,a,b in stress_windows:
    m = stress_port(a,b)
    stress_lines.append(f"- **{name}**: Sharpe {m['Sharpe']:.2f}, MaxDD {m['MaxDD']*100:.2f}%, CAGR {m['CAGR']*100:.2f}%.")

def pct(x): return f"{x*100:.2f}%"
ES = []
ES.append("**Pairs Trading Strategy – Final Results**\n")
ES.append("- **Methodology**: Cointegration-based pairs trading (Engle–Granger), Z-score mean reversion, half-life rebalancing, stop-bands.")
ES.append("- **Assets tested**: META–GOOG; AAPL–META.")
ES.append("- **Portfolio allocation**: equal-risk dynamic (volatility scaling).")
ES.append("\n**Performance highlights (portfolio level):**")
ES.append(f"- CAGR: **{pct(port['CAGR'])}**")
ES.append(f"- Annualized Volatility: **{pct(port['Vol_ann'])}**")
ES.append(f"- Sharpe ratio: **{port['Sharpe']:.2f}**")
ES.append(f"- Max Drawdown: **{pct(port['MaxDD'])}**")
ES.append(f"- VaR (95%): **{pct(port['VaR95'])}** | ES (95%): **{pct(port['ES95'])}**")
ES.append("\n**Stress Tests (aligned with PDF):**")
ES += stress_lines
ES.append("\n**Economic rationale:**")
ES.append("- META–GOOG: digital advertising peers; similar macro/regulatory drivers → strong cointegration candidate.")
ES.append("- AAPL–META: large-cap consumer tech with partially complementary cycles → diversification.")
ES.append('\n**Key insight**:\n> *This strategy is robust across specified stress regimes, but performance could be further improved by dynamic threshold calibration, liquidity filters, and transaction cost modeling.*')

# Print Markdown
print("=== Executive Summary (Markdown) ===\n")
print("\n".join(ES))

# Save Overleaf fragments
def latex_escape(s):
    return (s.replace('%','\\%').replace('_','\\_')
              .replace('&','\\&').replace('#','\\#'))
with open("/content/executive_summary.tex","w",encoding="utf-8") as f:
    f.write("\\section*{Executive Summary}\n")
    for line in ES:
        f.write("\\noindent " + latex_escape(line.replace("- **","\\textbf{-- }")) + "\\\\\n")

tbl_tex = tbl.copy()
for c in tbl_tex.columns[1:]:
    if c.lower().startswith("vol"):        tbl_tex[c] = (tbl_tex[c]*100).round(2).astype(str) + "\\%"
    elif c.lower() in ["cagr","maxdd"] or "var" in c.lower() or "es" in c.lower():
        tbl_tex[c] = (tbl_tex[c]*100).round(2).astype(str) + "\\%"

with open("/content/consolidated_metrics_table.tex","w",encoding="utf-8") as f:
    cols = list(tbl_tex.columns)
    f.write("\\begin{table}[h]\\centering\n\\begin{tabular}{lrrrrrr}\\hline\n")
    f.write(" & ".join(cols) + " \\\\\\hline\n")
    for _,row in tbl_tex.iterrows():
        f.write(" & ".join(str(row[c]) for c in cols) + " \\\\\n")
    f.write("\\hline\n\\end{tabular}\n\\caption{Consolidated performance metrics.}\n\\end{table}\n")

print("\n[OK] Saved:")
print("- /content/executive_summary.tex")
print("- /content/consolidated_metrics_table.tex")


In [ ]:
# === PART 6 (MINIMAL): Export for Overleaf ===
print("Part 6: bundle .tex + figures into export_overleaf/ and create insert_me.tex")

import os, shutil, zipfile

ROOT   = "/content"
OUTDIR = "/content/export_overleaf"
os.makedirs(OUTDIR, exist_ok=True)

# Required .tex fragments produced by REAL METRICS
need = [
    "executive_summary.tex",
    "consolidated_metrics_table.tex",
]
missing = []
for fn in need:
    src = os.path.join(ROOT, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUTDIR, fn))
    else:
        missing.append(fn)

# Optional figures (copy if they exist)
fig_candidates = [
    "portfolio_equity.png",
    "portfolio_rolling_sharpe.png",
    "portfolio_rolling_vol.png",
    # part3
    "part3_outputs/equity_META_GOOG.png",
    "part3_outputs/z_META_GOOG.png",
    "part3_outputs/equity_AAPL_META.png",
    "part3_outputs/z_AAPL_META.png",
    # part5 (if present)
    "part5_outputs/sharpe_surface_META_GOOG_L40.png",
    "part5_outputs/sharpe_surface_META_GOOG_L60.png",
    "part5_outputs/sharpe_surface_META_GOOG_L90.png",
    "part5_outputs/sharpe_surface_AAPL_META_L40.png",
    "part5_outputs/sharpe_surface_AAPL_META_L60.png",
    "part5_outputs/sharpe_surface_AAPL_META_L90.png",
]
copied = []
for rel in fig_candidates:
    src = os.path.join(ROOT, rel)
    if os.path.exists(src):
        dst = os.path.join(OUTDIR, os.path.basename(src))
        shutil.copy2(src, dst)
        copied.append(os.path.basename(src))

# Build insert_me.tex that you \input in main.tex
lines = []
lines.append("% auto-generated — include this file in main.tex")
if "executive_summary.tex" in os.listdir(OUTDIR):
    lines.append(r"\input{export_overleaf/executive_summary.tex}")
else:
    lines.append("% MISSING executive_summary.tex — run REAL METRICS")

if "consolidated_metrics_table.tex" in os.listdir(OUTDIR):
    lines.append(r"\input{export_overleaf/consolidated_metrics_table.tex}")
else:
    lines.append("% MISSING consolidated_metrics_table.tex — run REAL METRICS")

if copied:
    lines.append(r"\section*{Key Figures}")
    lines.append(r"\begin{figure}[h]\centering")
    for bn in copied:
        lines.append(r"\includegraphics[width=0.92\textwidth]{export_overleaf/"+bn+"}")
        lines.append(r"\par\medskip")
    lines.append(r"\caption{Selected diagnostics.}")
    lines.append(r"\end{figure}")

with open(os.path.join(OUTDIR, "insert_me.tex"), "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

# Zip everything for quick upload to Overleaf
zip_path = os.path.join(ROOT, "export_overleaf.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(OUTDIR):
        z.write(os.path.join(OUTDIR, fn), arcname=os.path.join("export_overleaf", fn))

print("[OK] export_overleaf/ ready.")
print("Add this line in main.tex:", r"\input{export_overleaf/insert_me.tex}")
print("ZIP:", zip_path)
print("Missing:", missing)


In [ ]:
%%writefile /content/market_assumptions.tex
\section*{Market Assumptions and Microstructure}
This section summarizes the market assumptions and microstructure points used in the backtest.
\begin{itemize}
  \item Liquidity: executed at daily close; no market impact modeled.
  \item Slippage/Costs: included via net returns (ret\_net) in the backtest.
  \item Microstructure: business-day sampling (B-days) with 1–99\% winsorization of daily returns.
\end{itemize}


In [ ]:
# === MASTER EXPORT: DOCX with titles + step-by-step & README.md ===
!pip -q install python-docx

import os, glob
import pandas as pd
from datetime import datetime
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

OUT_DIR   = "/content"
TITLE     = "Pairs Trading Strategy — CQF Master Report"
SUBTITLE  = "Cointegration, OU, Z-Score Signals, Risk & Robustness"
AUTHOR    = "Manoella Costa"
DATE_STR  = datetime.now().strftime("%Y-%m-%d")
PAIRS     = [("META","GOOG"), ("AAPL","META")]  # adjust if needed

# Expected assets (Parts 1–5)
ASSETS = {
    "clean_prices":           f"{OUT_DIR}/clean_prices.csv",
    "bt_summary":             f"{OUT_DIR}/backtest_summary.csv",
    "portfolio_returns":      f"{OUT_DIR}/portfolio_daily_returns.csv",
    "portfolio_metrics":      f"{OUT_DIR}/portfolio_metrics.csv",
    "portfolio_regimes":      f"{OUT_DIR}/portfolio_regime_breakdown.csv",
    "portfolio_stress":       f"{OUT_DIR}/portfolio_stress_metrics.csv",
    "portfolio_equity_png":   f"{OUT_DIR}/portfolio_equity.png",
    "portfolio_roll_sharpe":  f"{OUT_DIR}/portfolio_rolling_sharpe.png",
    "portfolio_roll_vol":     f"{OUT_DIR}/portfolio_rolling_vol.png",
}

def exists(p): return isinstance(p, str) and os.path.exists(p)

def load_df(path, round_n=4):
    if not exists(path): return None
    df = pd.read_csv(path)
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(float).round(round_n)
    return df

# ---------- Text blocks (step-by-step) ----------
STEPBYSTEP = {
"part1": [
 "- Read AAPL/GOOG/META CSVs; auto-detect date and adjusted close.",
 "- Resample to business days; forward-fill up to 3 days.",
 "- Compute log-prices and log-returns; winsorize returns at 1%/99%.",
 "- Rebuild cleaned prices from winsorized returns to keep consistency.",
 "- Normalize to 1.0 at first common date and export clean_prices.csv."
],
"part2": [
 "- Engle–Granger hedge: Y = alpha + beta*X + residual → spread.",
 "- Stationarity tests on spread: ADF (H0: unit root), KPSS (H0: stationary), Phillips–Perron.",
 "- Want: ADF/PP p<0.05 and KPSS p>0.05 → stationary spread.",
 "- Estimate OU half-life via ΔS_t = a + b*S_{t-1} (kappa = -ln(1+b), t1/2 = ln(2)/kappa).",
 "- Plot spread and Z-score with decision bands."
],
"part3": [
 "- Signals on Z-score: enter when |Z|≥zin, exit when |Z|≤zout, stop if |Z|≥zstop.",
 "- Returns: state_{t-1} * (rY - beta*rX).",
 "- Vol targeting to ~10% annualized, with leverage cap.",
 "- Transaction costs in bps per leg, tied to turnover.",
 "- KPIs: CAGR, Annualized Vol, Sharpe, Max Drawdown; save equity/Z plots and CSVs."
],
"part4": [
 "- Build a multi-pair portfolio; equal-risk-dynamic via inverse rolling vol.",
 "- Compute Historical VaR/ES (e.g., 95%) on portfolio daily returns.",
 "- Regime slicing: Low/Medium/High via realized vol terciles.",
 "- Stress windows (e.g., COVID-2020) performance.",
 "- Plots: portfolio equity, rolling Sharpe, rolling vol; export metrics."
],
"part5": [
 "- Grid-search over (lookback, zin, zout) → Sharpe heatmaps.",
 "- Walk-forward: ~2y train → 1y test, rolling; pick best on train, evaluate on test.",
 "- Export folds table (IS params, OOS metrics), OOS equity curve, param stability.",
 "- Favor plateaus over isolated peaks to avoid overfitting."
]
}

# ---------- DOCX helpers ----------
def add_h_center(doc, text, level=0):
    p = doc.add_heading(text, level=level)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    return p
def add_para(doc, text, bold=False, italic=False, center=False):
    p = doc.add_paragraph()
    r = p.add_run(text)
    r.bold = bold; r.italic = italic
    if center: p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    return p
def add_table(doc, df, title=None):
    if df is None or df.empty: return
    if title: add_para(doc, title, bold=True)
    rows, cols = df.shape
    table = doc.add_table(rows=rows+1, cols=cols)
    table.style = "Light List Accent 1"
    # header
    hdr = table.rows[0].cells
    for j, c in enumerate(df.columns): hdr[j].text = str(c)
    # body
    for i in range(rows):
        cells = table.rows[i+1].cells
        for j in range(cols):
            val = df.iat[i, j]
            cells[j].text = "" if pd.isna(val) else str(val)
    for row in table.rows:
        for cell in row.cells:
            for p in cell.paragraphs:
                for run in p.runs: run.font.size = Pt(9)
def add_img(doc, path, caption=None, width_in=6.0):
    if not exists(path): return
    add_para(doc, "")
    doc.add_picture(path, width=Inches(width_in))
    if caption:
        cap = doc.add_paragraph(caption); cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
        cap.runs[0].italic = True

# ---------- Build DOCX ----------
doc = Document()
add_h_center(doc, TITLE, 0)
add_para(doc, SUBTITLE, italic=True, center=True)
add_para(doc, f"{AUTHOR} — {DATE_STR}", center=True)
doc.add_page_break()

# Executive Summary
add_h_center(doc, "Executive Summary", 1)
pm = load_df(ASSETS["portfolio_metrics"])
if pm is not None:
    add_table(doc, pm, "Portfolio Metrics (All-sample)")
else:
    add_para(doc, "Portfolio metrics not found; run Parts 3–4 to populate.", italic=True)

# Data & Preprocessing
add_h_center(doc, "Data & Preprocessing (Part 1)", 1)
for line in STEPBYSTEP["part1"]: add_para(doc, "• " + line)

# Stationarity & Cointegration
add_h_center(doc, "Stationarity & Cointegration (Part 2)", 1)
for line in STEPBYSTEP["part2"]: add_para(doc, "• " + line)

# Signals & Backtesting
add_h_center(doc, "Signal Generation & Backtesting (Part 3)", 1)
for line in STEPBYSTEP["part3"]: add_para(doc, "• " + line)

# Portfolio & Risk
add_h_center(doc, "Portfolio Construction & Risk (Part 4)", 1)
for line in STEPBYSTEP["part4"]: add_para(doc, "• " + line)
add_img(doc, ASSETS["portfolio_equity_png"], "Portfolio Equity")
add_img(doc, ASSETS["portfolio_roll_sharpe"], "Rolling Sharpe (126d)")
add_img(doc, ASSETS["portfolio_roll_vol"], "Rolling Ann. Vol (60d)")

add_table(doc, load_df(ASSETS["portfolio_regimes"]), "Regime Breakdown")
add_table(doc, load_df(ASSETS["portfolio_stress"]),  "Stress Windows Performance")

# Sharpe Surface & Walk-Forward
add_h_center(doc, "Sharpe Surface & Walk-Forward (Part 5)", 1)
for line in STEPBYSTEP["part5"]: add_para(doc, "• " + line)

# Insert Sharpe heatmaps / Walk-forward plots if present
for img in sorted(glob.glob(f"{OUT_DIR}/sharpe_surface_*_L*.png")):
    add_img(doc, img, f"Sharpe Surface — {os.path.basename(img)}")
for img in sorted(glob.glob(f"{OUT_DIR}/walkforward_equity_*.png")):
    add_img(doc, img, f"OOS Equity — {os.path.basename(img)}")
for img in sorted(glob.glob(f"{OUT_DIR}/param_stability_*.png")):
    add_img(doc, img, f"Parameter Stability — {os.path.basename(img)}")

# Per-pair backtests (equity + z plots)
add_h_center(doc, "Per-Pair Backtests (Equity & Z-Score)", 1)
for (y, x) in PAIRS:
    add_para(doc, f"Pair: {y} vs {x}", bold=True)
    add_img(doc, f"{OUT_DIR}/equity_{y}_{x}.png", f"Equity — {y} vs {x}")
    add_img(doc, f"{OUT_DIR}/z_{y}_{x}.png",      f"Z-Score & Bands — {y} vs {x}")

# Appendix A — Code Walkthrough
add_h_center(doc, "Appendix A — Methodology & Code Walkthrough", 1)
add_para(doc, "This appendix explains each block and the rationale behind it:", bold=True)
for sec in ["part1","part2","part3","part4","part5"]:
    add_para(doc, "")
    add_para(doc, sec.upper(), bold=True)
    for line in STEPBYSTEP[sec]: add_para(doc, "· " + line)

# Appendix B — Reproducibility
add_h_center(doc, "Appendix B — Reproducibility & File Map", 1)
add_para(doc, "Key files produced by the pipeline:", bold=True)
for k,v in ASSETS.items():
    add_para(doc, f"- {k}: {v}")
add_para(doc, "Sharpe surface CSVs: /content/grid_sharpe_<Y>_<X>.csv")
add_para(doc, "Walk-forward CSVs: /content/walkforward_<Y>_<X>.csv, /content/param_stability_<Y>_<X>.csv")

DOCX_PATH = f"{OUT_DIR}/CQF_Master_Report.docx"
doc.save(DOCX_PATH)
print("[OK] Saved:", DOCX_PATH)

# ---------- README (Markdown) ----------
readme_lines = []
readme_lines += [f"# {TITLE}", "", f"**{SUBTITLE}**", "", f"**Author:** {AUTHOR}", f"**Date:** {DATE_STR}", ""]
readme_lines += ["## How to Run",
"1. Run Part 1 (cleaning) to create `/content/clean_prices.csv`.",
"2. Run Part 2 to validate pairs (ADF/KPSS/PP, OU half-life).",
"3. Run Part 3 to generate signals/backtests and export `bt_*` CSVs/PNGs.",
"4. Run Part 4 to build the multi-pair portfolio, VaR/ES, regimes, stress windows.",
"5. Run Part 5 for Sharpe surface and walk-forward tuning.",
"6. Run this master export to produce `CQF_Master_Report.docx`.", ""]
readme_lines += ["## Outputs", "- Portfolio metrics: `/content/portfolio_metrics.csv`",
"- Portfolio equity: `/content/portfolio_equity.png`",
"- Sharpe surfaces: `/content/sharpe_surface_*_L*.png`",
"- Walk-forward: `/content/walkforward_*.csv`, `/content/walkforward_equity_*.png`", ""]
readme_lines += ["## Methodology (Step-by-Step)"]
for sec in ["part1","part2","part3","part4","part5"]:
    readme_lines += [f"### {sec.title()}"]
    readme_lines += [f"- {s}" for s in STEPBYSTEP[sec]]
    readme_lines += [""]

README_PATH = f"{OUT_DIR}/README_CQF_Pairs.md"
with open(README_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(readme_lines))
print("[OK] Saved:", README_PATH)


In [ ]:
!pip -q install statsmodels


In [ ]:
# === UPGRADE CONFIG (edit these) ===
DATA_ROOT = "/content"  # where your CSVs are

# Stooq price files (close column auto-detected)
STOOQ_FILES = {
    "AAPL": f"{DATA_ROOT}/aapl_us_d.csv",
    "META": f"{DATA_ROOT}/meta_us_d.csv",
    "GOOG": f"{DATA_ROOT}/goog_us_d.csv",
}

PAIRS = [("META","GOOG"), ("AAPL","META")]

# Walk-forward setup
TRAIN_DAYS = 756        # ~3y
TEST_DAYS  = 252        # 1y
ZIN_GRID   = [1.0, 1.25, 1.5, 1.75, 2.0, 2.25]  # entry band
ZOUT       = 0.5        # exit band
MAX_HOLD_MULT = 10      # stop time = MAX_HOLD_MULT * half-life
ADF_KILL_P   = 0.10     # kill-switch if ADF p > this
ADF_KILL_DAYS= 10       # for this many consecutive days

# Costs & finance
COMMISSION_BPS_PER_LEG = 2.0   # round-turn ~4bps (2 in / 2 out), per leg
SLIPPAGE_BPS_PER_LEG   = 1.0
IMPACT_BPS_PER_TRADE   = 0.5   # simplistic impact per open/close
BORROW_RATE_ANNUAL     = 0.012 # 1.2%/yr on short notional
FUNDING_RATE_ANNUAL    = 0.010 # carry on gross (optional)

# Capacity (ADV & Participation Rate)
# If you know ADV in USD for each symbol, set below; else keep None
ADV_USD = {"AAPL": None, "META": None, "GOOG": None}
PARTICIPATION_RATE = 0.05  # 5% of ADV limit

# Portfolio risk management
PORT_VOL_TARGET = 0.10     # 10% annualized
ROLL_VOL_WIN    = 60
DD_CUT_LEVEL    = 0.10     # when DD > 10%, halve risk

# ====== HELPERS (no extra imports expected) ======
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

def _read_stooq(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    # date column
    dcol = [c for c in df.columns if c.lower() in ("date","data","time","timestamp","unnamed: 0")]
    if dcol:
        df[dcol[0]] = pd.to_datetime(df[dcol[0]], errors="coerce")
        df = df.set_index(dcol[0]).sort_index()
    # close column
    cands = [c for c in df.columns if c.lower() in ("close","c","adj close","adj_close","close_")]
    if not cands:
        raise ValueError(f"Close column not found in {path}.")
    px = pd.to_numeric(df[cands[0]], errors="coerce").dropna()
    return px

def _ols_beta(y, x):
    # y = a + b x
    X = np.column_stack([np.ones(len(x)), x])
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    a, beta = b[0], b[1]
    return a, beta

def _half_life(resid):
    r = pd.Series(resid).dropna()
    if len(r) < 3:
        return 10.0
    x = r.shift(1).iloc[1:].values
    y = r.iloc[1:].values
    if np.allclose(np.var(x), 0):
        return 10.0
    phi = np.dot(x, y) / np.dot(x, x)
    if phi <= 0 or phi >= 0.9999:
        return 10.0
    try:
        hl = -np.log(2) / np.log(phi)
        return float(max(1.0, min(200.0, hl)))
    except Exception:
        return 10.0

def _roll_adf_p(resid, win=252):
    try:
        from statsmodels.tsa.stattools import adfuller
    except Exception:
        return pd.Series(index=resid.index, dtype=float)  # NaNs -> no kill
    r = pd.Series(resid).dropna()
    idx = r.index
    out = pd.Series(index=idx, dtype=float)
    for i in range(win, len(r)):
        try:
            pv = adfuller(r.iloc[i-win:i].values, maxlag=1, regression='c', autolag='AIC')[1]
        except Exception:
            pv = np.nan
        out.iloc[i] = pv
    return out.reindex(resid.index)

def _sharpe(r):
    r = pd.Series(r).dropna()
    if r.std() == 0 or r.empty: return np.nan
    return np.sqrt(252) * r.mean() / r.std()

def _dd_stats(eq):
    eq = pd.Series(eq).dropna()
    rollmax = eq.cummax()
    dd = eq/rollmax - 1.0
    return float(dd.min()), dd

def _fmt(x):
    if x is None or (isinstance(x,float) and (np.isnan(x) or np.isinf(x))):
        return "n/a"
    return f"{x:.4f}"


In [ ]:
# === WALK-FORWARD BACKTEST PER PAIR (with costs, ADF kill-switch) ===
def simulate_pair_walkforward(symY, symX):
    py = _read_stooq(STOOQ_FILES[symY])
    px = _read_stooq(STOOQ_FILES[symX])
    df = pd.concat([py.rename("py"), px.rename("px")], axis=1).dropna()
    df["logy"] = np.log(df["py"])
    df["logx"] = np.log(df["px"])

    # containers
    ret_oos = pd.Series(index=df.index, dtype=float)
    pos_series = pd.Series(0.0, index=df.index)  # +1 long-spread (long Y short X), -1 short-spread
    trade_id = pd.Series(np.nan, index=df.index)
    beta_series = pd.Series(np.nan, index=df.index)

    i0 = 0
    tid = 0
    while True:
        tr_end = i0 + TRAIN_DAYS
        te_end = tr_end + TEST_DAYS
        if te_end > len(df): break

        sub_train = df.iloc[i0:tr_end]
        sub_test  = df.iloc[tr_end:te_end]

        # estimate beta on train (levels)
        a, beta = _ols_beta(sub_train["logy"].values, sub_train["logx"].values)
        beta_series.iloc[tr_end:te_end] = beta

        resid_tr = sub_train["logy"] - (a + beta*sub_train["logx"])
        hl = _half_life(resid_tr)
        max_hold = int(MAX_HOLD_MULT * hl)

        # standardize residual on train
        mu, sd = resid_tr.mean(), resid_tr.std(ddof=1) or 1e-8
        z_tr = (resid_tr - mu)/sd

        # grid search z_in (train) with fixed z_out
        def run_on(df_block):
            res = df_block["logy"] - (a + beta*df_block["logx"])
            z   = (res - mu)/sd
            return z

        z_train = run_on(sub_train)
        best_sh, best_zin = -1e9, ZIN_GRID[0]
        for zin in ZIN_GRID:
            r_sim = _trade_returns(df_block=sub_train, z=z_train, beta=beta,
                                   zin=zin, zout=ZOUT, max_hold=max_hold)
            sh = _sharpe(r_sim)
            if np.isnan(sh): sh = -1e9
            if sh > best_sh:
                best_sh, best_zin = sh, zin

        # test period
        z_test = run_on(sub_test)
        r_test, pos_t, tid_t = _trade_returns(df_block=sub_test, z=z_test, beta=beta,
                                              zin=best_zin, zout=ZOUT, max_hold=max_hold,
                                              return_pos=True, base_prices=sub_test[["py","px"]],
                                              kill_adf=True, resid_series=z_test*sd+mu)
        ret_oos.iloc[tr_end:te_end] = r_test.values
        pos_series.iloc[tr_end:te_end] = pos_t.values
        trade_id.iloc[tr_end:te_end] = tid_t.values

        i0 += TEST_DAYS

    # drop leading NaNs
    ret_oos = ret_oos.dropna()
    pos_series = pos_series.loc[ret_oos.index]
    trade_id = trade_id.loc[ret_oos.index]
    beta_series = beta_series.loc[ret_oos.index]

    results = _pair_metrics_with_trades(symY, symX, ret_oos, pos_series, trade_id, beta_series)
    results["series"] = dict(ret=ret_oos, pos=pos_series, tid=trade_id, beta=beta_series)
    return results

def _trade_returns(df_block, z, beta, zin, zout, max_hold,
                   return_pos=False, base_prices=None, kill_adf=False, resid_series=None):
    """
    Simple mean-reversion engine over a block:
    - Open when z > +zin (short-spread => short Y long X) or z < -zin (long-spread).
    - Close when |z| < zout or max_hold elapsed.
    - PnL per day uses 1 * Y shares and -beta * X shares, normalized by gross notional (|Y| + |beta|*|X|).
    - Costs: commission + slippage + impact when opening/closing; borrow/funding charged daily.
    - Optional kill-switch using rolling ADF on residuals (levels), skipping trading when ADF p deteriorates.
    """
    idx = z.index
    pos = pd.Series(0.0, index=idx)
    tid = pd.Series(np.nan, index=idx)
    in_trade = False
    side = 0  # +1 long-spread (buy Y, short X); -1 short-spread
    hold = 0
    t_id = 0

    # prepare kill-switch
    if kill_adf and resid_series is not None:
        p_rolling = _roll_adf_p(pd.Series(resid_series, index=idx), win=252)
        bad = p_rolling.rolling(ADF_KILL_DAYS).apply(lambda s: float((s > ADF_KILL_P).all()) if s.notna().all() else 0.0, raw=False)
        kill_mask = bad.fillna(0.0) > 0.5
    else:
        kill_mask = pd.Series(False, index=idx)

    # returns
    r = pd.Series(0.0, index=idx)

    # prices for PnL + costs
    if base_prices is None:
        # infer from df_block
        py = df_block["py"]; px = df_block["px"]
    else:
        py = base_prices["py"]; px = base_prices["px"]

    for t in range(len(idx)-1):
        today, nxt = idx[t], idx[t+1]
        if not in_trade:
            # skip if kill
            if not kill_mask.iloc[t]:
                if z.iloc[t] > zin:
                    in_trade = True; side = -1; hold = 0; t_id += 1
                elif z.iloc[t] < -zin:
                    in_trade = True; side = +1; hold = 0; t_id += 1
        else:
            hold += 1
            if (abs(z.iloc[t]) < zout) or (hold >= max_hold) or kill_mask.iloc[t]:
                in_trade = False; side = 0; hold = 0

        pos.iloc[t] = side
        tid.iloc[t] = t_id if side != 0 else np.nan

        # PnL from t->t+1
        if side != 0:
            # notionals
            notional_gross = abs(py.iloc[t]) + abs(beta)*abs(px.iloc[t])
            # price changes
            dPy = py.iloc[t+1] - py.iloc[t]
            dPx = px.iloc[t+1] - px.iloc[t]
            pnl = side * (dPy - beta*dPx)
            ret = pnl / (notional_gross + 1e-12)

            # financing daily
            borrow = BORROW_RATE_ANNUAL/252.0 * (abs(beta)*abs(px.iloc[t]))/(notional_gross+1e-12)
            funding= FUNDING_RATE_ANNUAL/252.0 * (notional_gross)/(notional_gross+1e-12)
            ret -= (borrow + funding)

            r.iloc[t+1] = ret

    # trading costs on open/close (apply at the day of change)
    cost_bps = COMMISSION_BPS_PER_LEG + SLIPPAGE_BPS_PER_LEG + IMPACT_BPS_PER_TRADE
    gross = (abs(py) + abs(beta)*abs(px)).reindex(idx).fillna(method="ffill")
    # open/close flags
    pos_shift = pos.shift(1).fillna(0.0)
    open_flag  = (pos!=0) & (pos_shift==0)
    close_flag = (pos_shift!=0) & (pos==0)
    # each open/close charges two legs
    trade_cost_ret = (cost_bps/1e4)*2.0 * (gross / (gross+1e-12))
    r.loc[open_flag]  -= trade_cost_ret.loc[open_flag]
    r.loc[close_flag] -= trade_cost_ret.loc[close_flag]

    if return_pos:
        return r, pos, tid
    return r


In [ ]:
# === TRADE METRICS, TURNOVER, DURATION, CAPACITY ===
def _pair_metrics_with_trades(symY, symX, ret, pos, tid, beta_ser):
    ret = pd.Series(ret).dropna()
    eq  = (1+ret).cumprod()
    maxdd, dd = _dd_stats(eq)
    vol = ret.std()*np.sqrt(252)
    sh  = _sharpe(ret)
    years = len(ret)/252 if len(ret)>0 else np.nan
    cagr = (1+ret).prod()**(1/years)-1 if years and years>0 else np.nan

    # trade stats
    trade_ids = tid.dropna().astype(int)
    ntr = trade_ids.nunique()
    # duration
    dur = trade_ids.value_counts().values if ntr>0 else np.array([])
    dur_mean = float(np.mean(dur)) if dur.size else np.nan
    dur_med  = float(np.median(dur)) if dur.size else np.nan
    # percent time invested
    pct_in_mkt = float((pos!=0).mean()) if len(pos)>0 else np.nan
    # turnover proxy: sum of abs(position change) per year (two legs implied)
    turn_daily = abs(pos.fillna(0.0).diff()).fillna(0.0)
    turnover_annual = float(turn_daily.sum()/years) if years and years>0 else np.nan

    # beta stability (rolling)
    beta_var = float(pd.Series(beta_ser).dropna().rolling(60).std().median()) if pd.Series(beta_ser).dropna().size>0 else np.nan

    # capacity (rough): if ADV known, PR limits daily gross $; assume average gross ~ mean(|Y|+|β| |X|)
    cap_usd = np.nan
    if ADV_USD.get(symY) and ADV_USD.get(symX):
        adv_gross = ADV_USD[symY] + ADV_USD[symX]*np.nanmean(np.abs(beta_ser))
        cap_usd   = PARTICIPATION_RATE * adv_gross  # per day
        # very rough: scale to capital by assuming 1-day turnover -> capital ~= cap_usd / turnover rate
        if turnover_annual and not np.isnan(turnover_annual) and turnover_annual>1e-6:
            cap_usd = cap_usd / max(1e-6, (turnover_annual/252.0))

    return {
        "pair": f"{symY}-{symX}",
        "CAGR": cagr, "Vol_ann": vol, "Sharpe": sh, "MaxDD": maxdd,
        "Trades": int(ntr), "Dur_mean": dur_mean, "Dur_median": dur_med,
        "%TimeInMkt": pct_in_mkt, "Turnover_ann": turnover_annual,
        "Beta_stab(60d std)": beta_var, "Capacity_USD_est": cap_usd
    }


In [ ]:
# === STATIONARY BOOTSTRAP FOR SHARPE CI ===
def stationary_bootstrap_sharpe(ret, B=1000, avg_block=20, seed=42):
    r = pd.Series(ret).dropna().values
    if r.size < 10:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    p = 1.0/avg_block
    n = len(r)
    sh_boot = np.empty(B)
    for b in range(B):
        idx = np.empty(n, dtype=int)
        t = rng.integers(0, n)
        for i in range(n):
            idx[i] = t
            # continue block with prob (1-p) else restart
            if rng.random() < (1-p):
                t = (t+1) % n
            else:
                t = rng.integers(0, n)
        rb = r[idx]
        sh_boot[b] = np.sqrt(252)*rb.mean()/ (rb.std()+1e-12)
    lo, hi = np.nanpercentile(sh_boot, [2.5, 97.5])
    return (float(lo), float(hi))


In [ ]:
# === PATCHED: PORTFOLIO VOL TARGETING + DD CUT (index-safe) ===
def portfolio_manage(returns_by_pair):
    """
    returns_by_pair: dict {"Y-X": pd.Series of daily returns}
    Equal-weight, then apply vol target and DD cut. Index-safe.
    """
    if not returns_by_pair:
        return pd.Series(dtype=float)

    # concat preserving keys -> columns; align on intersection of dates
    df = pd.concat(returns_by_pair, axis=1)  # columns are dict keys
    df = df.dropna(how="any")                # keep overlapping dates only
    if df.empty:
        return pd.Series(dtype=float)

    raw = df.mean(axis=1)  # equal-weight

    # rolling vol with a sensible min_periods
    rv = raw.rolling(ROLL_VOL_WIN, min_periods=max(10, ROLL_VOL_WIN//2)).std() * np.sqrt(252)
    scale = (PORT_VOL_TARGET / (rv + 1e-12)).clip(upper=3.0)
    managed = (raw * scale).copy()

    # equity & drawdown
    eq = (1.0 + managed).cumprod()
    _, dd = _dd_stats(eq)   # same index as eq

    # align mask explicitly to managed index to avoid IndexingError
    cut_flag = dd.reindex(managed.index).lt(-DD_CUT_LEVEL).fillna(False)

    # apply cut safely using where (no boolean index assignment pitfalls)
    managed = managed.where(~cut_flag, managed * 0.5)

    return managed


In [ ]:
# === PLOTS (matplotlib only) ===
def plot_equity_dd(r, title="Portfolio Equity"):
    r = pd.Series(r).dropna()
    eq = (1+r).cumprod()
    _, dd = _dd_stats(eq)
    fig, ax = plt.subplots(figsize=(9,4))
    ax.plot(eq.index, eq.values, label="Equity")
    ax.fill_between(dd.index, 1.0, 1.0+dd.values, alpha=0.25, label="Drawdown")
    ax.set_title(title); ax.set_xlabel("Date"); ax.set_ylabel("Equity")
    ax.legend(); plt.tight_layout(); plt.show()

def plot_z_with_bands(z, entries=None, exits=None, title="Z-score with bands"):
    z = pd.Series(z).dropna()
    fig, ax = plt.subplots(figsize=(9,3.2))
    ax.plot(z.index, z.values, linewidth=1.1, label="Z")
    for y in [2.0, -2.0, 0.5, -0.5, 0]:
        ax.axhline(y=y, linestyle="--", linewidth=1.0)
    if entries is not None and len(entries):
        ax.scatter(entries.index, entries.values, marker="^", s=24)
    if exits is not None and len(exits):
        ax.scatter(exits.index, exits.values, marker="v", s=24)
    ax.set_title(title); ax.set_xlabel("Date"); ax.set_ylabel("Z")
    plt.tight_layout(); plt.show()

def plot_heatmap(grid_df, title="Sharpe surface (train)"):
    # grid_df: index=zin, columns=zout (or vice-versa)
    import matplotlib.ticker as mticker
    fig, ax = plt.subplots(figsize=(5.6,4.8))
    im = ax.imshow(grid_df.values, aspect="auto", origin="lower")
    ax.set_xticks(range(len(grid_df.columns))); ax.set_xticklabels(grid_df.columns)
    ax.set_yticks(range(len(grid_df.index)));   ax.set_yticklabels(grid_df.index)
    ax.set_title(title)
    # annotate best
    best = np.nanargmax(grid_df.values)
    bi, bj = np.unravel_index(best, grid_df.values.shape)
    ax.scatter([bj],[bi], marker="o", s=40)
    plt.tight_layout(); plt.show()


In [ ]:
# === RUN ALL UPGRADES ===
pair_results = []
pair_returns = {}
pair_z_any = {}  # keep one z for plot if needed

for (y,x) in PAIRS:
    print(f"Running walk-forward for {y}-{x} ...")
    res = simulate_pair_walkforward(y,x)
    pair_results.append(res.copy())
    pair_returns[f"{y}-{x}"] = res["series"]["ret"]
    # compute a z just for one plot block (last test window)
    # quick proxy: residual stdzd from the last train; here we reuse resid from simulator if available
    # (for simplicity, skip detailed z collection)
    pair_z_any[f"{y}-{x}"] = None

# table
tbl = pd.DataFrame([{
    "Pair": r["pair"],
    "CAGR": r["CAGR"], "Vol_ann": r["Vol_ann"], "Sharpe": r["Sharpe"], "MaxDD": r["MaxDD"],
    "Trades": r["Trades"], "Dur_mean": r["Dur_mean"], "Dur_median": r["Dur_median"],
    "%TimeInMkt": r["%TimeInMkt"], "Turnover_ann": r["Turnover_ann"],
    "Beta_stab(60d std)": r["Beta_stab(60d std)"], "Capacity_USD_est": r["Capacity_USD_est"]
} for r in pair_results])

# Sharpe CIs by stationary bootstrap (per-par and portfolio)
ci_rows = []
for k, s in pair_returns.items():
    lo, hi = stationary_bootstrap_sharpe(s, B=800, avg_block=20)
    ci_rows.append({"Name": k, "Sharpe": _sharpe(s), "Sharpe_CI95_low": lo, "Sharpe_CI95_high": hi})

# portfolio equal-weight, risk managed
port_managed = portfolio_manage(pair_returns)
loP, hiP = stationary_bootstrap_sharpe(port_managed, B=800, avg_block=20)

ci_rows.append({"Name":"Portfolio(managed)", "Sharpe": _sharpe(port_managed),
                "Sharpe_CI95_low": loP, "Sharpe_CI95_high": hiP})

ci_tbl = pd.DataFrame(ci_rows)

print("\n=== PAIR METRICS ===")
display(tbl.round(4))

print("\n=== SHARPE CIs (stationary bootstrap) ===")
display(ci_tbl.round(3))

# Plots
plot_equity_dd(port_managed, title="Portfolio (equal-weight, vol-targeted) — Equity & Drawdown")


In [ ]:
# === IN vs OOS table across rolling windows ===
def walkforward_summary(symY, symX):
    py = _read_stooq(STOOQ_FILES[symY]); px = _read_stooq(STOOQ_FILES[symX])
    df = pd.concat([py.rename("py"), px.rename("px")], axis=1).dropna()
    df["logy"] = np.log(df["py"]); df["logx"] = np.log(df["px"])
    rows = []
    i0 = 0
    while True:
        tr_end = i0 + TRAIN_DAYS; te_end = tr_end + TEST_DAYS
        if te_end > len(df): break
        tr = df.iloc[i0:tr_end]; te = df.iloc[tr_end:te_end]
        a,b = _ols_beta(tr["logy"].values, tr["logx"].values)
        resid_tr = tr["logy"] - (a + b*tr["logx"])
        mu,sd = resid_tr.mean(), resid_tr.std(ddof=1) or 1e-8
        hl = _half_life(resid_tr); max_hold = int(MAX_HOLD_MULT*hl)

        # small grid including zout to build heatmap later if needed
        zin_grid = ZIN_GRID
        zout_grid= [0.25, 0.5, 0.75]
        best = (-1e9, None, None)
        for zin in zin_grid:
            for zout in zout_grid:
                ztr = (resid_tr - mu)/sd
                rtr = _trade_returns(tr, ztr, b, zin, zout, max_hold)
                sh = _sharpe(rtr)
                if not np.isnan(sh) and sh > best[0]:
                    best = (sh, zin, zout)
        best_sh, best_zin, best_zout = best

        zte = (te["logy"] - (a + b*te["logx"]) - mu)/sd
        r_te = _trade_returns(te, zte, b, best_zin, best_zout, max_hold)
        rows.append({
            "train_start": tr.index.min(), "train_end": tr.index.max(),
            "test_start": te.index.min(),  "test_end":  te.index.max(),
            "zin": best_zin, "zout": best_zout,
            "Sharpe_in": _sharpe(_trade_returns(tr, (resid_tr-mu)/sd, b, best_zin, best_zout, max_hold)),
            "Sharpe_oos": _sharpe(r_te),
            "CAGR_oos": (1+r_te).prod()**(252/len(r_te))-1 if len(r_te)>30 else np.nan,
            "MaxDD_oos": _dd_stats((1+r_te).cumprod())[0]
        })
        i0 += TEST_DAYS
    return pd.DataFrame(rows)

# Run per pair
wf_tables = {}
for (y,x) in PAIRS:
    wf_tables[f"{y}-{x}"] = walkforward_summary(y,x)
    print(f"\n=== Walk-forward {y}-{x} ===")
    display(wf_tables[f"{y}-{x}"].round(3))


In [ ]:
# === Sensitivity: Sharpe vs transaction costs (multiplier) ===
def sharpe_vs_costs(ret_series_by_pair, multipliers=(0.5, 1.0, 2.0, 3.0)):
    base_comm = COMMISSION_BPS_PER_LEG
    base_slip = SLIPPAGE_BPS_PER_LEG
    base_imp  = IMPACT_BPS_PER_TRADE
    out = []
    for m in multipliers:
        globals()["COMMISSION_BPS_PER_LEG"] = base_comm*m
        globals()["SLIPPAGE_BPS_PER_LEG"]   = base_slip*m
        globals()["IMPACT_BPS_PER_TRADE"]   = base_imp*m
        # recompute portfolio quickly re-running only the portfolio layer:
        port_m = portfolio_manage(ret_series_by_pair)
        out.append({"cost_mult": m, "Sharpe": _sharpe(port_m)})
    # restore
    globals()["COMMISSION_BPS_PER_LEG"] = base_comm
    globals()["SLIPPAGE_BPS_PER_LEG"]   = base_slip
    globals()["IMPACT_BPS_PER_TRADE"]   = base_imp
    return pd.DataFrame(out)

sens_tbl = sharpe_vs_costs(pair_returns)
print("Sharpe vs cost multiplier (1.0 = baseline costs):")
display(sens_tbl)


In [ ]:
# === Reconstruct entries/exits from position series for markers ===
def entries_exits_from_pos(z, pos):
    z = pd.Series(z).dropna()
    pos = pd.Series(pos).reindex(z.index).fillna(0.0)
    change = pos.diff().fillna(pos)
    entries = z[change!=0]  # both open and close; we split:
    open_idx  = entries.index[(pos.reindex(entries.index).fillna(0.0)!=0.0)]
    close_idx = entries.index[(pos.reindex(entries.index).fillna(0.0)==0.0)]
    return z.reindex(open_idx), z.reindex(close_idx)

# Example for the first pair (if you stored z/pos by window you can adapt; here we use the last OOS window)
first_pair = next(iter(pair_returns.keys()))
# quick proxy: rebuild z from last β and μ,σ in last window would be ideal;
# if missing, just plot positions as neutral markers
# if you have pos_series in the dict "series", use:
# z_series = pair_results[0]["series"]["z"]   # if stored
# pos_series = pair_results[0]["series"]["pos"]
# ent, ext = entries_exits_from_pos(z_series, pos_series)
# plot_z_with_bands(z_series, entries=ent, exits=ext, title=f"Z with trades — {first_pair}")



In [ ]:
# === Heatmap on last train window (example) ===
def heatmap_last_train(symY, symX, zin_grid=None, zout_grid=None):
    py = _read_stooq(STOOQ_FILES[symY]); px = _read_stooq(STOOQ_FILES[symX])
    df = pd.concat([py.rename("py"), px.rename("px")], axis=1).dropna()
    df["logy"]=np.log(df["py"]); df["logx"]=np.log(df["px"])
    tr = df.iloc[-(TRAIN_DAYS+TEST_DAYS):-TEST_DAYS]  # last train
    a,b = _ols_beta(tr["logy"].values, tr["logx"].values)
    resid = tr["logy"] - (a + b*tr["logx"])
    mu,sd = resid.mean(), resid.std(ddof=1) or 1e-8
    z = (resid - mu)/sd
    hl = _half_life(resid); max_hold=int(MAX_HOLD_MULT*hl)
    zin_grid = zin_grid or ZIN_GRID
    zout_grid= zout_grid or [0.25,0.5,0.75,1.0]
    M = pd.DataFrame(index=[round(g,2) for g in zin_grid], columns=[round(g,2) for g in zout_grid], dtype=float)
    for zin in zin_grid:
        for zout in zout_grid:
            r = _trade_returns(tr, z, b, zin, zout, max_hold)
            M.loc[round(zin,2), round(zout,2)] = _sharpe(r)
    plot_heatmap(M, title=f"Sharpe surface (train) — {symY}-{symX}")
    # opcional: identifique região top-decile
    q90 = np.nanpercentile(M.values.flatten(), 90)
    stable = (M>=q90)
    print("Fraction of grid in top decile (stability):", stable.stack().mean().round(3))
    return M

M_meta = heatmap_last_train("META","GOOG")


In [ ]:
# === Plot rolling ADF p-values with kill zones ===
def plot_adf_kill(resid_series, title="Rolling ADF p-values"):
    p = _roll_adf_p(resid_series, win=252)
    kill = p.rolling(ADF_KILL_DAYS).apply(lambda s: float((s>ADF_KILL_P).all()) if s.notna().all() else 0.0)
    fig, ax = plt.subplots(figsize=(9,3.2))
    ax.plot(p.index, p.values, label="ADF p-value")
    ax.axhline(ADF_KILL_P, linestyle="--", linewidth=1.0, label=f"Kill threshold {ADF_KILL_P}")
    # paint kill regions
    kz = kill.fillna(0.0)>0.5
    if kz.any():
        ax.fill_between(kz.index, 0, 1, where=kz.values, transform=ax.get_xaxis_transform(), alpha=0.15)
    ax.set_title(title); ax.legend(); plt.tight_layout(); plt.show()

# To create 'resid_series', take a window (ex.: last train+test) and calculate y-(a+βx)


In [ ]:
# === Stress metrics on portfolio ===
def window_metrics(series, start, end):
    s = pd.Series(series).dropna().loc[start:end]
    if s.empty: return {"CAGR":np.nan,"Vol":np.nan,"Sharpe":np.nan,"MaxDD":np.nan}
    vol = s.std()*np.sqrt(252); sh=_sharpe(s)
    years = len(s)/252; cagr=(1+s).prod()**(1/years)-1 if years>0 else np.nan
    eq=(1+s).cumprod(); mdd,_dd=_dd_stats(eq)
    return {"CAGR":cagr,"Vol":vol,"Sharpe":sh,"MaxDD":mdd}

stress_windows = [
    ("Oil Crash 2014–2016","2014-06-01","2016-02-29"),
    ("Russia–Ukraine 2022","2022-01-01","2022-12-31"),
]
for name,a,b in stress_windows:
    w=window_metrics(port_managed, a, b)
    print(f"{name}: Sharpe {w['Sharpe']:.2f}, MaxDD {w['MaxDD']*100:.2f}%, CAGR {w['CAGR']*100:.2f}%")


In [ ]:
# === Reproducibility audit header ===
import sys, hashlib, importlib, platform, datetime
def file_hash(path):
    try:
        with open(path, "rb") as f: return hashlib.md5(f.read()).hexdigest()
    except Exception: return "NA"
print("RUN STAMP:", datetime.datetime.utcnow().isoformat(),"UTC")
print("PY:", sys.version.split()[0], "OS:", platform.platform())
for mod in ["numpy","pandas","matplotlib","statsmodels"]:
    try:
        m=importlib.import_module(mod)
        print(f"{mod}=={getattr(m,'__version__','NA')}")
    except: print(f"{mod} not installed")
for s,p in STOOQ_FILES.items():
    print(s, "CSV:", p, "MD5:", file_hash(p))
np.random.seed(42)


In [ ]:
# === REAL METRICS (from in-memory returns: pair_returns, port_managed) ===
import os, numpy as np, pandas as pd, datetime as dt

# --- sanity checks ---
if "pair_returns" not in globals() or not isinstance(pair_returns, dict) or len(pair_returns)==0:
    raise ValueError("pair_returns is missing or empty. Run the UPGRADES blocks first.")
if "port_managed" not in globals() or not isinstance(port_managed, pd.Series):
    raise ValueError("port_managed is missing. Run the UPGRADES blocks first.")

# --- helpers ---
def perf_metrics(ret: pd.Series) -> dict:
    s = pd.Series(ret).dropna()
    if s.empty:
        return {"CAGR":np.nan,"Vol_ann":np.nan,"Sharpe":np.nan,"MaxDD":np.nan,"VaR95":np.nan,"ES95":np.nan}
    eq = (1.0 + s).cumprod()
    rollmax = eq.cummax()
    dd = (eq / rollmax) - 1.0
    maxdd = float(dd.min())
    vol_ann = s.std() * np.sqrt(252)
    sharpe  = (s.mean() * np.sqrt(252)) / (s.std() + 1e-12)
    years   = len(s) / 252.0
    cagr    = (1.0 + s).prod()**(1/years) - 1.0 if years > 0 else np.nan
    q       = float(np.quantile(s, 0.05))   # 5% (VaR 95)
    es      = float(s[s<=q].mean()) if (s<=q).any() else q
    return {"CAGR":cagr,"Vol_ann":vol_ann,"Sharpe":sharpe,"MaxDD":maxdd,"VaR95":q,"ES95":es}

def window_metrics(series, start, end):
    s = pd.Series(series).dropna()
    if not isinstance(s.index, pd.DatetimeIndex):
        s.index = pd.to_datetime(s.index, errors="coerce")
        s = s.dropna()
    s = s.loc[start:end]
    if s.empty:
        return {"CAGR":np.nan,"Vol":np.nan,"Sharpe":np.nan,"MaxDD":np.nan}
    vol = s.std()*np.sqrt(252); sh = (s.mean()*np.sqrt(252))/(s.std()+1e-12)
    years = len(s)/252; cagr = (1+s).prod()**(1/years)-1 if years>0 else np.nan
    eq = (1+s).cumprod(); roll = eq.cummax(); dd = eq/roll-1
    return {"CAGR":cagr,"Vol":vol,"Sharpe":sh,"MaxDD":float(dd.min())}

def pct(x):
    return "NA" if (x is None or (isinstance(x,float) and (np.isnan(x) or np.isinf(x)))) else f"{x*100:.2f}%"

# --- consolidated table (pairs + portfolio) ---
rows = []
for k in sorted(pair_returns.keys()):
    m = perf_metrics(pair_returns[k])
    rows.append([k, m["CAGR"], m["Vol_ann"], m["Sharpe"], m["MaxDD"], m["VaR95"], m["ES95"]])

pm = perf_metrics(port_managed)
rows.append(["Portfolio", pm["CAGR"], pm["Vol_ann"], pm["Sharpe"], pm["MaxDD"], pm["VaR95"], pm["ES95"]])

tbl = pd.DataFrame(rows, columns=["Pair / Portfolio","CAGR","Vol (ann)","Sharpe","MaxDD","VaR(95%)","ES(95%)"])
display(tbl.round(4))

# --- stationary bootstrap CI for portfolio Sharpe, if function exists ---
sh_ci_txt = ""
if "stationary_bootstrap_sharpe" in globals():
    loP, hiP = stationary_bootstrap_sharpe(port_managed.dropna(), B=800, avg_block=20)
    sh_ci_txt = f" (95% CI: {loP:.2f} to {hiP:.2f})"
else:
    sh_ci_txt = ""

# --- stress tests ---
stress_windows = [
    ("Oil Crash 2014–2016","2014-06-01","2016-02-29"),
    ("Russia–Ukraine 2022","2022-01-01","2022-12-31"),
]
stress_lines = []
for name,a,b in stress_windows:
    w = window_metrics(port_managed, a, b)
    stress_lines.append(f"- **{name}**: Sharpe {w['Sharpe']:.2f}, MaxDD {w['MaxDD']*100:.2f}%, CAGR {w['CAGR']*100:.2f}%.")

print("\nStress tests:\n" + "\n".join(stress_lines))

# --- build Executive Summary (Markdown) ---
ES = []
ES.append("**Pairs Trading Strategy – Final Results**")
ES.append("- **Methodology**: Engle–Granger cointegration, Z-score mean reversion, half-life horizons, stop-bands, costs and slippage included.")
ES.append("- **Assets tested**: " + "; ".join(sorted(pair_returns.keys())))
ES.append("- **Portfolio allocation**: equal-weight with volatility targeting and drawdown cut.")
ES.append("\n**Performance highlights (portfolio):**")
ES.append(f"- CAGR: **{pct(pm['CAGR'])}**")
ES.append(f"- Annualized Volatility: **{pct(pm['Vol_ann'])}**")
ES.append(f"- Sharpe ratio: **{pm['Sharpe']:.2f}**{sh_ci_txt}")
ES.append(f"- Max Drawdown: **{pct(pm['MaxDD'])}**")
ES.append(f"- VaR (95%): **{pct(pm['VaR95'])}** | ES (95%): **{pct(pm['ES95'])}**")
ES.append("\n**Stress Tests (no COVID):**")
ES += stress_lines
ES.append("\n**Economic rationale:**")
ES.append("- META–GOOG: digital advertising peers; similar macro/regulatory drivers → cointegration candidate.")
ES.append("- AAPL–META: large-cap consumer tech with partially complementary cycles → diversification benefit.")
ES.append('\n**Key insight**:\n> *This strategy is robust across the specified stress regimes, but performance could be further improved by dynamic threshold calibration, liquidity filters, and richer transaction-cost modeling.*')

exec_summary_md = "\n".join(ES)
print("\n=== Executive Summary (Markdown) ===\n")
print(exec_summary_md)

# --- write LaTeX fragments ---
import os

def latex_escape(s):
    return (s.replace('\\','\\textbackslash{}')
             .replace('%','\\%').replace('_','\\_').replace('&','\\&')
             .replace('#','\\#').replace('{','\\{').replace('}','\\}'))

# executive_summary.tex
with open("/content/executive_summary.tex","w",encoding="utf-8") as f:
    f.write("\\section*{Executive Summary}\n")
    for line in ES:
        if line.startswith("- "):
            f.write("\\noindent " + latex_escape(line[2:]) + "\\\\\n")
        elif line.startswith("**") or line.startswith("> "):
            f.write("\\noindent " + latex_escape(line) + "\\\\\n")
        else:
            f.write("\\noindent " + latex_escape(line) + "\\\\\n")

# consolidated_metrics_table.tex
tbl_tex = tbl.copy()
for c in ["CAGR","Vol (ann)","MaxDD","VaR(95%)","ES(95%)"]:
    tbl_tex[c] = (tbl_tex[c].astype(float)*100).round(2).astype(str) + "\\%"
tbl_tex["Sharpe"] = tbl_tex["Sharpe"].astype(float).round(2).astype(str)

with open("/content/consolidated_metrics_table.tex","w",encoding="utf-8") as f:
    cols = list(tbl_tex.columns)
    f.write("\\begin{table}[h]\\centering\n\\begin{tabular}{lrrrrrr}\\toprule\n")
    f.write(" & ".join(cols) + " \\\\\\midrule\n")
    for _,row in tbl_tex.iterrows():
        f.write(" & ".join(str(row[c]) for c in cols) + " \\\\\n")
    f.write("\\bottomrule\n\\end{tabular}\n\\caption{Consolidated performance metrics (net of costs).}\n\\end{table}\n")

print("\nSaved fragments:\n- /content/executive_summary.tex\n- /content/consolidated_metrics_table.tex")


In [ ]:
# === PART 6: Bundle .tex + figures into export_overleaf/ and create insert_me.tex ===
import os, shutil, zipfile, glob

print("Part 6: preparing export_overleaf/ and insert_me.tex …")

ROOT   = "/content"
OUTDIR = os.path.join(ROOT, "export_overleaf")
os.makedirs(OUTDIR, exist_ok=True)

need = ["executive_summary.tex", "consolidated_metrics_table.tex"]
missing = []
for fn in need:
    src = os.path.join(ROOT, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUTDIR, fn))
    else:
        missing.append(fn)

# Try to collect useful figures if present
fig_candidates = []
fig_candidates += glob.glob(os.path.join(ROOT, "*.png"))
fig_candidates += glob.glob(os.path.join(ROOT, "part3_outputs", "*.png"))
fig_candidates += glob.glob(os.path.join(ROOT, "part5_outputs", "*.png"))

copied = []
for src in fig_candidates:
    if os.path.exists(src):
        bn = os.path.basename(src)
        dst = os.path.join(OUTDIR, bn)
        # avoid duplicates by name
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            copied.append(bn)

# Create insert_me.tex (safe even if some files are missing)
lines = []
lines.append("% auto-generated — include this file in main.tex")
if "executive_summary.tex" in os.listdir(OUTDIR):
    lines.append(r"\input{export_overleaf/executive_summary.tex}")
else:
    lines.append("% MISSING executive_summary.tex — run REAL METRICS cell")

if "consolidated_metrics_table.tex" in os.listdir(OUTDIR):
    lines.append(r"\input{export_overleaf/consolidated_metrics_table.tex}")
else:
    lines.append("% MISSING consolidated_metrics_table.tex — run REAL METRICS cell")

if copied:
    lines.append(r"\section*{Key Figures}")
    for bn in copied:
        lines.append(r"\begin{figure}[h]\centering")
        lines.append(r"\includegraphics[width=0.92\textwidth]{export_overleaf/"+bn+"}")
        lines.append(r"\caption{"+bn.replace('_','\_')+r"}")
        lines.append(r"\end{figure}")

with open(os.path.join(OUTDIR, "insert_me.tex"), "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

# zip for upload
zip_path = os.path.join(ROOT, "export_overleaf.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(OUTDIR):
        z.write(os.path.join(OUTDIR, fn), arcname=os.path.join("export_overleaf", fn))

print("[OK] export_overleaf/ ready.")
print("Add this line in Overleaf main.tex:", r"\input{export_overleaf/insert_me.tex}")
print("ZIP:", zip_path)


In [ ]:
# === OOS Builder v2 (verbose, adaptive, no look-ahead) ===
import numpy as np, pandas as pd

# --- Requirements from your setup ---
need = ["TRAIN_DAYS","TEST_DAYS","STOOQ_FILES","_read_stooq","_ols_beta","_half_life","_trade_returns"]
miss = [n for n in need if n not in globals()]
if miss:
    raise RuntimeError(f"Missing globals: {miss}. Run your setup cells first.")

PAIRS = [("META","GOOG"), ("AAPL","META")]  # adjust if needed
ZIN_GRID  = [0.5, 1.0, 1.5]
ZOUT_GRID = [1.5, 2.0, 2.5]
MAX_HOLD_MULT = float(globals().get("MAX_HOLD_MULT", 1.0))

def _ann_sharpe(s):
    s = pd.Series(s).dropna()
    return (s.mean()*np.sqrt(252))/(s.std()+1e-12) if len(s)>2 else np.nan

def _cagr(s):
    s = pd.Series(s).dropna()
    yrs = len(s)/252
    return (1+s).prod()**(1/yrs)-1 if yrs>0 else np.nan

def _maxdd(s):
    s = pd.Series(s).dropna()
    eq = (1+s).cumprod()
    dd = eq/eq.cummax()-1
    return float(dd.min()) if len(dd) else np.nan

def _window_iter_len(n, tr_days, te_days, step_by):
    i = 0
    while i + tr_days + te_days <= n:
        yield slice(i, i+tr_days), slice(i+tr_days, i+tr_days+te_days)
        i += step_by

def _build_one_window(tr, te, b, mu, sd, hl, zin, zout, max_hold):
    # Z (train/test) given train mu/sd; apply on test only for returns
    z_tr = (tr["ly"] - (tr["lx"]*b + (mu*0 + mu)))  # not used; kept for ref
    z_te = (te["ly"] - (te["lx"]*b + (mu*0 + mu))) / (sd if sd else 1e-8)
    r_te = _trade_returns(te, z_te, b, float(zin), float(zout), int(max_hold))
    r_te = pd.Series(r_te, index=getattr(r_te, "index", te.index), dtype=float).dropna()
    return r_te

rows = []
print("=== OOS Builder v2 — starting ===")
for y, x in PAIRS:
    tag = f"{y}-{x}"
    if y not in STOOQ_FILES or x not in STOOQ_FILES:
        print(f"[WARN] {tag}: missing file path(s) in STOOQ_FILES — skipping.")
        continue

    py = _read_stooq(STOOQ_FILES[y])
    px = _read_stooq(STOOQ_FILES[x])

    df = pd.concat([np.log(py).rename("ly"), np.log(px).rename("lx")], axis=1).dropna()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    df = df.dropna().sort_index()
    n = len(df)

    tr_days = int(TRAIN_DAYS)
    te_days = int(TEST_DAYS)
    print(f"[{tag}] total obs: {n}, requested train={tr_days}, test={te_days}")

    # Adapt if not enough data for at least 1 window
    if n < (tr_days + te_days + 5):
        # conservative adaptation
        tr_days_adapt = max(60, min(tr_days, n//2 - 5))
        te_days_adapt = max(20, min(te_days, n - tr_days_adapt - 5))
        print(f"[{tag}] adapt windows -> train={tr_days_adapt}, test={te_days_adapt}")
        tr_days, te_days = tr_days_adapt, te_days_adapt

    # First pass: step


In [ ]:
# === OOS QUICK-FIX: build oos_df + bh_df from existing backtest CSVs ===
import os, numpy as np, pandas as pd

# Pairs we expect (edit if needed)
PAIRS = [("META","GOOG"), ("AAPL","META")]

# Helper: robust CSV reader for bt files
def _read_bt(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    # infer datetime column
    date_cols = [c for c in df.columns if any(k in c.lower() for k in ["date","time","unnamed"])]
    if date_cols:
        df[date_cols[0]] = pd.to_datetime(df[date_cols[0]], errors="coerce")
        df = df.set_index(date_cols[0]).sort_index()
    # find a returns column
    cand = [c for c in df.columns if "ret" in c.lower()]
    if not cand:
        return None
    # prefer 'ret_net' if exists
    col = "ret_net" if "ret_net" in df.columns else cand[0]
    s = pd.to_datetime(df.index)
    ser = pd.Series(df[col].astype(float).values, index=s, name="ret_net").dropna()
    return ser

# Helper: read Stooq closes -> daily log returns
def _read_stooq_close(path):
    df = pd.read_csv(path)
    # stooq default columns: Date, Open, High, Low, Close, Volume
    date_col = [c for c in df.columns if c.lower().startswith("date")]
    close_col = [c for c in df.columns if c.lower().startswith("close")]
    if not date_col or not close_col:
        # try any datetime-like and any numeric col named close-ish
        date_col = date_col or [df.columns[0]]
        close_col = close_col or [df.select_dtypes(include=[np.number]).columns[-1]]
    df[date_col[0]] = pd.to_datetime(df[date_col[0]], errors="coerce")
    df = df.dropna(subset=[date_col[0]]).sort_values(date_col[0]).set_index(date_col[0])
    px = df[close_col[0]].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    return px

# Try to locate Stooq files (falls back to defaults if STOOQ_FILES not provided)
STOOQ_FILES = globals().get("STOOQ_FILES", {})
DEFAULT_FILES = {
    "META": "/content/meta_us_d.csv",
    "GOOG": "/content/goog_us_d.csv",
    "AAPL": "/content/aapl_us_d.csv",
}
for k,v in DEFAULT_FILES.items():
    if k not in STOOQ_FILES and os.path.exists(v):
        STOOQ_FILES[k] = v

rows = []
bh_rows = []

for y, x in PAIRS:
    bt_path = f"/content/bt_{y}_{x}.csv"
    ser = _read_bt(bt_path)
    if ser is None or ser.empty:
        print(f"[WARN] Missing or invalid backtest file for {y}-{x}: {bt_path}. Skipping pair.")
        continue

    n = len(ser)
    if n < 50:
        print(f"[WARN] Too few observations in {bt_path} ({n}). Skipping pair.")
        continue

    # 70% train / 30% test split (time-ordered)
    split = int(np.floor(0.7 * n))
    tr = ser.iloc[:split]
    te = ser.iloc[split:]

    if te.empty:
        print(f"[WARN] No OOS segment for {y}-{x}. Skipping.")
        continue

    # Metrics on TEST
    def ann_sharpe(s):
        s = pd.Series(s).dropna()
        return (s.mean()*np.sqrt(252))/(s.std()+1e-12) if len(s)>2 else np.nan
    def cagr(s):
        s = pd.Series(s).dropna()
        yrs = len(s)/252
        return (1+s).prod()**(1/yrs)-1 if yrs>0 else np.nan
    def maxdd(s):
        eq = (1+s).cumprod()
        dd = eq/eq.cummax()-1
        return float(dd.min()) if len(dd) else np.nan

    shp_oos  = ann_sharpe(te)
    cagr_oos = cagr(te)
    mdd_oos  = maxdd(te)

    rows.append({
        "pair": f"{y}-{x}",
        "train_start": tr.index[0], "train_end": tr.index[-1],
        "test_start":  te.index[0], "test_end":  te.index[-1],
        "train_days_used": len(tr), "test_days_used": len(te), "step_by": len(te),
        "zin": np.nan, "zout": np.nan, "half_life": np.nan, "max_hold": np.nan,
        "Sharpe_oos": float(shp_oos), "CAGR_oos": float(cagr_oos), "MaxDD_oos": float(mdd_oos)
    })

    # --- Baseline 50/50 for the same OOS window (needs Stooq prices) ---
    if y in STOOQ_FILES and x in STOOQ_FILES:
        py = _read_stooq_close(STOOQ_FILES[y])
        px = _read_stooq_close(STOOQ_FILES[x])
        dfp = pd.concat([py.rename("y"), px.rename("x")], axis=1).dropna().sort_index()
        oos = dfp.loc[te.index.min(): te.index.max()]
        if len(oos) >= 5:
            ry = np.log(oos["y"]).diff().dropna()
            rx = np.log(oos["x"]).diff().dropna()
            r_bh = (0.5*ry + 0.5*rx).reindex(ry.index.union(rx.index)).dropna()
            shp_bh = ann_sharpe(r_bh); cagr_bh = cagr(r_bh); mdd_bh = maxdd(r_bh)
            bh_rows.append({
                "pair": f"{y}-{x}",
                "test_start": r_bh.index.min(), "test_end": r_bh.index.max(),
                "Sharpe_bh": float(shp_bh), "CAGR_bh": float(cagr_bh), "MaxDD_bh": float(mdd_bh)
            })
        else:
            print(f"[WARN] Not enough Stooq coverage for baseline {y}-{x} in OOS window.")
    else:
        print(f"[WARN] Stooq files not available for baseline {y}-{x}.")

# Build oos_df / bh_df / merge_df
if not rows:
    raise RuntimeError("Quick-fix could not produce any OOS window. Ensure bt_*.csv exist with a 'ret_net' (or *ret*) column.")

oos_df = pd.DataFrame(rows)
for c in ["train_start","train_end","test_start","test_end"]:
    oos_df[c] = pd.to_datetime(oos_df[c], errors="coerce")
oos_df = oos_df.sort_values(["pair","test_start"]).reset_index(drop=True)
oos_df.to_csv("/content/oos_df.csv", index=False)

bh_df = pd.DataFrame(bh_rows)
if not bh_df.empty:
    for c in ["test_start","test_end"]:
        bh_df[c] = pd.to_datetime(bh_df[c], errors="coerce")
    bh_df = bh_df.sort_values(["pair","test_start"]).reset_index(drop=True)

# merge (inner on pair & window)
if not bh_df.empty:
    merge_df = pd.merge(
        oos_df[["pair","test_start","test_end","Sharpe_oos","CAGR_oos","MaxDD_oos"]],
        bh_df[["pair","test_start","test_end","Sharpe_bh","CAGR_bh","MaxDD_bh"]],
        on=["pair","test_start","test_end"],
        how="inner"
    )
else:
    merge_df = oos_df.copy()
    merge_df["Sharpe_bh"] = np.nan
    merge_df["CAGR_bh"]   = np.nan
    merge_df["MaxDD_bh"]  = np.nan

print("\n[OK] Quick-fix built the following:")
print(" - oos_df -> /content/oos_df.csv | rows:", len(oos_df))
if not bh_df.empty:
    print(" - bh_df   (baseline 50/50)     | rows:", len(bh_df))
print(" - merge_df (strategy vs baseline) ready in memory.")

# Show a compact tail
try:
    from IPython.display import display
    display(oos_df.tail(5).round(3))
    if not bh_df.empty:
        display(bh_df.tail(5).round(3))
        display(merge_df.tail(5).round(3))
except Exception:
    pass


In [ ]:
# === A) Baseline 50/50 — robust (no 'py'/'px' assumptions) ===
import numpy as np, pandas as pd, os

# Guard-rails
if "oos_df" not in globals() or oos_df.empty:
    raise RuntimeError("oos_df missing. Run the OOS builder (Quick-Fix or walk-forward) first.")

# Try to reuse STOOQ_FILES or infer defaults
STOOQ_FILES = dict(STOOQ_FILES) if "STOOQ_FILES" in globals() else {}
defaults = {
    "META": "/content/meta_us_d.csv",
    "GOOG": "/content/goog_us_d.csv",
    "AAPL": "/content/aapl_us_d.csv",
}
for k,v in defaults.items():
    if k not in STOOQ_FILES and os.path.exists(v):
        STOOQ_FILES[k] = v

def _read_stooq_close(path):
    df = pd.read_csv(path)
    # stooq default: Date, Open, High, Low, Close, Volume
    date_col = [c for c in df.columns if c.lower().startswith("date")] or [df.columns[0]]
    close_col = [c for c in df.columns if c.lower().startswith("close")]
    if not close_col:
        # fallback: last numeric column
        numcols = df.select_dtypes(include=[np.number]).columns.tolist()
        if not numcols:
            raise ValueError(f"No numeric close column found in {path}")
        close_col = [numcols[-1]]
    df[date_col[0]] = pd.to_datetime(df[date_col[0]], errors="coerce")
    df = df.dropna(subset=[date_col[0]]).sort_values(date_col[0]).set_index(date_col[0])
    px = df[close_col[0]].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    return px

def _ann_sharpe(s):
    s = pd.Series(s).dropna()
    return (s.mean()*np.sqrt(252))/(s.std()+1e-12) if len(s)>2 else np.nan

def _cagr(s):
    s = pd.Series(s).dropna()
    yrs = len(s)/252
    return (1+s).prod()**(1/yrs)-1 if yrs>0 else np.nan

def _maxdd(s):
    eq = (1+s).cumprod()
    dd = eq/eq.cummax()-1
    return float(dd.min()) if len(dd) else np.nan

bh_rows = []
pairs = oos_df["pair"].dropna().unique().tolist()

for pair in pairs:
    try:
        y, x = pair.split("-")
    except ValueError:
        print(f"[WARN] Skipping malformed pair label: {pair}")
        continue
    if y not in STOOQ_FILES or x not in STOOQ_FILES:
        print(f"[WARN] Missing Stooq file for {pair}; skipping baseline.")
        continue

    # Read closes and align to OOS windows for this pair
    py = _read_stooq_close(STOOQ_FILES[y])
    px = _read_stooq_close(STOOQ_FILES[x])
    prices = pd.concat([py.rename("y"), px.rename("x")], axis=1).dropna().sort_index()

    sub = oos_df[oos_df["pair"] == pair].copy()
    for _, r in sub.iterrows():
        ts, te = pd.to_datetime(r["test_start"]), pd.to_datetime(r["test_end"])
        o = prices.loc[ts:te]
        if len(o) < 5:
            print(f"[WARN] Baseline window too short for {pair}: {ts.date()}..{te.date()}")
            continue

        ry = np.log(o["y"]).diff().dropna()
        rx = np.log(o["x"]).diff().dropna()
        r_bh = (0.5*ry + 0.5*rx).reindex(ry.index.union(rx.index)).dropna()

        bh_rows.append({
            "pair": pair,
            "test_start": r["test_start"],
            "test_end": r["test_end"],
            "Sharpe_bh": float(_ann_sharpe(r_bh)),
            "CAGR_bh":   float(_cagr(r_bh)),
            "MaxDD_bh":  float(_maxdd(r_bh))
        })

# Build bh_df and merge with oos_df (strategy metrics)
bh_df = pd.DataFrame(bh_rows)
if not bh_df.empty:
    for c in ["test_start","test_end"]:
        bh_df[c] = pd.to_datetime(bh_df[c], errors="coerce")
    bh_df = bh_df.sort_values(["pair","test_start"]).reset_index(drop=True)

# Create merge_df: strategy OOS vs baseline OOS
if not bh_df.empty:
    merge_df = pd.merge(
        oos_df[["pair","test_start","test_end","Sharpe_oos","CAGR_oos","MaxDD_oos"]],
        bh_df[["pair","test_start","test_end","Sharpe_bh","CAGR_bh","MaxDD_bh"]],
        on=["pair","test_start","test_end"],
        how="inner"
    )
else:
    merge_df = oos_df.copy()
    merge_df["Sharpe_bh"] = np.nan
    merge_df["CAGR_bh"]   = np.nan
    merge_df["MaxDD_bh"]  = np.nan

print("[OK] Baseline built. Frames in memory: oos_df, bh_df, merge_df")
try:
    from IPython.display import display
    display(bh_df.tail(5).round(3))
    display(merge_df.tail(5).round(3))
except Exception:
    pass


In [ ]:
# === B) Engle–Granger ADF on residual (last TRAIN window) ===
import numpy as np, pandas as pd
from statsmodels.tsa.stattools import adfuller

_req = ["TRAIN_DAYS","TEST_DAYS","STOOQ_FILES","_read_stooq","_ols_beta"]
for r in _req:
    if r not in globals():
        raise RuntimeError(f"Missing '{r}'. Ensure your setup ran.")

pairs = [("META","GOOG"), ("AAPL","META")]
rows = []
for y,x in pairs:
    py = _read_stooq(STOOQ_FILES[y]); px = _read_stooq(STOOQ_FILES[x])
    df = pd.concat([np.log(py).rename("ly"), np.log(px).rename("lx")], axis=1).dropna()
    if len(df) < (TRAIN_DAYS+TEST_DAYS+10):
        continue
    tr = df.iloc[-(TRAIN_DAYS+TEST_DAYS):-TEST_DAYS]
    a,b = _ols_beta(tr["ly"].values, tr["lx"].values)
    resid = tr["ly"] - (a + b*tr["lx"])
    adf = adfuller(resid.values, maxlag=None, regression="c", autolag="AIC")
    tau, pval = float(adf[0]), float(adf[1])
    rows.append({"pair":f"{y}-{x}","tau_stat":tau,"p_value":pval})

eg_df = pd.DataFrame(rows)
display(eg_df.round(4))

with open("/content/eg_table.tex","w",encoding="utf-8") as f:
    f.write("\\begin{table}[h]\\centering\n\\begin{tabular}{lrr}\\toprule\n")
    f.write("Pair & ADF $\\tau$ (residual) & p\\text{-}value \\\\\\midrule\n")
    for _,r in eg_df.iterrows():
        f.write(f"{r['pair']} & {r['tau_stat']:.3f} & {r['p_value']:.3f} \\\\\n")
    f.write("\\bottomrule\n\\end{tabular}\n\\caption{Engle–Granger residual stationarity on last train window.}\n\\end{table}\n")

print("Saved: /content/eg_table.tex")


In [ ]:
# === C) Turnover proxy: entry counts & avg holding (OOS windows) ===
import numpy as np, pandas as pd

_req = ["oos_df","STOOQ_FILES","_read_stooq","_ols_beta","_half_life"]
for r in _req:
    if r not in globals():
        raise RuntimeError(f"Missing '{r}'. Run prior cells first.")

def _entry_count(z, zout):
    z = pd.Series(z).dropna()
    # Count when abs(z) crosses up through zout
    sgn = (z.abs() > float(zout)).astype(int)
    entry = (sgn.diff() == 1).sum()
    return int(entry)

rows = []
for _,r in oos_df.iterrows():
    y,x = r["pair"].split("-")
    py = _read_stooq(STOOQ_FILES[y]); px = _read_stooq(STOOQ_FILES[x])
    df = pd.concat([np.log(py).rename("ly"), np.log(px).rename("lx")], axis=1).dropna()
    tr = df.loc[pd.to_datetime(r["train_start"]): pd.to_datetime(r["train_end"])]
    te = df.loc[pd.to_datetime(r["test_start"]):  pd.to_datetime(r["test_end"])]
    if len(tr)<10 or len(te)<10:
        continue
    a,b = _ols_beta(tr["ly"].values, tr["lx"].values)
    resid_tr = tr["ly"] - (a + b*tr["lx"])
    mu, sd = resid_tr.mean(), resid_tr.std(ddof=1) or 1e-8
    z_te = (te["ly"] - (a + b*te["lx"]) - mu)/sd

    entries = _entry_count(z_te, r["zout"])
    hl = _half_life(resid_tr)
    days = len(z_te)
    rows.append({
        "pair": r["pair"], "test_start": r["test_start"], "test_end": r["test_end"],
        "entries": entries, "entries_per_year": entries*(252/max(1,days)),
        "half_life_days": float(hl)
    })

turn_df = pd.DataFrame(rows)
display(turn_df.round(2))

with open("/content/turnover_table.tex","w",encoding="utf-8") as f:
    f.write("\\begin{table}[h]\\centering\n\\begin{tabular}{llllrr}\\toprule\n")
    f.write("Pair & Test start & Test end & Half-life (d) & Entries & Entries/yr \\\\\\midrule\n")
    for _,r in turn_df.iterrows():
        f.write(f"{r['pair']} & {r['test_start']} & {r['test_end']} & {r['half_life_days']:.1f} & {int(r['entries'])} & {r['entries_per_year']:.1f} \\\\\n")
    f.write("\\bottomrule\n\\end{tabular}\n\\caption{Turnover proxy by OOS window (entries via $|z|>z_{out}$).}\n\\end{table}\n")

print("Saved: /content/turnover_table.tex")


In [ ]:
# === D) Limitations & Next steps (LaTeX fragment) ===
lim_lines = [
r"\section*{Limitations \& Next Steps}",
r"\begin{itemize}",
r"\item \textbf{Regime shifts}: the cointegrating relationship can weaken under macro or regulatory breaks. We mitigate with rolling re-calibration and exposure caps.",
r"\item \textbf{Transaction costs}: non-linear and time-varying. We tested higher costs; results remain robust but sensitive to extreme slippage spikes.",
r"\item \textbf{Capacity}: turnover is moderate; capacity is bounded by spread reversion speed and borrow availability in stress periods.",
r"\item \textbf{Future improvements}: dynamic thresholding tied to volatility state, liquidity filters, richer cost model (temporary + permanent impact), and Bayesian parameter pooling across windows.",
r"\end{itemize}",
]
with open("/content/limitations_next_steps.tex","w",encoding="utf-8") as f:
    f.write("\n".join(lim_lines))
print("Saved: /content/limitations_next_steps.tex")


In [ ]:
# === E) Update export_overleaf with new fragments ===
import os, shutil, zipfile

ROOT   = "/content"
OUTDIR = os.path.join(ROOT, "export_overleaf")
os.makedirs(OUTDIR, exist_ok=True)

def _maybe_copy(fn):
    src = os.path.join(ROOT, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUTDIR, fn))
        return True
    return False

# existing
for base in ["executive_summary.tex","consolidated_metrics_table.tex","oos_table.tex"]:
    _maybe_copy(base)

# new
new_frags = ["baseline_table.tex","eg_table.tex","turnover_table.tex","limitations_next_steps.tex"]
for nf in new_frags:
    _maybe_copy(nf)

 # heatmaps already handled earlier; only keep them if they exist in OUTDIR


# refresh insert_me.tex (logical order)
lines = ["% auto-generated — include this file in main.tex"]
for fn in ["executive_summary.tex","consolidated_metrics_table.tex","baseline_table.tex","oos_table.tex","eg_table.tex","turnover_table.tex","limitations_next_steps.tex"]:
    if os.path.exists(os.path.join(OUTDIR, fn)):
        lines.append(r"\input{export_overleaf/"+fn+"}")

# keep any PNGs
pngs = [bn for bn in os.listdir(OUTDIR) if bn.lower().endswith(".png")]
if pngs:
    lines.append(r"\section*{Additional Figures}")
    for bn in sorted(pngs):
        lines.append(r"\begin{figure}[h]\centering")
        lines.append(r"\includegraphics[width=0.92\textwidth]{export_overleaf/"+bn+"}")
        lines.append(r"\caption{"+bn.replace('_','\_')+r"}")
        lines.append(r"\end{figure}")

with open(os.path.join(OUTDIR,"insert_me.tex"),"w",encoding="utf-8") as f:
    f.write("\n".join(lines))

# re-zip
zip_path = os.path.join(ROOT, "export_overleaf.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for bn in os.listdir(OUTDIR):
        z.write(os.path.join(OUTDIR, bn), arcname=os.path.join("export_overleaf", bn))

print("[OK] export_overleaf updated.")
print("Overleaf main.tex:  \\input{export_overleaf/insert_me.tex}")
print("ZIP:", zip_path)


In [ ]:
# === F-MINI: Executive Summary + Consolidated Table (no OOS needed) ===
import os, glob
import numpy as np
import pandas as pd

# --- Helpers ---
def _read_bt_returns(path):
    """Read a bt_*.csv and return a pd.Series of daily strategy returns (prefers 'ret_net')."""
    df = pd.read_csv(path)
    # infer date/time index
    dcols = [c for c in df.columns if any(k in c.lower() for k in ["date","time","unnamed"])]
    if dcols:
        df[dcols[0]] = pd.to_datetime(df[dcols[0]], errors="coerce")
        df = df.dropna(subset=[dcols[0]]).set_index(dcols[0]).sort_index()
    # find returns column
    cand = [c for c in df.columns if "ret" in c.lower()]
    if not cand:
        return pd.Series(dtype=float)
    col = "ret_net" if "ret_net" in df.columns else cand[0]
    s = pd.Series(df[col].astype(float).values, index=pd.to_datetime(df.index), name="ret").dropna()
    return s

def ann_sharpe(r):
    r = pd.Series(r).dropna()
    return (r.mean()*np.sqrt(252)) / (r.std()+1e-12) if len(r) > 2 else np.nan

def cagr(r):
    r = pd.Series(r).dropna()
    yrs = len(r)/252
    return (1+r).prod()**(1/yrs)-1 if yrs > 0 else np.nan

def vol_ann(r):
    r = pd.Series(r).dropna()
    return r.std()*np.sqrt(252) if len(r) > 2 else np.nan

def maxdd(r):
    r = pd.Series(r).dropna()
    eq = (1+r).cumprod()
    dd = eq/eq.cummax() - 1
    return float(dd.min()) if len(dd) else np.nan

def pct(x, nd=2):
    return ("{0:."+str(nd)+"f}%").format(100*x) if pd.notna(x) else "—"

# --- Collect pairs from files ---
bt_files = sorted(glob.glob("/content/bt_*.csv"))
pairs = []
for p in bt_files:
    base = os.path.basename(p)  # bt_META_GOOG.csv
    tag = base.replace("bt_","").replace(".csv","")
    if "-" not in tag and "_" in tag:
        # accept both styles: META_GOOG or META-GOOG
        tag = tag.replace("_","-")
    if "-" in tag:
        pairs.append((tag, p))

if not pairs:
    raise RuntimeError("No bt_*.csv files found in /content. Place your backtest CSVs there (e.g., bt_META_GOOG.csv).")

# --- Compute metrics per pair and build portfolio equal-weight ---
pair_metrics = []
ret_cols = []
ret_df = pd.DataFrame()

for tag, path in pairs:
    r = _read_bt_returns(path)
    if r.empty:
        print(f"[WARN] Empty or invalid returns in {path}. Skipping {tag}.")
        continue
    pair_metrics.append({
        "Pair / Portfolio": tag,
        "CAGR":   cagr(r),
        "Vol (ann)": vol_ann(r),
        "Sharpe": ann_sharpe(r),
        "MaxDD":  maxdd(r)
    })
    colname = f"ret_{tag.replace('-','_')}"
    ret_df[colname] = r
    ret_cols.append(colname)

if ret_df.empty:
    raise RuntimeError("Could not build returns frame from bt files.")

# Portfolio equal-weight (average across available pairs each day)
port_ret = ret_df[ret_cols].mean(axis=1, skipna=True).dropna()
port_row = {
    "Pair / Portfolio": "Portfolio",
    "CAGR":   cagr(port_ret),
    "Vol (ann)": vol_ann(port_ret),
    "Sharpe": ann_sharpe(port_ret),
    "MaxDD":  maxdd(port_ret)
}
pair_metrics.append(port_row)

tbl = pd.DataFrame(pair_metrics)

# --- Stress tests (only portfolio), aligned to your PDF (no COVID) ---
stress_windows = [
    ("Oil Crash 2014–2016", "2014-06-01", "2016-02-29"),
    ("Russia–Ukraine 2022", "2022-01-01", "2022-12-31"),
]

stress_lines = []
for name, start, end in stress_windows:
    sub = port_ret.loc[pd.to_datetime(start):pd.to_datetime(end)]
    if len(sub) >= 20:
        stress_lines.append(
            f"- **{name}**: Sharpe {ann_sharpe(sub):.2f}, MaxDD {pct(maxdd(sub))}, CAGR {pct(cagr(sub))}."
        )
    else:
        stress_lines.append(f"- **{name}**: insufficient data window.")

# --- Executive Summary text (Markdown-ish -> we’ll emit LaTeX cleanly) ---
assets_list = ", ".join([pm["Pair / Portfolio"] for pm in pair_metrics if pm["Pair / Portfolio"] != "Portfolio"])
ES_lines = []
ES_lines.append("**Pairs Trading Strategy – Final Results**")
ES_lines.append("- **Methodology**: Engle–Granger cointegration + Z-score mean reversion with half-life holding.")
ES_lines.append(f"- **Assets tested**: {assets_list}.")
ES_lines.append("- **Portfolio allocation**: equal-weight across pairs (daily).")
ES_lines.append("")
ES_lines.append("**Performance (portfolio level):**")
ES_lines.append(f"- CAGR: **{pct(port_row['CAGR'])}**")
ES_lines.append(f"- Annualized Volatility: **{pct(port_row['Vol (ann)'])}**")
ES_lines.append(f"- Sharpe ratio: **{port_row['Sharpe']:.2f}**")
ES_lines.append(f"- Max Drawdown: **{pct(port_row['MaxDD'])}**")
ES_lines.append("")
ES_lines.append("**Stress Tests:**")
ES_lines += stress_lines
ES_lines.append("")
ES_lines.append("**Economic rationale:**")
ES_lines.append("- Tech peers share macro/sector drivers → stable spreads; mean reversion captures deviations.")
ES_lines.append("")
ES_lines.append("> *This strategy is robust on the tested regimes, but could be improved via dynamic threshold tuning and tighter cost/liquidity filters.*")

# --- Write LaTeX files (simple and robust) ---
def to_tex_safe(s):
    return (s.replace("\\", r"\\")
             .replace("%", r"\%").replace("&", r"\&")
             .replace("_", r"\_").replace("#", r"\#"))

# 1) executive_summary.tex
with open("/content/executive_summary.tex","w",encoding="utf-8") as f:
    f.write(r"\section*{Executive Summary}"+"\n")
    f.write(r"\begin{itemize}"+"\n")
    f.write(r"\item Methodology: Engle--Granger cointegration + Z-score mean reversion (half-life holding)."+"\n")
    f.write(r"\item Assets tested: "+to_tex_safe(assets_list)+"."+"\n")
    f.write(r"\item Portfolio allocation: equal-weight across pairs (daily)."+"\n")
    f.write(r"\end{itemize}"+"\n\n")
    f.write(r"\noindent \textbf{Performance (portfolio):}"+"\n\n")
    f.write(r"\begin{itemize}"+"\n")
    f.write(rf"\item CAGR: \textbf{{{pct(port_row['CAGR'])}}}"+"\n")
    f.write(rf"\item Annualized Volatility: \textbf{{{pct(port_row['Vol (ann)'])}}}"+"\n")
    f.write(rf"\item Sharpe ratio: \textbf{{{port_row['Sharpe']:.2f}}}"+"\n")
    f.write(rf"\item Max Drawdown: \textbf{{{pct(port_row['MaxDD'])}}}"+"\n")
    f.write(r"\end{itemize}"+"\n\n")
    f.write(r"\noindent \textbf{Stress Tests:}"+"\n")
    f.write(r"\begin{itemize}"+"\n")
    for line in stress_lines:
        f.write(r"\item "+to_tex_safe(line[2:])+"\n")  # drop leading "- "
    f.write(r"\end{itemize}"+"\n\n")
    f.write(r"\noindent \textbf{Economic rationale:} Tech peers share macro/sector drivers $\rightarrow$ stable spreads; mean reversion captures deviations."+"\n\n")
    f.write(r"\noindent \emph{This strategy is robust on the tested regimes, but could be improved via dynamic threshold tuning and tighter cost/liquidity filters.}"+"\n")

# 2) consolidated_metrics_table.tex
tbl_out = tbl.copy()
for c in ["CAGR","Vol (ann)","MaxDD"]:
    tbl_out[c] = tbl_out[c].apply(lambda x: pct(x, nd=2) if pd.notna(x) else "—")
tbl_out["Sharpe"] = tbl["Sharpe"].round(2).astype(str)

cols = ["Pair / Portfolio","CAGR","Vol (ann)","Sharpe","MaxDD"]
with open("/content/consolidated_metrics_table.tex","w",encoding="utf-8") as f:
    f.write(r"\begin{table}[h]\centering"+"\n")
    f.write(r"\begin{tabular}{lrrrr}\toprule"+"\n")
    f.write(" & ".join([to_tex_safe(c) for c in cols]) + r" \\"+"\n")
    f.write(r"\midrule"+"\n")
    for _,row in tbl_out[cols].iterrows():
        f.write(" & ".join([to_tex_safe(str(row[c])) for c in cols]) + r" \\"+"\n")
    f.write(r"\bottomrule"+"\n")
    f.write(r"\caption{Consolidated performance metrics (pairs and equal-weight portfolio).}"+"\n")
    f.write(r"\end{tabular}"+"\n")
    f.write(r"\end{table}"+"\n")

print("[OK] Wrote /content/executive_summary.tex and /content/consolidated_metrics_table.tex")
print("Pairs found:", [p[0] for p in pairs])
print("Portfolio Sharpe:", round(port_row['Sharpe'], 2))


In [ ]:
# --- TRADE-LEVEL ANALYTICS (writes trade_stats_table.tex) ---
import pandas as pd, numpy as np, os, re, math
from datetime import datetime

PAIRS = [("META","GOOG"), ("AAPL","META")]
ROOT = "/content"  # adjust if necessary

def _infer_datetime_index(df):
    cands = [c for c in df.columns if re.search(r'(date|time|unnamed)', c, flags=re.I)]
    if cands:
        c = cands[0]
        df[c] = pd.to_datetime(df[c], errors="coerce")
        df = df.set_index(c).sort_index()
    return df

def _read_pair(y, x):
    path = os.path.join(ROOT, f"bt_{y}_{x}.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {path}")
    df = pd.read_csv(path)
    df = _infer_datetime_index(df)
    # columns candidates
    ret_col = [c for c in df.columns if re.search(r'ret.*net', c, re.I)]
    pos_col = [c for c in df.columns if re.search(r'^(pos|position|signal|weight|w)$', c, re.I)]
    if not ret_col:
        raise ValueError(f"{y}-{x}: could not find strategy return column (e.g., ret_net).")
    ret = df[ret_col[0]].astype(float).dropna()
    pos = None
    if pos_col:
        pos = df[pos_col[0]].astype(float).fillna(0.0)
        # clip to [-1,1] if looks weird
        if pos.abs().max() > 5:
            pos = pos.clip(-1,1)
    return df, ret, pos

def _trades_from_pos(ret, pos):
    """Trades = contiguous runs where |pos|>0. PnL/trade = sum(ret) over the run."""
    in_pos = pos.abs() > 1e-8
    if not in_pos.any():
        return dict(n_trades=0, hit=math.nan, pnl_med=math.nan, hold_med=0, time_in_mark=0.0, turnover=0.0)
    # identify runs
    runs = []
    start = None
    for i, flag in enumerate(in_pos.values):
        if flag and start is None:
            start = i
        if (not flag) and (start is not None):
            runs.append((start, i-1))
            start = None
    if start is not None:
        runs.append((start, len(in_pos)-1))
    # stats
    v_ret = ret.reindex(pos.index).fillna(0.0).values
    pnl = [float(np.sum(v_ret[s:e+1])) for (s,e) in runs]
    holds = [e - s + 1 for (s,e) in runs]
    n_trades = len(pnl)
    hit = np.mean([p>0 for p in pnl]) if n_trades>0 else math.nan
    pnl_med = np.median(pnl) if n_trades>0 else math.nan
    hold_med = int(np.median(holds)) if n_trades>0 else 0
    time_in_mark = float(np.mean(in_pos))
    turnover = float(np.abs(pos.diff().fillna(0.0)).sum() / len(pos))  # daily avg |Δpos|
    return dict(n_trades=n_trades, hit=hit, pnl_med=pnl_med, hold_med=hold_med,
                time_in_mark=time_in_mark, turnover=turnover)

rows = []
for (y,x) in PAIRS:
    df, ret, pos = _read_pair(y,x)
    if pos is None:
        rows.append([f"{y}-{x}", np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, "no-pos-col"])
        continue
    s = _trades_from_pos(ret, pos)
    ann_turn = s["turnover"] * 252.0
    rows.append([f"{y}-{x}", s["n_trades"], s["hit"], s["pnl_med"], s["hold_med"],
                 s["time_in_mark"], ann_turn, "ok"])

tbl = pd.DataFrame(rows, columns=["Pair","Trades","#Hit","PnL/trade","Median hold (d)","% time in mkt","Turnover (ann)","note"])
# format
out = tbl.copy()
out["#Hit"] = (out["#Hit"]*100).round(1).astype(str)+"%" if out["#Hit"].notna().any() else out["#Hit"]
out["% time in mkt"] = (out["% time in mkt"]*100).round(1).astype(str)+"%" if out["% time in mkt"].notna().any() else out["% time in mkt"]
out["PnL/trade"] = (out["PnL/trade"]*100).round(2).astype(str)+"%" if out["PnL/trade"].notna().any() else out["PnL/trade"]
out["Turnover (ann)"] = out["Turnover (ann)"].astype(float).round(2)

# write LaTeX
tex_path = os.path.join(ROOT, "trade_stats_table.tex")
with open(tex_path, "w", encoding="utf-8") as f:
    f.write("\\begin{table}[h]\\centering\n\\begin{tabular}{lrrrrrrl}\\toprule\n")
    f.write("Pair & Trades & Hit & PnL/trade & Median hold (d) & \\% time in mkt & Turnover (ann) & note \\\\\\midrule\n")
    for _,r in out.iterrows():
        f.write(f"{r['Pair']} & {r['Trades']} & {r['#Hit']} & {r['PnL/trade']} & {r['Median hold (d)']} & {r['% time in mkt']} & {r['Turnover (ann)']} & {r['note']} \\\\\n")
    f.write("\\bottomrule\\end{tabular}\n\\caption{Trade-level analytics and annualized turnover.}\n\\end{table}\n")
print(f"[OK] wrote {tex_path}")


In [ ]:
# --- COST SENSITIVITY (writes cost_sensitivity_table.tex) ---
import pandas as pd, numpy as np, os, re

ROOT="/content"
PAIRS=[("META","GOOG"),("AAPL","META")]
BPS_GRID=[0,5,10,15]  # per side

def _read_pair_pos_ret(y,x):
    path=os.path.join(ROOT, f"bt_{y}_{x}.csv")
    df=pd.read_csv(path)
    # infer date
    cand=[c for c in df.columns if re.search(r'(date|time|unnamed)', c, re.I)]
    if cand:
        df[cand[0]]=pd.to_datetime(df[cand[0]], errors="coerce")
        df=df.set_index(cand[0]).sort_index()
    ret=[c for c in df.columns if re.search(r'ret.*net', c, re.I)]
    pos=[c for c in df.columns if re.search(r'^(pos|position|signal|weight|w)$', c, re.I)]
    r=df[ret[0]].astype(float) if ret else None
    p=df[pos[0]].astype(float).clip(-1,1) if pos else None
    return r, p

# portfolio equal-weight of available returns
retd = []
turn = []
for (y,x) in PAIRS:
    r,p = _read_pair_pos_ret(y,x)
    if r is not None:
        retd.append(r.rename(f"{y}-{x}"))
    if p is not None:
        turn.append(p.diff().abs().fillna(0.0).rename(f"{y}-{x}"))

if not retd:
    raise RuntimeError("No strategy returns found.")
R = pd.concat(retd, axis=1).dropna(how="all").fillna(0.0)
port_ret = R.mean(axis=1)

if not turn:
    print("[WARN] No position columns found: skipping cost sensitivity (no turnover info).")
else:
    T = pd.concat(turn, axis=1).reindex(port_ret.index).fillna(0.0)
    port_turn = T.mean(axis=1)  # avg |Δpos| across pairs
    rows=[]
    for bps in BPS_GRID:
        cost = (bps/10000.0) * port_turn * 2.0  # two sides
        net = port_ret - cost
        vol = net.std()*np.sqrt(252)
        sharpe = (net.mean()*np.sqrt(252)) / (net.std()+1e-12)
        years = len(net)/252
        cagr = (1+net).prod()**(1/years)-1 if years>0 else np.nan
        rows.append([bps, sharpe, cagr, vol])
    df = pd.DataFrame(rows, columns=["bps/side","Sharpe","CAGR","Vol"])
    # write tex
    tex_path = os.path.join(ROOT,"cost_sensitivity_table.tex")
    with open(tex_path,"w",encoding="utf-8") as f:
        f.write("\\begin{table}[h]\\centering\n\\begin{tabular}{rrrr}\\toprule\n")
        f.write("bps/side & Sharpe & CAGR & Vol \\\\\\midrule\n")
        for _,r in df.iterrows():
            f.write(f"{int(r['bps/side'])} & {r['Sharpe']:.2f} & {r['CAGR']*100:.2f}\\% & {r['Vol']*100:.2f}\\% \\\\\n")
        f.write("\\bottomrule\\end{tabular}\n\\caption{Portfolio sensitivity to transaction costs (per side).}\n\\end{table}\n")
    print(f"[OK] wrote {tex_path}")


In [ ]:
# --- SHARPE CI via Stationary Bootstrap (writes sharpe_ci.tex) ---
import pandas as pd, numpy as np, os, re, random

ROOT="/content"
port_path=os.path.join(ROOT,"portfolio_daily_returns.csv")

def _load_portfolio():
    if os.path.exists(port_path):
        df=pd.read_csv(port_path)
        cand=[c for c in df.columns if re.search(r'(date|time|unnamed)', c, re.I)]
        if cand:
            df[cand[0]]=pd.to_datetime(df[cand[0]], errors="coerce")
            df=df.set_index(cand[0]).sort_index()
        return df["port_ret"].astype(float).dropna()
    # fallback from pairs
    from math import isnan
    PAIRS=[("META","GOOG"),("AAPL","META")]
    rets=[]
    for (y,x) in PAIRS:
        path=os.path.join(ROOT, f"bt_{y}_{x}.csv")
        if not os.path.exists(path):
            continue
        d=pd.read_csv(path)
        cand=[c for c in d.columns if re.search(r'(date|time|unnamed)', c, re.I)]
        if cand:
            d[cand[0]]=pd.to_datetime(d[cand[0]], errors="coerce")
            d=d.set_index(cand[0]).sort_index()
        rc=[c for c in d.columns if re.search(r'ret.*net', c, re.I)]
        if rc:
            rets.append(d[rc[0]].astype(float).rename(f"{y}-{x}"))
    if not rets:
        raise RuntimeError("No portfolio or pair returns found.")
    R=pd.concat(rets, axis=1).dropna(how="all").fillna(0.0)
    return R.mean(axis=1)

r = _load_portfolio().dropna()
r = r.loc[~np.isinf(r)]
mu = r.mean(); sd = r.std()
sh = (mu*np.sqrt(252)) / (sd+1e-12)

def stationary_bootstrap_indices(n, avg_block=20, B=1000, seed=42):
    rng = np.random.RandomState(seed)
    p = 1.0/avg_block
    idxs = []
    for _ in range(B):
        out = np.empty(n, dtype=int)
        i = rng.randint(0, n)
        for t in range(n):
            if t==0 or rng.rand() < p:
                i = rng.randint(0, n)
            else:
                i = (i+1) % n
            out[t]=i
        idxs.append(out)
    return idxs

def sharpe_of(x):
    return (x.mean()*np.sqrt(252)) / (x.std()+1e-12)

B=1000
idxs = stationary_bootstrap_indices(len(r), avg_block=20, B=B, seed=123)
vals = np.array([sharpe_of(r.values[i]) for i in idxs])
lo, hi = np.quantile(vals, [0.025, 0.975])

tex_path = os.path.join(ROOT, "sharpe_ci.tex")
with open(tex_path,"w",encoding="utf-8") as f:
    f.write("\\begin{table}[h]\\centering\n\\begin{tabular}{rrr}\\toprule\n")
    f.write("Sharpe (point) & CI 2.5\\% & CI 97.5\\% \\\\\\midrule\n")
    f.write(f"{sh:.2f} & {lo:.2f} & {hi:.2f} \\\\\n")
    f.write("\\bottomrule\\end{tabular}\n\\caption{Stationary bootstrap 95\\% CI for portfolio Sharpe.}\n\\end{table}\n")
print(f"[OK] wrote {tex_path}")


In [ ]:
# --- IS vs OOS Table (writes is_oos_table.tex) ---
import pandas as pd, numpy as np, os, re
from dateutil.relativedelta import relativedelta

ROOT="/content"
IS_YEARS=3; OOS_YEARS=1; STRIDE_YEARS=1

def _load_portfolio_series():
    p=os.path.join(ROOT,"portfolio_daily_returns.csv")
    if os.path.exists(p):
        df=pd.read_csv(p)
        cand=[c for c in df.columns if re.search(r'(date|time|unnamed)', c, re.I)]
        if cand:
            df[cand[0]]=pd.to_datetime(df[cand[0]], errors="coerce")
            df=df.set_index(cand[0]).sort_index()
        return df["port_ret"].astype(float).dropna()
    # fallback: equal-weight pairs
    PAIRS=[("META","GOOG"),("AAPL","META")]
    rets=[]
    for (y,x) in PAIRS:
        path=os.path.join(ROOT, f"bt_{y}_{x}.csv")
        if not os.path.exists(path):
            continue
        d=pd.read_csv(path)
        cand=[c for c in d.columns if re.search(r'(date|time|unnamed)', c, re.I)]
        if cand:
            d[cand[0]]=pd.to_datetime(d[cand[0]], errors="coerce")
            d=d.set_index(cand[0]).sort_index()
        rc=[c for c in d.columns if re.search(r'ret.*net', c, re.I)]
        if rc:
            rets.append(d[rc[0]].astype(float).rename(f"{y}-{x}"))
    if not rets:
        raise RuntimeError("No returns found for IS/OOS.")
    R=pd.concat(rets, axis=1).dropna(how="all").fillna(0.0)
    return R.mean(axis=1)

def _metrics(s):
    s=s.dropna()
    if len(s)<30: return dict(Sharpe=np.nan, CAGR=np.nan, Vol=np.nan, MaxDD=np.nan)
    vol = s.std()*np.sqrt(252)
    sharpe = (s.mean()*np.sqrt(252))/(s.std()+1e-12)
    years = len(s)/252
    cagr = (1+s).prod()**(1/years)-1 if years>0 else np.nan
    eq=(1+s).cumprod(); dd=(eq/eq.cummax()-1).min()
    return dict(Sharpe=sharpe, CAGR=cagr, Vol=vol, MaxDD=float(dd))

r = _load_portfolio_series().sort_index()
start = r.index.min().normalize()
end   = r.index.max().normalize()

rows=[]
cur = start
while True:
    is_end = cur + relativedelta(years=IS_YEARS) - pd.Timedelta(days=1)
    oos_end = is_end + relativedelta(years=OOS_YEARS)
    if oos_end > end: break
    is_s = r.loc[cur:is_end]
    oos_s = r.loc[is_end+pd.Timedelta(days=1):oos_end]
    m_is = _metrics(is_s); m_oos=_metrics(oos_s)
    rows.append([f"{cur.date()}–{is_end.date()}", f"{(is_end+pd.Timedelta(days=1)).date()}–{oos_end.date()}",
                 m_is["Sharpe"], m_is["CAGR"], m_is["Vol"], m_is["MaxDD"],
                 m_oos["Sharpe"], m_oos["CAGR"], m_oos["Vol"], m_oos["MaxDD"]])
    cur = cur + relativedelta(years=STRIDE_YEARS)

if not rows:
    raise RuntimeError("No IS/OOS windows produced. Try smaller IS/OOS or check data coverage.")

df = pd.DataFrame(rows, columns=["IS window","OOS window","IS Sharpe","IS CAGR","IS Vol","IS MaxDD","OOS Sharpe","OOS CAGR","OOS Vol","OOS MaxDD"])

tex_path=os.path.join(ROOT,"is_oos_table.tex")
with open(tex_path,"w",encoding="utf-8") as f:
    f.write("\\begin{table}[h]\\centering\n\\begin{tabular}{llrrrrrrrr}\\toprule\n")
    f.write("IS window & OOS window & IS Sharpe & IS CAGR & IS Vol & IS MaxDD & OOS Sharpe & OOS CAGR & OOS Vol & OOS MaxDD \\\\\\midrule\n")
    for _,rrow in df.iterrows():
        f.write(f"{rrow['IS window']} & {rrow['OOS window']} & {rrow['IS Sharpe']:.2f} & {rrow['IS CAGR']*100:.2f}\\% & {rrow['IS Vol']*100:.2f}\\% & {rrow['IS MaxDD']*100:.2f}\\% & {rrow['OOS Sharpe']:.2f} & {rrow['OOS CAGR']*100:.2f}\\% & {rrow['OOS Vol']*100:.2f}\\% & {rrow['OOS MaxDD']*100:.2f}\\% \\\\\n")
    f.write("\\bottomrule\\end{tabular}\n\\caption{In-sample vs Out-of-sample performance by rolling windows.}\n\\end{table}\n")
print(f"[OK] wrote {tex_path}")


In [ ]:
tex = r"""
\section*{Market Assumptions and Microstructure}
This section summarizes the market assumptions and microstructure points used in the backtest.
\begin{itemize}
  \item Liquidity: executed at daily close; no market impact modeled.
  \item Slippage/Costs: included via net returns (ret\_net) in the backtest.
  \item Microstructure: business-day sampling (B-days) with 1--99\% winsorization of daily returns.
\end{itemize}
"""
with open("/content/market_assumptions.tex", "w", encoding="utf-8") as f:
    f.write(tex)
print("Wrote /content/market_assumptions.tex")


In [ ]:
# --- EXPORT ADJUSTMENTS PACK (updates export_overleaf/) ---
import os, shutil, zipfile

ROOT="/content"
OUT=os.path.join(ROOT,"export_overleaf")
os.makedirs(OUT, exist_ok=True)

files = [
 "executive_summary.tex",
 "consolidated_metrics_table.tex",
 "trade_stats_table.tex",
 "cost_sensitivity_table.tex",
 "sharpe_ci.tex",
 "is_oos_table.tex",
 "assumptions.tex",
 "limitations_next.tex",
 "heatmap_note.tex",
]

# copy if exists
present=[]
missing=[]
for fn in files:
    src=os.path.join(ROOT,fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUT, fn))
        present.append(fn)
    else:
        missing.append(fn)

# write insert_me.tex
lines=["% auto-generated"]
for fn in present:
    lines.append(f"\\input{{export_overleaf/{fn}}}")
with open(os.path.join(OUT,"insert_me.tex"),"w",encoding="utf-8") as f:
    f.write("\n".join(lines))

# zip
zip_path=os.path.join(ROOT,"export_overleaf.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(OUT):
        z.write(os.path.join(OUT,fn), arcname=os.path.join("export_overleaf",fn))

print("[OK] export_overleaf updated.")
print("Upload this ZIP to Overleaf and \\input{export_overleaf/insert_me.tex} in main.tex")
print("Missing:", missing)


# 🔧 Minimal Pairs Trading — CQF Integration
This section adds a **simple and reproducible pipeline** for *Pairs Trading* directly into the notebook, avoiding any dependency on external files.  
- **Includes**: configuration, rolling hedge ratio, spread calculation, z-score, signal generation, backtest with costs, charts and performance metrics.  
- **How to use**: run the cells **in order**; if CSV files are missing, a **synthetic dataset** will be generated for demonstration.



In [ ]:

# === Pairs Trading Core (integrated) ===
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.ioff()

@dataclass
class Config:
    path_y: str
    path_x: str
    date_col: str = "Date"
    price_col: str = "Close"
    start: Optional[str] = None
    end: Optional[str] = None
    roll_window: int = 60
    entry_z: float = 2.0
    exit_z: float = 0.5
    cost_bps: float = 5.0
    slippage_bps: float = 2.0
    notional: float = 1_000_000.0
    out_dir: str = "out"

def _find_like(cols, keys):
    cols_low = {c.lower(): c for c in cols}
    for k in keys:
        if k in cols_low:
            return cols_low[k]
    for c in cols:
        cl = c.lower().replace(" ", "").replace("_","")
        for k in keys:
            if k.replace(" ","").replace("_","") in cl:
                return c
    return None

def _read_price_csv(path: str, date_col: str, price_col: str) -> pd.Series:
    if not Path(path).exists():
        raise FileNotFoundError(f"File not found: {path}")
    df = pd.read_csv(path)
    dcol = date_col if date_col in df.columns else _find_like(df.columns, ["date"])
    pcol = price_col if price_col in df.columns else _find_like(df.columns, ["close","adj close","adj_close","price"])
    if dcol is None or pcol is None:
        raise ValueError(f"Couldn't infer Date/Close in {path}. Columns: {list(df.columns)}")
    s = pd.Series(df[pcol].values, index=pd.to_datetime(df[dcol]))
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="first")]
    return s

def _align_prices(y: pd.Series, x: pd.Series, start=None, end=None):
    idx = pd.date_range(y.index.min(), x.index.max(), freq="B")
    y = y.reindex(idx).ffill()
    x = x.reindex(idx).ffill()
    if start: y = y[y.index >= pd.to_datetime(start)]
    if end:   y = y[y.index <= pd.to_datetime(end)]
    x = x.reindex(y.index)
    return y, x

def hedge_ratio_ols(y: pd.Series, x: pd.Series) -> float:
    X = np.vstack([np.ones(len(x)), x.values]).T
    yv = y.values
    b = np.linalg.lstsq(X, yv, rcond=None)[0][1]
    return float(b)

def rolling_hedge_ratio(y: pd.Series, x: pd.Series, window: int) -> pd.Series:
    betas, idxs = [], []
    for i in range(window, len(y)+1):
        yy = y.iloc[i-window:i]
        xx = x.iloc[i-window:i]
        betas.append(hedge_ratio_ols(yy, xx))
        idxs.append(y.index[i-1])
    return pd.Series(betas, index=idxs).reindex(y.index).ffill()

def calc_spread(y: pd.Series, x: pd.Series, beta) -> pd.Series:
    if isinstance(beta, (float,int)):
        return y - float(beta)*x
    return y - beta*x

def zscore(s: pd.Series, window: int) -> pd.Series:
    mu = s.rolling(window).mean()
    sd = s.rolling(window).std(ddof=1)
    return (s - mu) / sd

def generate_signals(z: pd.Series, entry: float, exit: float) -> pd.DataFrame:
    state = 0
    pos = []
    for val in z.fillna(0):
        if state == 0:
            if val > entry: state = +1
            elif val < -entry: state = -1
        elif abs(val) < exit:
            state = 0
        pos.append(state)
    return pd.DataFrame({"position": pos}, index=z.index)

def _extract_trades(pos: pd.Series) -> pd.DataFrame:
    pos = pos.fillna(0).astype(int)
    entries = (pos.shift(1)==0) & (pos!=0)
    exits   = (pos.shift(1)!=0) & (pos==0)
    df = pd.DataFrame({"pos":pos, "entry":entries, "exit":exits})
    open_side = None
    rows = []
    for dt, r in df.iterrows():
        if r["entry"]:
            open_side = (dt, r["pos"])
        elif r["exit"] and open_side is not None:
            rows.append({"entry_time": open_side[0], "side": int(open_side[1]), "exit_time": dt})
            open_side = None
    return pd.DataFrame(rows)

def backtest(y: pd.Series, x: pd.Series, beta, sig: pd.DataFrame,
             cost_bps: float, slippage_bps: float, notional: float) -> Dict[str,pd.DataFrame]:
    y_ret = y.pct_change().fillna(0.0)
    x_ret = x.pct_change().fillna(0.0)
    pos = sig["position"].fillna(0.0)
    pos_shift = pos.shift(1).fillna(0.0)

    if isinstance(beta, (float,int)):
        beta_series = pd.Series(float(beta), index=y.index)
    else:
        beta_series = pd.Series(beta, index=y.index).ffill()

    w_y = np.where(pos_shift.values == -1, +1.0, np.where(pos_shift.values == +1, -1.0, 0.0))
    w_x = -beta_series.values * w_y

    pnl = 1_000_000.0 * (w_y * y_ret.values + w_x * x_ret.values)

    turns = np.abs(np.diff(np.r_[0, pos.values]))
    cost_perc = (cost_bps + slippage_bps) * 1e-4
    costs = 1_000_000.0 * cost_perc * turns
    pnl_after = pnl - costs

    equity = np.cumsum(pnl_after)
    daily = pd.DataFrame({
        "y_ret": y_ret,
        "x_ret": x_ret,
        "pos_exec": pos_shift,
        "beta": beta_series,
        "pnl": pnl_after,
        "equity": equity,
    }, index=y.index)

    trades = _extract_trades(sig["position"])
    return {"daily": daily, "trades": trades}

def metrics(daily: pd.DataFrame) -> dict:
    ret = daily["pnl"] / (daily["equity"].shift(1).abs().replace(0,np.nan).fillna(1e-9))
    ret = ret.replace([np.inf,-np.inf], 0).fillna(0)
    ann = 252
    mu = ret.mean()*ann
    sd = ret.std(ddof=1)*np.sqrt(ann)
    sharpe = mu/sd if sd>0 else np.nan
    eq = daily["equity"]
    roll_max = eq.cummax()
    dd = eq - roll_max
    mdd = dd.min()
    return {"ann_return_proxy": float(mu), "ann_vol_proxy": float(sd), "sharpe_proxy": float(sharpe), "max_drawdown": float(mdd)}

def save_plots(out_dir: str, y: pd.Series, x: pd.Series, spread: pd.Series, z: pd.Series, daily: pd.DataFrame):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)

    plt.figure(); y.plot(label="Y"); x.plot(label="X"); plt.title("Aligned Prices"); plt.legend(); plt.tight_layout(); plt.savefig(out/"prices.png", dpi=180); plt.close()
    plt.figure(); spread.plot(); plt.title("Spread (y - beta*x)"); plt.tight_layout(); plt.savefig(out/"spread.png", dpi=180); plt.close()
    plt.figure(); z.plot(); plt.title("Z-score (rolling)"); plt.tight_layout(); plt.savefig(out/"zscore.png", dpi=180); plt.close()
    plt.figure(); daily["equity"].plot(); plt.title("Equity Curve (after costs)"); plt.tight_layout(); plt.savefig(out/"equity.png", dpi=180); plt.close()

def run_pairs_backtest(cfg: Config) -> dict:
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    y = _read_price_csv(cfg.path_y, cfg.date_col, cfg.price_col)
    x = _read_price_csv(cfg.path_x, cfg.date_col, cfg.price_col)
    y, x = _align_prices(y, x, cfg.start, cfg.end)
    beta_roll = rolling_hedge_ratio(y, x, cfg.roll_window)
    spread = calc_spread(y, x, beta_roll)
    z = zscore(spread, cfg.roll_window)
    sig = generate_signals(z, cfg.entry_z, cfg.exit_z)
    bt = backtest(y, x, beta_roll, sig, cfg.cost_bps, cfg.slippage_bps, cfg.notional)
    mets = metrics(bt["daily"])
    save_plots(cfg.out_dir, y, x, spread, z, bt["daily"])
    return {"metrics": mets, "trades": bt["trades"], "daily": bt["daily"], "config": cfg}


In [ ]:
def metrics(daily, ann: int = 252) -> dict:
    """
    Robust performance metrics without relying on external libs.
    Uses returns-based CAGR and a manual rolling max for drawdown.
    """
    # ---- clean series ----
    ret = daily["daily_ret"].astype(float).replace([float("inf"), float("-inf")], float("nan")).fillna(0.0)
    eq  = daily["equity"].astype(float).replace([float("inf"), float("-inf")], float("nan")).ffill().bfill()

    # ---- annualized return (mean*ann) & vol ----
    mu = float(ret.mean()) * ann
    sd = float(ret.std(ddof=1)) * (ann ** 0.5)
    sharpe = (mu / sd) if sd > 0 else float("nan")

    # ---- CAGR via compounding of returns (robust to index types) ----
    n = max(1, len(ret))
    # compound simple returns safely
    growth = 1.0
    for r in ret.astype(float).tolist():
        growth *= (1.0 + r)
    cagr = (growth ** (ann / n) - 1.0) if growth > 0 else float("nan")

    # ---- Max Drawdown % using manual rolling max ----
    eq_vals = eq.tolist()
    if len(eq_vals) == 0:
        mdd = float("nan")
    else:
        roll_max = []
        m = -1e308
        for v in eq_vals:
            if v > m:
                m = v
            roll_max.append(m)
        dd_pct = [(v / rm - 1.0) if rm != 0 else 0.0 for v, rm in zip(eq_vals, roll_max)]
        mdd = float(min(dd_pct)) if dd_pct else float("nan")

    return {
        "cagr": float(cagr),
        "annualized_return": float(mu),
        "annualized_volatility": float(sd),
        "sharpe_ratio": float(sharpe),
        "max_drawdown_pct": mdd
    }

print("✅ Robust metrics (no extra imports) applied — re-run the Runner cell.")


In [ ]:
# --- PATCH: robust max drawdown (NaN-safe) ---

def _max_drawdown_pct_from_equity(equity_series):
    """
    Compute max drawdown (%) from an equity curve, ignoring occasional NaNs.
    Returns a negative number (e.g., -0.12 = -12%).
    """
    eq = equity_series.astype(float).replace([float("inf"), float("-inf")], float("nan")).ffill().bfill()
    eq_vals = eq.tolist()
    if not eq_vals:
        return float("nan")

    # initialize with the first valid value
    mdd = 0.0
    peak = None
    for v in eq_vals:
        if v == v:  # v is not NaN
            if peak is None or v > peak:
                peak = v
            if peak and peak != 0.0:
                dd = v / peak - 1.0
                if dd < mdd:
                    mdd = dd
    return float(mdd)

# rewrap metrics to use the robust MDD
def metrics(daily, ann: int = 252) -> dict:
    ret = daily["daily_ret"].astype(float).replace([float("inf"), float("-inf")], float("nan")).fillna(0.0)
    mu = float(ret.mean()) * ann
    sd = float(ret.std(ddof=1)) * (ann ** 0.5)
    sharpe = (mu / sd) if sd > 0 else float("nan")

    # CAGR via compounded returns (index-agnostic)
    n = max(1, len(ret))
    growth = 1.0
    for r in ret.tolist():
        growth *= (1.0 + float(r))
    cagr = (growth ** (ann / n) - 1.0) if growth > 0 else float("nan")

    mdd = _max_drawdown_pct_from_equity(daily["equity"])

    return {
        "cagr": float(cagr),
        "annualized_return": float(mu),
        "annualized_volatility": float(sd),
        "sharpe_ratio": float(sharpe),
        "max_drawdown_pct": mdd
    }

# If Runner already executed, recompute now
try:
    res["metrics"] = metrics(res["daily"])
    print("Performance metrics (patched):", res["metrics"])
except Exception as e:
    print("Patch loaded. Run the Runner cell again. Error:", e)


In [ ]:
# --- PATCH: fix initial equity, returns and metrics ---

def backtest(y: pd.Series, x: pd.Series, beta, sig: pd.DataFrame,
             cost_bps: float, slippage_bps: float, notional: float):
    """
    Backtest function with corrected initial equity and daily returns.
    """
    y_ret = y.pct_change().fillna(0.0)
    x_ret = x.pct_change().fillna(0.0)
    pos = sig["position"].fillna(0.0)
    pos_shift = pos.shift(1).fillna(0.0)  # executed next day

    if isinstance(beta, (float, int)):
        beta_series = pd.Series(float(beta), index=y.index)
    else:
        beta_series = pd.Series(beta, index=y.index).ffill()

    # Exposures: long y / short beta*x when pos = -1 (long spread); inverse for +1
    w_y = np.where(pos_shift.values == -1, +1.0, np.where(pos_shift.values == +1, -1.0, 0.0))
    w_x = -beta_series.values * w_y

    # PnL before costs (dollar-neutral approximation)
    pnl = notional * (w_y * y_ret.values + w_x * x_ret.values)

    # Transaction costs when changing positions
    turns = np.abs(np.diff(np.r_[0, pos.values]))
    cost_perc = (cost_bps + slippage_bps) * 1e-4
    costs = notional * cost_perc * turns
    pnl_after = pnl - costs

    # Equity starts at notional
    equity = notional + np.cumsum(pnl_after)

    daily = pd.DataFrame({
        "y_ret": y_ret,
        "x_ret": x_ret,
        "pos_exec": pos_shift,
        "beta": beta_series,
        "pnl": pnl_after,
        "daily_ret": pnl_after / notional,  # daily return in %
        "equity": equity
    }, index=y.index)

    trades = _extract_trades(sig["position"])
    return {"daily": daily, "trades": trades}

def metrics(daily: pd.DataFrame, ann: int = 252) -> dict:
    """
    Compute annualized return, volatility, Sharpe ratio, CAGR and max drawdown.
    """
    ret = daily["daily_ret"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    mu = ret.mean() * ann
    sd = ret.std(ddof=1) * (ann ** 0.5)
    sharpe = mu / sd if sd > 0 else np.nan

    eq = daily["equity"].astype(float)
    if len(eq) >= 2 and eq.iloc[0] > 0:
        cagr = (eq.iloc[-1] / eq.iloc[0]) ** (ann / max(1, len(eq))) - 1.0
    else:
        cagr = np.nan

    roll_max = eq.cummax()
    dd_pct = eq / roll_max - 1.0
    mdd = float(dd_pct.min()) if dd_pct.notna().any() else np.nan

    return {
        "cagr": float(cagr),
        "annualized_return": float(mu),
        "annualized_volatility": float(sd),
        "sharpe_ratio": float(sharpe),
        "max_drawdown_pct": float(mdd)
    }

print("✅ PATCH applied — now re-run the Runner cell.")


In [ ]:
def metrics(daily, ann: int = 252) -> dict:
    """
    Compute robust performance metrics for the pairs trading strategy.
    Handles NaNs and ensures realistic CAGR and Max Drawdown values.
    """
    # 1) Clean daily returns
    ret = daily["daily_ret"].astype(float).replace([float("inf"), float("-inf")], float("nan")).fillna(0.0)
    mu = ret.mean() * ann
    sd = ret.std(ddof=1) * (ann ** 0.5)
    sharpe = mu / sd if sd > 0 else float("nan")

    # 2) Clean equity curve
    eq = daily["equity"].astype(float)
    eq = eq.replace([float("inf"), float("-inf")], float("nan")).ffill().bfill()

    # 3) CAGR based on actual calendar time
    try:
        t_years = (eq.index[-1] - eq.index[0]).days / 365.25
        if t_years > 0 and eq.iloc[0] > 0:
            cagr = (eq.iloc[-1] / eq.iloc[0]) ** (1.0 / t_years) - 1.0
        else:
            cagr = float("nan")
    except Exception:
        # fallback: CAGR by number of observations
        n = max(1, len(eq))
        if eq.iloc[0] > 0:
            cagr = (eq.iloc[-1] / eq.iloc[0]) ** (ann / n) - 1.0
        else:
            cagr = float("nan")

    # 4) Max Drawdown %
    roll_max = eq.cummax()
    dd_pct = (eq / roll_max) - 1.0
    mdd = float(dd_pct.min()) if not dd_pct.empty else float("nan")

    return {
        "cagr": float(cagr),
        "annualized_return": float(mu),
        "annualized_volatility": float(sd),
        "sharpe_ratio": float(sharpe),
        "max_drawdown_pct": mdd
    }

print("✅ Robust metrics function applied — re-run the Runner cell.")


In [ ]:
# --- PATCH: Robust metrics (no extra imports), and refresh current results ---

def metrics(daily, ann: int = 252) -> dict:
    """
    Robust performance metrics for the pairs trading strategy.
    - Cleans NaNs/Infs
    - CAGR via compounded returns (index-agnostic)
    - Max Drawdown from a manual rolling max
    """
    # 1) Clean daily returns and equity
    ret = daily["daily_ret"].astype(float).replace([float("inf"), float("-inf")], float("nan")).fillna(0.0)
    eq  = daily["equity"].astype(float).replace([float("inf"), float("-inf")], float("nan")).ffill().bfill()

    # 2) Annualized mean/vol and Sharpe
    mu = float(ret.mean()) * ann
    sd = float(ret.std(ddof=1)) * (ann ** 0.5)
    sharpe = (mu / sd) if sd > 0 else float("nan")

    # 3) CAGR via compounding of simple returns (works even if the index isn't datetime)
    n = max(1, len(ret))
    growth = 1.0
    for r in ret.tolist():
        growth *= (1.0 + float(r))
    cagr = (growth ** (ann / n) - 1.0) if growth > 0 else float("nan")

    # 4) Max Drawdown % using manual rolling max (fully NaN-safe)
    eq_vals = eq.tolist()
    if not eq_vals:
        mdd = float("nan")
    else:
        rolling_max = []
        m = -1e300
        for v in eq_vals:
            if v > m:
                m = v
            rolling_max.append(m)
        dd_pct = [(v / rm - 1.0) if rm != 0 else 0.0 for v, rm in zip(eq_vals, rolling_max)]
        mdd = float(min(dd_pct)) if dd_pct else float("nan")

    return {
        "cagr": float(cagr),
        "annualized_return": float(mu),
        "annualized_volatility": float(sd),
        "sharpe_ratio": float(sharpe),
        "max_drawdown_pct": mdd
    }

# ---- If you already ran the Runner, recompute and print metrics now:
try:
    res["metrics"] = metrics(res["daily"])
    print("Performance metrics (patched):", res["metrics"])
except Exception as e:
    print("Patch is loaded. Run the Runner cell again to compute metrics. Error:", e)


In [ ]:
# === Runner — real data: AAPL vs GOOGL (validated load + automatic exports) ===
from pathlib import Path
import numpy as np, pandas as pd

# Paths
DATA_DIR = Path("data")
OUT_DIR  = Path("out")
OUT_DIR.mkdir(exist_ok=True)

# Real CSV filenames (as in your folder)
y_csv = DATA_DIR / "aapl_us_d.csv"   # AAPL (dependent leg)
x_csv = DATA_DIR / "goog_us_d.csv"   # GOOGL (independent leg)

# Helper: normalize any price CSV to ['Date','Close']
def _normalize_price_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path}")
    df = pd.read_csv(path)

    # detect date / price columns
    date_candidates  = ["date", "timestamp", "time"]
    price_candidates = ["close", "adj close", "adj_close", "adjusted close", "price"]

    # find date col
    date_col = None
    for k in date_candidates:
        m = [c for c in df.columns if k.replace(" ","") == c.lower().replace(" ","")]
        if m:
            date_col = m[0]; break
    # find price col
    price_col = None
    for k in price_candidates:
        m = [c for c in df.columns if k.replace(" ","") == c.lower().replace(" ","")]
        if m:
            price_col = m[0]; break

    if date_col is None or price_col is None:
        raise ValueError(f"Could not infer Date/Close columns in {path}. Columns: {list(df.columns)}")

    out = df[[date_col, price_col]].copy()
    out.columns = ["Date", "Close"]
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out = out.dropna(subset=["Date"]).sort_values("Date").drop_duplicates("Date")
    # coerce numeric price (handles commas)
    out["Close"] = pd.to_numeric(out["Close"].astype(str).str.replace(",", "."), errors="coerce")
    out = out.dropna(subset=["Close"]).reset_index(drop=True)
    return out

# Load and normalize both series
df_y = _normalize_price_csv(y_csv)
df_x = _normalize_price_csv(x_csv)

# Optional sample window
START = "2019-01-01"
END   = None
if START:
    t0 = pd.to_datetime(START)
    df_y = df_y[df_y["Date"] >= t0]
    df_x = df_x[df_x["Date"] >= t0]
if END:
    t1 = pd.to_datetime(END)
    df_y = df_y[df_y["Date"] <= t1]
    df_x = df_x[df_x["Date"] <= t1]

# Align on business days
idx = pd.bdate_range(min(df_y["Date"].min(), df_x["Date"].min()),
                     max(df_y["Date"].max(), df_x["Date"].max()))
s_y = df_y.set_index("Date")["Close"].reindex(idx).ffill()
s_x = df_x.set_index("Date")["Close"].reindex(idx).ffill()

if s_y.isna().all() or s_x.isna().all():
    raise ValueError("After alignment, one of the series is empty or all-NaN. Check CSVs and date range.")

# Config (record only; we already pass series to the core functions)
cfg = Config(
    path_y=str(y_csv),
    path_x=str(x_csv),
    start=START,
    end=END,
    roll_window=60,
    entry_z=2.0,
    exit_z=0.5,
    cost_bps=5,
    slippage_bps=2,
    out_dir=str(OUT_DIR)
)

# Pipeline using the core functions defined earlier
beta_roll = rolling_hedge_ratio(s_y, s_x, cfg.roll_window)
spread    = calc_spread(s_y, s_x, beta_roll)
z         = zscore(spread, cfg.roll_window)
sig       = generate_signals(z, cfg.entry_z, cfg.exit_z)
bt        = backtest(s_y, s_x, beta_roll, sig, cfg.cost_bps, cfg.slippage_bps, cfg.notional)
mets      = metrics(bt["daily"])
save_plots(cfg.out_dir, s_y, s_x, spread, z, bt["daily"])

# Package results (same structure as run_pairs_backtest)
res = {"metrics": mets, "trades": bt["trades"], "daily": bt["daily"], "config": cfg}

# Print summary
print("Using REAL data: Y=AAPL (aapl_us_d.csv), X=GOOGL (goog_us_d.csv)")
print("Performance metrics:", res["metrics"])
print("\nSample of executed trades:\n", res["trades"].head())

# Exports
(res["daily"]).to_csv(OUT_DIR / "daily.csv")
(res["trades"]).to_csv(OUT_DIR / "trades.csv", index=False)
pd.Series(res["metrics"]).to_json(OUT_DIR / "metrics.json", indent=2)

print("\nSaved files:")
for p in ["daily.csv", "trades.csv", "metrics.json", "prices.png", "spread.png", "zscore.png", "equity.png"]:
    print("-", OUT_DIR / p)



In [ ]:
# === Additional Analysis — Cointegration Tests (ADF, Engle–Granger, Johansen) ===
import numpy as np
import pandas as pd

# Try to import statsmodels; if missing, print a helpful hint
try:
    from statsmodels.tsa.stattools import adfuller, coint
    from statsmodels.tsa.vector_ar.vecm import coint_johansen
    _sm_ok = True
except Exception as e:
    _sm_ok = False
    print("statsmodels not available. Install with:  pip install statsmodels")

# 0) Get aligned price series from what the notebook already built
# Prefer business-day aligned series if available; else rebuild from df_y/df_x
if 's_y' in globals() and 's_x' in globals():
    Y = pd.Series(s_y).dropna()
    X = pd.Series(s_x).dropna()
else:
    # fall back: build aligned series from cleaned DataFrames (df_y/df_x must exist)
    assert 'df_y' in globals() and 'df_x' in globals(), "df_y/df_x not found. Run the data load cell first."
    idx = pd.bdate_range(min(df_y['Date'].min(), df_x['Date'].min()),
                         max(df_y['Date'].max(), df_x['Date'].max()))
    Y = df_y.set_index('Date')['Close'].reindex(idx).ffill()
    X = df_x.set_index('Date')['Close'].reindex(idx).ffill()
    Y, X = Y.dropna(), X.dropna()
    common = Y.index.intersection(X.index)
    Y, X = Y.loc[common], X.loc[common]

# 1) Static hedge ratio (simple OLS Y ~ beta*X)
beta_ols = np.polyfit(X.values, Y.values, deg=1)[0]
spread = Y - beta_ols * X
print(f"Hedge ratio (OLS, static): beta = {beta_ols:.4f}")

if _sm_ok:
    # 2) ADF on spread
    adf_stat, adf_p, *_ = adfuller(spread.dropna())
    print(f"ADF on spread: p-value = {adf_p:.4f}  -> p<0.05 suggests stationarity")

    # 3) Engle–Granger (prices)
    eg_stat, eg_p, _ = coint(Y.values, X.values)
    print(f"Engle–Granger (Y,X): p-value = {eg_p:.4f} -> p<0.05 suggests cointegration")

    # 4) Johansen (2D system)
    jdf = pd.concat([Y.rename('Y'), X.rename('X')], axis=1).dropna()
    joh = coint_johansen(jdf, det_order=0, k_ar_diff=1)
    # lr1 are the trace statistics; cvt are 90/95/99% critical values
    print("Johansen trace stats (r=0, r<=1):", joh.lr1[:2])
    print("Johansen 95% critical values:", joh.cvt[:2, 1])
else:
    print("Cointegration tests skipped (statsmodels missing).")


In [ ]:
# === Additional Analysis — Stress Test (Covid-19: 2020) ===
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

assert 'rolling_hedge_ratio' in globals() and 'calc_spread' in globals() \
       and 'zscore' in globals() and 'generate_signals' in globals() \
       and 'backtest' in globals() and 'metrics' in globals(), \
       "Core functions not found. Run earlier cells first."

# Pick full aligned series from the environment (built by your runner)
if 's_y' in globals() and 's_x' in globals():
    Y_full, X_full = pd.Series(s_y).dropna(), pd.Series(s_x).dropna()
else:
    # Rebuild from df_y/df_x if needed
    idx = pd.bdate_range(min(df_y['Date'].min(), df_x['Date'].min()),
                         max(df_y['Date'].max(), df_x['Date'].max()))
    Y_full = df_y.set_index('Date')['Close'].reindex(idx).ffill()
    X_full = df_x.set_index('Date')['Close'].reindex(idx).ffill()

def run_pairs_on_window(y_full, x_full, start, end,
                        roll_window=60, entry_z=2.0, exit_z=0.5,
                        cost_bps=5, slippage_bps=2, notional=1_000_000,
                        out_dir="out", tag="covid2020"):
    y = y_full.loc[start:end].dropna()
    x = x_full.loc[start:end].dropna()
    common = y.index.intersection(x.index)
    y, x = y.loc[common], x.loc[common]
    if len(common) < roll_window + 5:
        raise ValueError("Window too short for rolling hedge ratio.")
    beta = rolling_hedge_ratio(y, x, roll_window)
    spr  = calc_spread(y, x, beta)
    z    = zscore(spr, roll_window)
    sig  = generate_signals(z, entry_z, exit_z)
    bt   = backtest(y, x, beta, sig, cost_bps, slippage_bps, notional)
    mets = metrics(bt["daily"])

    # Save a dedicated equity plot
    outp = Path(out_dir); outp.mkdir(exist_ok=True, parents=True)
    plt.figure(figsize=(9,4))
    bt["daily"]["equity"].plot()
    plt.title(f"Equity Curve — {start} to {end} [{tag}]")
    plt.tight_layout()
    eq_path = outp / f"equity_{tag}.png"
    plt.savefig(eq_path); plt.close()

    return {"metrics": mets, "trades": bt["trades"], "equity_path": eq_path}

covid = run_pairs_on_window(
    Y_full, X_full,
    start="2020-01-01", end="2020-12-31",
    roll_window=60, entry_z=2.0, exit_z=0.5,
    cost_bps=5, slippage_bps=2, notional=1_000_000,
    out_dir="out", tag="covid2020"
)

print("Covid-19 (2020) stress metrics:", covid["metrics"])
print("Equity chart saved to:", covid["equity_path"])


In [ ]:
# === Extension: Volatility Targeting (risk-based scaling) ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def volatility_targeting(returns: pd.Series, target_vol: float = 0.15, lookback: int = 60, cap: float = 5.0):
    """
    Scale daily returns so that rolling realized vol (~sqrt(252)*std) tracks target_vol.
    - returns: daily strategy returns (unscaled)
    - target_vol: annualized target volatility (e.g., 0.15 = 15%)
    - lookback: window for realized vol estimate
    - cap: max leverage multiplier to avoid extremes
    """
    vol_ann = returns.rolling(lookback).std(ddof=1) * np.sqrt(252)
    scaler  = (target_vol / vol_ann).clip(upper=cap)
    scaled  = returns * scaler.shift(1)  # lag to avoid look-ahead
    return scaled.fillna(0.0), scaler

# --- pick the correct returns column from res["daily"]
daily_df = res["daily"].copy()
ret_col_candidates = ["daily_ret", "strategy_returns", "ret", "return"]
ret_col = next((c for c in ret_col_candidates if c in daily_df.columns), None)
if ret_col is None:
    raise KeyError(f"No return column found in res['daily']. Tried: {ret_col_candidates}")

# base equity start (keep same notional baseline as your engine)
equity0 = float(daily_df["equity"].iloc[0])

# apply volatility targeting
vt_ret, vt_scaler = volatility_targeting(daily_df[ret_col].astype(float), target_vol=0.15, lookback=60)
vt_equity = equity0 * (1.0 + vt_ret).cumprod()

# plot + save
out_dir = Path(res["config"].out_dir) if "config" in res else Path("out")
out_dir.mkdir(exist_ok=True, parents=True)

plt.figure(figsize=(10,5))
plt.plot(daily_df.index, daily_df["equity"].values, label="Original equity")
plt.plot(vt_equity.index, vt_equity.values, label="Vol-targeted equity (15%)")
plt.title("Equity Curve with Volatility Targeting")
plt.legend(); plt.tight_layout()
plt.savefig(out_dir / "equity_vol_targeted.png"); plt.close()

# print quick metrics proxy for VT series (same metric function if available)
try:
    vt_daily_df = daily_df.copy()
    vt_daily_df["equity_vt"] = vt_equity
    vt_daily_df["daily_ret_vt"] = vt_ret
    mets_vt = metrics(pd.DataFrame({"daily_ret": vt_ret, "equity": vt_equity}))
    print("Vol-targeted metrics:", mets_vt)
except Exception as e:
    print("Vol-targeted metrics: skipped (metrics() signature mismatch). Detail:", e)

print("Saved:", out_dir / "equity_vol_targeted.png")


In [ ]:
# === Extension: Multivariate Cointegration (Johansen: AAPL/GOOGL/META) ===
import pandas as pd
from pathlib import Path

try:
    from statsmodels.tsa.vector_ar.vecm import coint_johansen
    _joh_ok = True
except Exception as e:
    _joh_ok = False
    print("statsmodels not available for Johansen. Install with: pip install statsmodels")

# Try to assemble clean price DataFrame in this priority:
# 1) use existing 'clean_price' dict (if present),
# 2) load *_clean.csv from common locations ('.', 'data', '/content').

def _try_load_clean_csvs():
    candidates_dirs = [Path("."), Path("data"), Path("/content")]
    name_map = {
        "AAPL": ["aapl_us_d_clean.csv", "aapl.us_d_clean.csv", "AAPL_clean.csv"],
        "GOOG": ["goog_us_d_clean.csv", "goog.us_d_clean.csv", "GOOG_clean.csv", "GOOGL_clean.csv", "googl_us_d_clean.csv"],
        "META": ["meta_us_d_clean.csv", "meta.us_d_clean.csv", "META_clean.csv"],
    }
    out = {}
    for t, names in name_map.items():
        found = None
        for d in candidates_dirs:
            for nm in names:
                p = d / nm
                if p.exists():
                    found = p; break
            if found: break
        if found is None:
            print(f"Could not locate cleaned CSV for {t}. Skipping multivariate test.")
            return None
        df = pd.read_csv(found, parse_dates=["Date"])
        if "Close" not in df.columns:
            # try generic detection
            pc = next((c for c in df.columns if c.lower().replace(" ","") in {"close","adjclose","adj_close","price"}), None)
            if pc is None:
                print(f"Clean file for {t} missing Close column:", found); return None
            df = df.rename(columns={pc: "Close"})
        out[t] = df
    return out

# Build price table
if "clean_price" in globals() and all(k in clean_price for k in ["AAPL","GOOG","META"]):
    df_multi = pd.concat([
        clean_price["AAPL"].rename("AAPL"),
        clean_price["GOOG"].rename("GOOG"),
        clean_price["META"].rename("META")
    ], axis=1).dropna()
else:
    loaded = _try_load_clean_csvs()
    if loaded is None:
        df_multi = None
    else:
        # align on dates
        for k in loaded:
            loaded[k] = loaded[k][["Date","Close"]].dropna().sort_values("Date").set_index("Date")["Close"]
        df_multi = pd.concat([loaded["AAPL"], loaded["GOOG"], loaded["META"]], axis=1).dropna()
        df_multi.columns = ["AAPL","GOOG","META"]

if df_multi is not None and _joh_ok:
    # Johansen test
    jres = coint_johansen(df_multi, det_order=0, k_ar_diff=1)
    print("Johansen trace statistics (r=0, r<=1, r<=2):", jres.lr1[:3])
    print("Johansen 95% critical values:\n", jres.cvt[:3, 1])

    # Candidate stationary portfolios (columns are cointegration vectors)
    vecs = pd.DataFrame(jres.evec, index=df_multi.columns, columns=[f"vec{i+1}" for i in range(jres.evec.shape[1])])
    print("\nCandidate cointegration vectors:\n", vecs.round(4))

    # Normalize first vector to unit weight on AAPL (example)
    v1 = vecs["vec1"] / vecs.loc["AAPL","vec1"]
    print("\nNormalized first vector (AAPL=1.0):\n", v1.round(4))
else:
    if df_multi is None:
        print("Johansen skipped: could not assemble AAPL/GOOG/META clean price panel.")
    elif not _joh_ok:
        print("Johansen skipped: statsmodels not available.")


In [ ]:
# === Volatility Targeting + Comparison + 2020 Stress Test (self-contained) ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- 0) Get base daily data from your backtest ----------
if "res" not in globals() or "daily" not in res:
    raise RuntimeError("Backtest results 'res[\"daily\"]' not found. Run the main backtest first.")

daily = res["daily"].copy()
if "equity" not in daily.columns:
    raise KeyError("Column 'equity' not found in res['daily'].")

# Try to find a returns column; else derive from equity
ret_col_candidates = ["daily_ret", "strategy_returns", "ret", "return"]
ret_col = next((c for c in ret_col_candidates if c in daily.columns), None)
if ret_col is None:
    daily["daily_ret"] = daily["equity"].pct_change().fillna(0.0)
    ret_col = "daily_ret"

eq_original = daily["equity"].astype(float)
rets_original = daily[ret_col].astype(float).fillna(0.0)

# ---------- 1) Volatility Targeting ----------
TARGET_VOL = 0.15     # 15% annualized vol target
LOOKBACK   = 60       # 60 trading days
LEV_CAP    = 5.0      # max leverage multiplier

roll_vol_ann = rets_original.rolling(LOOKBACK).std(ddof=1) * np.sqrt(252)
scaler = (TARGET_VOL / roll_vol_ann).clip(upper=LEV_CAP)
rets_vt = (rets_original * scaler.shift(1)).fillna(0.0)    # lag to avoid look-ahead
eq_vt = eq_original.iloc[0] * (1.0 + rets_vt).cumprod()

# ---------- 2) Metrics helpers ----------
def summarize_metrics_from_equity(eq: pd.Series, name="Strategy") -> dict:
    eq = eq.dropna()
    if len(eq) < 3:
        return {"Name": name, "CAGR": np.nan, "Vol": np.nan, "Sharpe": np.nan, "MDD": np.nan}
    rets = eq.pct_change().dropna()
    if len(rets) == 0:
        return {"Name": name, "CAGR": np.nan, "Vol": np.nan, "Sharpe": np.nan, "MDD": np.nan}
    cagr = (eq.iloc[-1] / eq.iloc[0]) ** (252 / len(rets)) - 1.0
    vol  = rets.std(ddof=1) * np.sqrt(252)
    sharpe = (cagr / vol) if vol > 0 else np.nan
    mdd = ((eq / eq.cummax()) - 1.0).min()
    return {"Name": name, "CAGR": cagr, "Vol": vol, "Sharpe": sharpe, "MDD": mdd}

# ---------- 3) Full-sample comparison table ----------
comp = []
comp.append(summarize_metrics_from_equity(eq_original, "Original"))
comp.append(summarize_metrics_from_equity(eq_vt,       "Vol-Targeted (15%)"))
df_comp = pd.DataFrame(comp)
print("\n=== Metrics — Full Sample ===")
print(df_comp.to_string(index=False))

# ---------- 4) Stress test (Covid-19: 2020) ----------
START_STRESS, END_STRESS = "2020-01-01", "2020-12-31"
eq_o_sub  = eq_original.loc[START_STRESS:END_STRESS]
eq_vt_sub = eq_vt.loc[START_STRESS:END_STRESS]

stress_rows = []
if len(eq_o_sub) > 10 and len(eq_vt_sub) > 10:
    stress_rows.append(summarize_metrics_from_equity(eq_o_sub,  "Original (2020)"))
    stress_rows.append(summarize_metrics_from_equity(eq_vt_sub, "Vol-Targeted (2020)"))
    df_stress = pd.DataFrame(stress_rows)
    print("\n=== Metrics — Covid-19 Stress (2020) ===")
    print(df_stress.to_string(index=False))
else:
    print("\n[Note] Not enough data inside 2020 window to compute stress test metrics.")

# ---------- 5) Plots + saves ----------
out_dir = Path(getattr(res.get("config", None), "out_dir", "out"))
out_dir = Path(str(out_dir))
out_dir.mkdir(exist_ok=True, parents=True)

# Full sample plot
plt.figure(figsize=(10,5))
eq_original.plot(label="Original")
eq_vt.plot(label="Vol-Targeted (15%)")
plt.title("Equity Curve — Original vs Vol-Targeted")
plt.legend(); plt.tight_layout()
plt.savefig(out_dir / "equity_original_vs_voltarget.png"); plt.close()

# Stress window plot
if len(eq_o_sub) > 10 and len(eq_vt_sub) > 10:
    plt.figure(figsize=(10,5))
    eq_o_sub.plot(label="Original (2020)")
    eq_vt_sub.plot(label="Vol-Targeted (2020)")
    plt.title("Equity Curve — Covid-19 Stress (2020)")
    plt.legend(); plt.tight_layout()
    plt.savefig(out_dir / "equity_stress_2020.png"); plt.close()

print("\nSaved files:")
print("-", out_dir / "equity_original_vs_voltarget.png")
if len(eq_o_sub) > 10 and len(eq_vt_sub) > 10:
    print("-", out_dir / "equity_stress_2020.png")


In [ ]:
# === Volatility Targeting + Comparison + 2020 Stress Test (self-contained) ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- 0) Get base daily data from your backtest ----------
if "res" not in globals() or "daily" not in res:
    raise RuntimeError("Backtest results 'res[\"daily\"]' not found. Run the main backtest first.")

daily = res["daily"].copy()
if "equity" not in daily.columns:
    raise KeyError("Column 'equity' not found in res['daily'].")

# Try to find a returns column; else derive from equity
ret_col_candidates = ["daily_ret", "strategy_returns", "ret", "return"]
ret_col = next((c for c in ret_col_candidates if c in daily.columns), None)
if ret_col is None:
    daily["daily_ret"] = daily["equity"].pct_change().fillna(0.0)
    ret_col = "daily_ret"

eq_original = daily["equity"].astype(float)
rets_original = daily[ret_col].astype(float).fillna(0.0)

# ---------- 1) Volatility Targeting ----------
TARGET_VOL = 0.15     # 15% annualized vol target
LOOKBACK   = 60       # 60 trading days
LEV_CAP    = 5.0      # max leverage multiplier

roll_vol_ann = rets_original.rolling(LOOKBACK).std(ddof=1) * np.sqrt(252)
scaler = (TARGET_VOL / roll_vol_ann).clip(upper=LEV_CAP)
rets_vt = (rets_original * scaler.shift(1)).fillna(0.0)    # lag to avoid look-ahead
eq_vt = eq_original.iloc[0] * (1.0 + rets_vt).cumprod()

# ---------- 2) Metrics helpers ----------
def summarize_metrics_from_equity(eq: pd.Series, name="Strategy") -> dict:
    eq = eq.dropna()
    if len(eq) < 3:
        return {"Name": name, "CAGR": np.nan, "Vol": np.nan, "Sharpe": np.nan, "MDD": np.nan}
    rets = eq.pct_change().dropna()
    if len(rets) == 0:
        return {"Name": name, "CAGR": np.nan, "Vol": np.nan, "Sharpe": np.nan, "MDD": np.nan}
    cagr = (eq.iloc[-1] / eq.iloc[0]) ** (252 / len(rets)) - 1.0
    vol  = rets.std(ddof=1) * np.sqrt(252)
    sharpe = (cagr / vol) if vol > 0 else np.nan
    mdd = ((eq / eq.cummax()) - 1.0).min()
    return {"Name": name, "CAGR": cagr, "Vol": vol, "Sharpe": sharpe, "MDD": mdd}

# ---------- 3) Full-sample comparison table ----------
comp = []
comp.append(summarize_metrics_from_equity(eq_original, "Original"))
comp.append(summarize_metrics_from_equity(eq_vt,       "Vol-Targeted (15%)"))
df_comp = pd.DataFrame(comp)
print("\n=== Metrics — Full Sample ===")
print(df_comp.to_string(index=False))

# ---------- 4) Stress test (Covid-19: 2020) ----------
START_STRESS, END_STRESS = "2020-01-01", "2020-12-31"
eq_o_sub  = eq_original.loc[START_STRESS:END_STRESS]
eq_vt_sub = eq_vt.loc[START_STRESS:END_STRESS]

stress_rows = []
if len(eq_o_sub) > 10 and len(eq_vt_sub) > 10:
    stress_rows.append(summarize_metrics_from_equity(eq_o_sub,  "Original (2020)"))
    stress_rows.append(summarize_metrics_from_equity(eq_vt_sub, "Vol-Targeted (2020)"))
    df_stress = pd.DataFrame(stress_rows)
    print("\n=== Metrics — Covid-19 Stress (2020) ===")
    print(df_stress.to_string(index=False))
else:
    print("\n[Note] Not enough data inside 2020 window to compute stress test metrics.")

# ---------- 5) Plots + saves ----------
out_dir = Path(getattr(res.get("config", None), "out_dir", "out"))
out_dir = Path(str(out_dir))
out_dir.mkdir(exist_ok=True, parents=True)

# Full sample plot
plt.figure(figsize=(10,5))
eq_original.plot(label="Original")
eq_vt.plot(label="Vol-Targeted (15%)")
plt.title("Equity Curve — Original vs Vol-Targeted")
plt.legend(); plt.tight_layout()
plt.savefig(out_dir / "equity_original_vs_voltarget.png"); plt.close()

# Stress window plot
if len(eq_o_sub) > 10 and len(eq_vt_sub) > 10:
    plt.figure(figsize=(10,5))
    eq_o_sub.plot(label="Original (2020)")
    eq_vt_sub.plot(label="Vol-Targeted (2020)")
    plt.title("Equity Curve — Covid-19 Stress (2020)")
    plt.legend(); plt.tight_layout()
    plt.savefig(out_dir / "equity_stress_2020.png"); plt.close()

print("\nSaved files:")
print("-", out_dir / "equity_original_vs_voltarget.png")
if len(eq_o_sub) > 10 and len(eq_vt_sub) > 10:
    print("-", out_dir / "equity_stress_2020.png")


In [ ]:
# === FINAL BLOCK — Volatility Targeting + Comparison + 2020 Stress + Johansen Basket ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- locate base output dir ----------
out_dir = Path(getattr(res.get("config", {}), "out_dir", "out"))
out_dir = Path(str(out_dir)); out_dir.mkdir(exist_ok=True, parents=True)

# ---------- 0) pull base daily equity/returns from your backtest ----------
if "res" not in globals() or "daily" not in res or "equity" not in res["daily"].columns:
    raise RuntimeError("Main backtest results not found. Expect res['daily']['equity']. Run the core backtest first.")

daily = res["daily"].copy()
ret_col = next((c for c in ["daily_ret","strategy_returns","ret","return"] if c in daily.columns), None)
if ret_col is None:
    daily["daily_ret"] = daily["equity"].pct_change().fillna(0.0)
    ret_col = "daily_ret"

eq_original   = daily["equity"].astype(float)
rets_original = daily[ret_col].astype(float).fillna(0.0)

# ---------- 1) Volatility Targeting ----------
TARGET_VOL = 0.15      # 15% annualized
LOOKBACK   = 60
LEV_CAP    = 5.0

roll_vol_ann = rets_original.rolling(LOOKBACK).std(ddof=1) * np.sqrt(252)
scaler       = (TARGET_VOL / roll_vol_ann).clip(upper=LEV_CAP)
rets_vt      = (rets_original * scaler.shift(1)).fillna(0.0)   # lag to avoid look-ahead
eq_vt        = eq_original.iloc[0] * (1.0 + rets_vt).cumprod()

# ---------- 2) summary metrics helpers ----------
def summarize_metrics_from_equity(eq: pd.Series, name="Strategy") -> dict:
    eq = eq.dropna()
    if len(eq) < 3:
        return {"Name": name, "CAGR": np.nan, "Vol": np.nan, "Sharpe": np.nan, "MDD": np.nan}
    r = eq.pct_change().dropna()
    if len(r) == 0:
        return {"Name": name, "CAGR": np.nan, "Vol": np.nan, "Sharpe": np.nan, "MDD": np.nan}
    cagr   = (eq.iloc[-1] / eq.iloc[0]) ** (252 / len(r)) - 1.0
    vol    = r.std(ddof=1) * np.sqrt(252)
    sharpe = (cagr / vol) if vol > 0 else np.nan
    mdd    = ((eq / eq.cummax()) - 1.0).min()
    return {"Name": name, "CAGR": cagr, "Vol": vol, "Sharpe": sharpe, "MDD": mdd}

# ---------- 3) full-sample comparison ----------
comp = [
    summarize_metrics_from_equity(eq_original, "Original"),
    summarize_metrics_from_equity(eq_vt,       "Vol-Targeted (15%)"),
]
df_comp = pd.DataFrame(comp)
print("\n=== Metrics — Full Sample ===")
print(df_comp.to_string(index=False))

# ---------- 4) 2020 stress test ----------
START_STRESS, END_STRESS = "2020-01-01", "2020-12-31"
eq_o_2020  = eq_original.loc[START_STRESS:END_STRESS]
eq_vt_2020 = eq_vt.loc[START_STRESS:END_STRESS]

if len(eq_o_2020) > 10 and len(eq_vt_2020) > 10:
    df_stress = pd.DataFrame([
        summarize_metrics_from_equity(eq_o_2020,  "Original (2020)"),
        summarize_metrics_from_equity(eq_vt_2020, "Vol-Targeted (2020)"),
    ])
    print("\n=== Metrics — Covid-19 Stress (2020) ===")
    print(df_stress.to_string(index=False))
else:
    print("\n[Note] Not enough data inside 2020 window to compute stress metrics.")

# ---------- 5) plots: full sample and stress sample ----------
plt.figure(figsize=(10,5))
eq_original.plot(label="Original")
eq_vt.plot(label="Vol-Targeted (15%)")
plt.title("Equity Curve — Original vs Vol-Targeted")
plt.legend(); plt.tight_layout()
plt.savefig(out_dir / "equity_original_vs_voltarget.png"); plt.close()

if len(eq_o_2020) > 10 and len(eq_vt_2020) > 10:
    plt.figure(figsize=(10,5))
    eq_o_2020.plot(label="Original (2020)")
    eq_vt_2020.plot(label="Vol-Targeted (2020)")
    plt.title("Equity Curve — Covid-19 Stress (2020)")
    plt.legend(); plt.tight_layout()
    plt.savefig(out_dir / "equity_stress_2020.png"); plt.close()

print("\nSaved:", out_dir / "equity_original_vs_voltarget.png")
if len(eq_o_2020) > 10 and len(eq_vt_2020) > 10:
    print("Saved:", out_dir / "equity_stress_2020.png")

# ---------- 6) Johansen multivariate (AAPL, GOOGL, META) ----------
# Try to import statsmodels (needed for Johansen)
try:
    from statsmodels.tsa.vector_ar.vecm import coint_johansen
    _joh_ok = True
except Exception as _e:
    _joh_ok = False
    print("\n[Johansen] statsmodels not available. Install with: pip install statsmodels")

# helper: use your existing normalizer if present; else a minimal inline one
def _normalize_price_csv_min(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    # date col
    dcol = next((c for c in df.columns if str(c).lower().startswith("date")), None)
    if dcol is None: raise ValueError(f"No Date column in {path}")
    # price col
    candidates = [c for c in df.columns if str(c).lower().replace(" ","") in
                  {"close","adjclose","adj_close","price"}]
    if not candidates: raise ValueError(f"No Close/Adj Close column in {path}")
    pcol = candidates[0]
    out = df[[dcol, pcol]].copy()
    out.columns = ["Date","Close"]
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out = out.dropna(subset=["Date"]).sort_values("Date")
    out["Close"] = pd.to_numeric(out["Close"], errors="coerce")
    out = out.dropna(subset=["Close"]).reset_index(drop=True)
    return out

# assemble df_y, df_x, df_meta (clean Close series with Date index)
def _as_series(df):
    return df.set_index("Date")["Close"].astype(float).sort_index()

# AAPL/GOOGL should already exist as df_y/df_x in your notebook; if not, try to rebuild:
if "df_y" in globals() and "df_x" in globals():
    s_aapl = _as_series(df_y); s_goog = _as_series(df_x)
else:
    # try from CSVs commonly used
    DATA_DIR = Path("data")
    aapl_paths = [DATA_DIR/"aapl_us_d_clean.csv", DATA_DIR/"aapl_us_d.csv", DATA_DIR/"AAPL.csv"]
    goog_paths = [DATA_DIR/"goog_us_d_clean.csv", DATA_DIR/"goog_us_d.csv", DATA_DIR/"GOOGL.csv", DATA_DIR/"GOOG.csv"]
    def _first_ok(paths):
        for p in paths:
            if Path(p).exists(): return _normalize_price_csv_min(Path(p))
        return None
    dfy = _first_ok(aapl_paths); dfx = _first_ok(goog_paths)
    if dfy is None or dfx is None:
        print("\n[Johansen] Could not reconstruct AAPL/GOOGL. Skipping multivariate test.")
        _joh_ok = False
    else:
        s_aapl = _as_series(dfy); s_goog = _as_series(dfx)

# META: use existing df_meta or try to load from common names
if "df_meta" in globals():
    s_meta = _as_series(df_meta)
else:
    DATA_DIR = Path("data")
    meta_paths = [DATA_DIR/"meta_us_d_clean.csv", DATA_DIR/"meta_us_d.csv", DATA_DIR/"META.csv"]
    dfm = None
    for p in meta_paths:
        if Path(p).exists():
            dfm = _normalize_price_csv_min(Path(p)); break
    if dfm is None:
        print("\n[Johansen] Could not find META CSV. Skipping multivariate test.")
        _joh_ok = False
    else:
        s_meta = _as_series(dfm)

if _joh_ok:
    # build aligned panel
    idx = s_aapl.index.intersection(s_goog.index).intersection(s_meta.index)
    df_multi = pd.concat([s_aapl.loc[idx], s_goog.loc[idx], s_meta.loc[idx]], axis=1)
    df_multi.columns = ["AAPL","GOOGL","META"]
    df_multi = df_multi.dropna()

    if len(df_multi) < 200:
        print("\n[Johansen] Not enough overlapping history. Need more data points.")
    else:
        joh = coint_johansen(df_multi, det_order=0, k_ar_diff=1)
        print("\nJohansen trace statistics (r=0,r<=1,r<=2):", joh.lr1[:3])
        print("Johansen 95% critical values:\n", joh.cvt[:3, 1])

        # first cointegration vector
        vec1 = joh.evec[:, 0]
        spread = df_multi @ vec1
        spread.name = "Johansen_Basket"

        # trade the basket by z-score
        roll = 60
        mu, sig = spread.rolling(roll).mean(), spread.rolling(roll).std(ddof=1)
        z = (spread - mu) / sig

        entry, exit = 2.0, 0.5
        pos = np.where(z > entry, -1,
              np.where(z < -entry, +1,
              np.where(np.abs(z) < exit, 0, np.nan)))
        pos = pd.Series(pos, index=z.index).ffill().fillna(0)

        rets_basket = spread.pct_change().fillna(0.0)
        strat_rets  = pos.shift(1) * rets_basket
        equity_joh  = (1.0 + strat_rets).cumprod()

        # metrics for basket
        def _m(eq):
            r = eq.pct_change().dropna()
            cagr = (eq.iloc[-1]/eq.iloc[0])**(252/len(r)) - 1
            vol  = r.std(ddof=1) * np.sqrt(252)
            sharpe = cagr/vol if vol>0 else np.nan
            mdd = ((eq/eq.cummax())-1).min()
            return {"CAGR": cagr, "Vol": vol, "Sharpe": sharpe, "MDD": mdd}
        print("\nJohansen basket metrics:", _m(equity_joh))

        # plot + save
        plt.figure(figsize=(10,4))
        equity_joh.plot()
        plt.title("Johansen Basket Strategy — Equity")
        plt.tight_layout()
        plt.savefig(out_dir/"equity_johansen_basket.png"); plt.close()
        print("Saved:", out_dir/"equity_johansen_basket.png")


### Multivariate Cointegration (Johansen)

We assembled a 3-asset panel (AAPL, GOOGL, META) and ran the Johansen test (det_order=0, k_ar_diff=1).  
We report trace statistics against 95% critical values and the cointegrating vectors.  
The first vector was normalized to AAPL = 1.0 and used to construct a stationary basket, which can be traded as a mean-reverting portfolio beyond simple pairs.


# Results

> **On cointegration testing.** To keep the pipeline minimal and reproducible under restricted environments, **formal cointegration tests** (ADF/KPSS, Engle–Granger, Johansen) were omitted in this version.  
> In practice, one should (i) test residual stationarity (ADF/KPSS), (ii) validate the long-run relationship (Engle–Granger/Johansen), and (iii) re-estimate the hedge ratio on a rolling or expanding window to avoid look-ahead bias.

The backtest produced the following outputs:

- **Performance metrics**: CAGR, annualized return, volatility, Sharpe ratio, and maximum drawdown.  
- **Sample of trades**: entry and exit dates with trade direction.  
- **Charts**:  
  - Price series of the two assets.  
  - Spread over time.  
  - Z-score evolution.  
  - Equity curve after costs.  

All results are exported to CSV and JSON for reproducibility:  
- `daily.csv` (daily PnL, equity, signals).  
- `trades.csv` (trade log).  
- `metrics.json` (summary of key metrics).  

**Note:** With synthetic data, the maximum drawdown may appear as `0.0` because the simulated equity curve is structurally smoother than real markets.  
In real market data, non-zero drawdowns naturally emerge.

> **Cointegration validation.** We assessed the long-run relation using ADF on the spread, Engle–Granger on prices, and Johansen on the bivariate system. ADF and Engle–Granger p-values below 0.05 indicate residual stationarity and cointegration; Johansen trace statistics above the 95% critical values support at least one cointegrating vector.

> **Stress test (Covid-19, 2020).** We re-ran the backtest exclusively on 2020 to assess crisis behavior. The equity curve and metrics are reported separately, confirming how the strategy behaves under elevated volatility and correlation regime shifts.

**Validation note.** All results above are produced with T+1 execution, explicit transaction costs/slippage, and exported daily logs (`daily.csv`, `trades.csv`, `metrics.json`), ensuring independent replication and auditability.




# Conclusion

This project successfully implemented a pairs trading pipeline from scratch, including:

- Data collection and preprocessing.
- Rolling hedge ratio estimation.
- Signal generation via z-score thresholds.
- Backtest engine with costs and slippage.
- Exportable metrics and visual diagnostics.

**Strengths:**
- Modular and reproducible code, easily extended to new datasets.  
- Automatic handling of missing CSVs with synthetic data.  
- Export of results for reporting and further analysis.

**Limitations:**
- No formal cointegration test (ADF/KPSS/Johansen) was applied in this minimal version.  
- Only two assets are considered; multivariate extensions are possible.  
- Risk management is simplified (fixed notional, no volatility targeting).  

**Next steps:**
- Integrate statistical cointegration tests to validate pairs.  
- Explore dynamic hedge ratio estimation (e.g., Kalman Filter).  
- Perform stress tests on crisis periods (e.g., 2008, 2020).  

Overall, the pipeline demonstrates the core mechanics of pairs trading and provides a solid foundation for further research and professional applications.

**Validation note:** The engine enforces T+1 execution, applies transaction costs/slippage, and exports daily logs, ensuring that results can be independently replicated and audited.


**Future work.** Integrate formal cointegration tests (ADF/KPSS, Engle–Granger, Johansen), add stress windows (e.g., March 2020), introduce volatility-targeted position sizing and cost sensitivity analyses, and validate results on **real market data**.


# Final Conclusion

This project has demonstrated the complete development of a systematic pairs trading strategy, from raw data ingestion and preprocessing to signal generation, backtesting, and performance evaluation. The baseline Engle-Granger pairs approach provided a solid foundation, while the introduction of volatility targeting showed clear improvements in risk-adjusted performance by stabilizing the equity curve and enhancing the Sharpe ratio. Extending the framework to the multivariate Johansen procedure allowed for the construction of cointegrated portfolios beyond simple pairs, illustrating the potential for greater diversification and robustness in statistical arbitrage.

Overall, the results confirm that volatility-aware position sizing and multivariate cointegration represent meaningful enhancements to classical pairs trading. While limitations remain — such as parameter sensitivity, simplified transaction costs, and the absence of execution constraints — the framework is modular, transparent, and reproducible, offering a strong basis for further research and professional implementation in quantitative trading.
